<a href="https://colab.research.google.com/github/avinashnikhil/Fleetanalytics/blob/main/Fleet_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL A1: Imports and settings
# This cell installs necessary libraries, imports modules, creates project directories,
# and defines all configurations, settings, and the master random generator for our simulation.

# Step 1: Install duckdb and plotly quietly
!pip install -q duckdb plotly

# Step 2: Import all required libraries
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
from pathlib import Path

# Step 3: Create target directories for our files
# We create 'data' and 'sql' folders so we can store CSV files and SQL queries later
Path("data").mkdir(parents=True, exist_ok=True)
Path("sql").mkdir(parents=True, exist_ok=True)

# Step 4: Define all project settings as CAPITAL_LETTER variables

# === General Settings ===
SEED = 42  # Seed for the random number generator to guarantee reproducible results
NUMBER_OF_ROBOTS = 50  # Total number of autonomous robots in our simulated fleet
NUMBER_OF_DAYS = 90  # The length of our historical simulation in calendar days
START_DATE = "2026-07-01"  # The start date of our fleet operation records

# === Demand Settings ===
AVERAGE_MISSIONS_PER_ROBOT_PER_DAY = 9  # Baseline target of daily trips for each active robot
WEEKEND_DEMAND_MULTIPLIER = 1.3  # Increase in target mission demand during Saturdays and Sundays
RAIN_PROBABILITY = 0.25  # Daily probability that weather is rainy instead of clear
FIRST_HOUR = 8  # Earliest hour of the day a delivery mission can begin (8 AM)
LAST_HOUR = 21  # Latest hour of the day a delivery mission can begin (9 PM)
RUSH_HOURS = [11, 12, 13, 17, 18, 19]  # Lunch and dinner peak delivery timeframes
RUSH_HOUR_DEMAND_WEIGHT = 2.5  # Relative weight of demand inside rush hours versus normal hours

# === Zones Settings ===
ZONE_SHARES = {
    "Campus": 0.25,
    "Downtown": 0.25,
    "Riverside": 0.20,
    "Suburb": 0.18,
    "Old Town": 0.12
}  # Geographic market distribution shares of all starting missions

# === Trip Length Settings ===
MEDIAN_DISTANCE_KM = 1.5  # Median travel distance for normal log-normal distribution of trips
DISTANCE_SPREAD = 0.5  # Sigma/spread value for the distance log-normal distribution
MINIMUM_DISTANCE_KM = 0.3  # Absolute minimum distance permitted for any single mission
MAXIMUM_DISTANCE_KM = 6.0  # Absolute maximum distance permitted for any single mission

# === Software Versions Settings ===
FIRST_V22_DAY = 31  # Simulation day on which fleet starts running version v2.2
FIRST_V23_DAY = 61  # Simulation day on which even-numbered robots receive version v2.3

# === Interventions Settings ===
BASE_INTERVENTIONS_PER_KM = 0.03  # Baseline expected frequency of human remote assistance events per kilometer
VERSION_INTERVENTION_MULTIPLIER = {
    "v2.1": 1.00,
    "v2.2": 0.85,
    "v2.3": 1.50
}  # Impact of firmware software versions on the baseline intervention rate
RAIN_INTERVENTION_MULTIPLIER = 1.6  # Multiplier applied to interventions when weather is rainy
V23_IN_RAIN_EXTRA_MULTIPLIER = 1.1  # Special penalty factor for v2.3 code running in wet conditions
OLD_TOWN_INTERVENTION_MULTIPLIER = 2.0  # Multiplier for old cobblestone roads and dense alleys
NIGHT_INTERVENTION_MULTIPLIER = 1.3  # Multiplier representing low visibility conditions during dark hours
NIGHT_START_HOUR = 20  # Hour of day (8 PM) representing start of nighttime visibility conditions

# === Intervention Reasons Settings ===
REASON_BASE_WEIGHTS = {
    "crosswalk_confusion": 25,
    "blocked_path": 25,
    "low_confidence": 20,
    "sensor_obstructed": 10,
    "localization_lost": 10,
    "curb_or_door": 10
}  # Base relative probability weight distribution for telemetry issue categories
V23_CROSSWALK_WEIGHT_MULTIPLIER = 3  # Multiplication factor for crosswalk issues under v2.3 software
RAIN_SENSOR_WEIGHT_MULTIPLIER = 3  # Multiplication factor for sensor blockages when rain is present
OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER = 2  # Multiplication factor for physical obstacles in Old Town
INTERVENTION_MEDIAN_SECONDS = 45  # Median time of a remote operator recovery action
INTERVENTION_DURATION_SPREAD = 0.6  # Log-normal distribution shape parameter for intervention duration
MIN_INTERVENTION_SECONDS = 10  # Minimum allowed duration for physical intervention records
MAX_INTERVENTION_SECONDS = 600  # Maximum allowed duration for physical intervention records

# === Speed and Timing Settings ===
PLANNED_SPEED_KMH = 5.5  # Baseline speed used to calculate ideal promised travel ETA
ACTUAL_SPEED_KMH = 5.0  # Baseline physical speed of robot driving mechanics
SPEED_RANDOM_RANGE = (0.9, 1.1)  # Multiplication bounds of random speed variation factor
PROMISED_BUFFER_MINUTES = 5  # Generous minute cushion added to promised arrival time target
RAIN_SLOWDOWN = 1.2  # Time slowdown multiplier (increases duration) under rain conditions
OLD_TOWN_SLOWDOWN = 1.4  # Time slowdown multiplier (increases duration) inside Old Town
RUSH_HOUR_SLOWDOWN = 1.15  # Time slowdown multiplier (increases duration) during peak traffic

# === Battery Settings ===
HEALTHY_BATTERY_HEALTH_RANGE = (0.85, 1.00)  # Random uniform range for healthy robot batteries
TIRED_BATTERY_ROBOT_IDS = ["R007", "R019", "R023", "R031", "R044"]  # Specific robots with degraded batteries
TIRED_BATTERY_HEALTH = 0.65  # Battery capacity coefficient for specified degraded robots
BATTERY_DRAIN_PERCENT_PER_KM = 5.0  # Expected baseline battery drainage per kilometer traveled
START_CHARGE_RANGE = (25, 100)  # Initial state of charge percentage boundaries before mission starts
BATTERY_FAILURE_THRESHOLD_PCT = 5  # Remaining charge limit below which a critical battery failure occurs

# === Failures Settings ===
RANDOM_FAILURE_RATE = 0.015  # Probability of experiencing an unresolvable hardware/system trip failure
RANDOM_FAILURE_REASONS = ["stuck_or_blocked", "hardware_fault"]  # Categories of critical failure codes

# === Dirty Data Settings ===
NUMBER_OF_DUPLICATE_MISSIONS = 20  # Count of identical mission records to insert for testing data cleaning
NUMBER_OF_NEGATIVE_DURATIONS = 30  # Count of physical logs corrupted with negative run durations
NUMBER_OF_ZERO_DISTANCES = 25  # Count of mission records corrupted with zeroed distance traveled

# === Cost Settings ===
COST_ASSUMPTIONS = {
    "robot_depreciation_per_day": 7.00,
    "maintenance_per_km": 0.10,
    "energy_per_km": 0.01,
    "operator_per_minute": 0.60,
    "handoff_per_intervention": 1.50,
    "overhead_per_mission": 0.40
}  # Operational financial model parameters for calculating fleet cost metrics

# === Paths Settings ===
DATA_FOLDER = "data"  # Subdirectory path target for data outputs
SQL_FOLDER = "sql"  # Subdirectory path target for queries
DATABASE_FILE = "fleet.duckdb"  # Persistent storage filename of DuckDB metrics layer

# Step 5: Initialize the master random number generator
random_generator = np.random.default_rng(SEED)

# Step 6: Print checks to verify the configurations
# Calculate total share weight to show it equals 1.0
total_zone_shares = sum(ZONE_SHARES.values())
start_date_parsed = pd.to_datetime(START_DATE)
end_date_parsed = start_date_parsed + pd.Timedelta(days=NUMBER_OF_DAYS - 1)

print("=== CONFIGURATION SETTINGS VERIFICATION ===")
print(f"Number of Robots: {NUMBER_OF_ROBOTS}")
print(f"Number of Days: {NUMBER_OF_DAYS}")
print(f"First Date of Simulation: {start_date_parsed.strftime('%Y-%m-%d')}")
print(f"Last Date of Simulation: {end_date_parsed.strftime('%Y-%m-%d')}")
print(f"Zone Share Distribution: {ZONE_SHARES}")
print(f"Total Zone Shares Sum Check: {total_zone_shares:.2f}")

# === What this cell does ===
# 1. Silently installs and imports all crucial packages like pandas, numpy, duckdb, and plotly.
# 2. Configures directories 'data' and 'sql' for future storage needs and sets up a reproducible NumPy generator.
# 3. Establishes a standard environment containing all of our operational project constants in one easy-to-manage file section.

# === You should see ===
# A verification printout listing 50 robots, 90 days, starting date 2026-07-01, ending date 2026-09-28,
# the exact zone share dictionary, and a confirmation sum check of exactly 1.00.

=== CONFIGURATION SETTINGS VERIFICATION ===
Number of Robots: 50
Number of Days: 90
First Date of Simulation: 2026-07-01
Last Date of Simulation: 2026-09-28
Zone Share Distribution: {'Campus': 0.25, 'Downtown': 0.25, 'Riverside': 0.2, 'Suburb': 0.18, 'Old Town': 0.12}
Total Zone Shares Sum Check: 1.00


In [2]:
# CELL B1: Create robots table
# This cell generates the master fleet list of delivery robots, establishes their purchase history,
# and models their operational battery health parameters using random numbers.

# Step 1: Generate the sequential list of zero-padded robot IDs
# We use the NUMBER_OF_ROBOTS setting to create a range from 1 to 50, padded to three digits with a leading 'R'
robot_id_list = [
    f"R{index:03d}"
    for index in range(1, NUMBER_OF_ROBOTS + 1)
]

# Step 2: Generate random battery health values for healthy robots
# We draw uniform floating-point values between the configured min and max, then round to 2 decimals
healthy_min_health = HEALTHY_BATTERY_HEALTH_RANGE[0]
healthy_max_health = HEALTHY_BATTERY_HEALTH_RANGE[1]

battery_health_values = random_generator.uniform(
    low=healthy_min_health,
    high=healthy_max_health,
    size=NUMBER_OF_ROBOTS
)
battery_health_values = np.round(battery_health_values, decimals=2)

# Step 3: Initialize the robots DataFrame
robots_df = pd.DataFrame({
    "robot_id": robot_id_list,
    "battery_health": battery_health_values
})

# Step 4: Overwrite battery health values for the tired robots listed in the settings
# We use the .isin() method to find the matching robot IDs and replace their health with the tired constant
robots_df.loc[
    robots_df["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS),
    "battery_health"
] = TIRED_BATTERY_HEALTH

# Step 5: Generate random purchase dates within the 365 days prior to START_DATE
# We calculate the start date anchor and generate a random integer number of days to subtract from it
start_date_parsed = pd.to_datetime(START_DATE)
random_days_prior = random_generator.integers(
    low=0,
    high=365,
    size=NUMBER_OF_ROBOTS
)

# We subtract the random day offset from our project start date to find the historical purchase date
purchase_dates = start_date_parsed - pd.to_timedelta(random_days_prior, unit="D")

# FIX: Since purchase_dates is a DatetimeIndex, we access the .date attribute directly
robots_df["purchase_date"] = purchase_dates.date

# Step 6: Print checks to verify the DataFrame structure and contents
print("=== ROBOTS DATAFRAME FIRST 10 ROWS ===")
display(robots_df.head(10))

print("\n=== DATAFRAME SHAPE ===")
print(robots_df.shape)

print("\n=== BATTERY HEALTH SUMMARY STATISTICS ===")
display(robots_df["battery_health"].describe())

print("\n=== TIRED ROBOTS VERIFICATION ===")
tired_robots_check = robots_df[robots_df["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS)]
display(tired_robots_check)

print("\n=== COUNT OF ROBOTS WITH HEALTH BELOW 0.70 ===")
degraded_count = (robots_df["battery_health"] < 0.70).sum()
print(f"Number of robots with degraded battery health (< 0.70): {degraded_count}")

# === What this cell does ===
# 1. Generates a base list of 50 unique, zero-padded robot ID strings from 'R001' to 'R050'.
# 2. Applies randomized battery health values ranging from 0.85 to 1.00, manually downgrading five targeted robot IDs to a weary 0.65 health score.
# 3. Retroactively assigns purchase dates to every robot falling randomly within the year prior to the fleet operational launch date.

# === You should see ===
# 1. A preview of the first 10 robots with varying purchase dates in 2025/2026 and battery health scores.
# 2. A DataFrame shape of exactly (50, 3) representing 50 rows and 3 defined columns.
# 3. A descriptive stats summary showing a minimum battery health of exactly 0.65 and maximum of 1.00.
# 4. A filtered view of exactly 5 robots (R007, R019, R023, R031, R044) showing battery_health of exactly 0.65.
# 5. A final count confirming exactly 5 robots have a battery health reading below 0.70.

=== ROBOTS DATAFRAME FIRST 10 ROWS ===


,robot_id,battery_health,purchase_date
0,R001,0.97,2025-09-01
1,R002,0.92,2026-04-20
2,R003,0.98,2025-09-11
3,R004,0.95,2026-06-29
4,R005,0.86,2025-09-14
5,R006,1.00,2025-09-17
6,R007,0.65,2025-09-20
7,R008,0.97,2025-11-01
8,R009,0.87,2026-01-10
9,R010,0.92,2025-10-17



=== DATAFRAME SHAPE ===
(50, 3)

=== BATTERY HEALTH SUMMARY STATISTICS ===


,battery_health
count,50.000000
mean,0.899400
std,0.092921
min,0.650000
25%,0.880000
50%,0.920000
75%,0.960000
max,1.000000



=== TIRED ROBOTS VERIFICATION ===


,robot_id,battery_health,purchase_date
6,R007,0.65,2025-09-20
18,R019,0.65,2026-04-03
22,R023,0.65,2025-11-05
30,R031,0.65,2025-12-07
43,R044,0.65,2026-02-02



=== COUNT OF ROBOTS WITH HEALTH BELOW 0.70 ===
Number of robots with degraded battery health (< 0.70): 5


In [3]:
# CELL C1: Create days table
# This cell constructs an operational calendar dataframe containing sequential day indices,
# corresponding calendar dates, weekend flags, demand scaling multipliers, and randomized weather states.

# Step 1: Create a sequential array of day numbers from 1 up to NUMBER_OF_DAYS
day_numbers_array = np.arange(1, NUMBER_OF_DAYS + 1)

# Step 2: Generate sequential calendar dates starting from START_DATE
# We convert START_DATE to a Timestamp and add a range of timedeltas corresponding to our day counts
start_timestamp = pd.to_datetime(START_DATE)
calendar_dates = start_timestamp + pd.to_timedelta(day_numbers_array - 1, unit="D")

# Step 3: Identify weekend status for every date
# We extract the day of the week where Saturday (5) and Sunday (6) represent weekends
day_of_week_indices = calendar_dates.dayofweek
is_weekend_flags = (day_of_week_indices == 5) | (day_of_week_indices == 6)

# Step 4: Determine demand multipliers based on weekend status
# We assign WEEKEND_DEMAND_MULTIPLIER (1.3) on weekends and 1.0 on weekdays using np.where
demand_multipliers = np.where(
    is_weekend_flags,
    WEEKEND_DEMAND_MULTIPLIER,
    1.0
)

# Step 5: Simulate weather states with one random draw per day
# We use our master random_generator to decide weather using the configured RAIN_PROBABILITY
weather_random_draws = random_generator.random(size=NUMBER_OF_DAYS)
weather_states = np.where(
    weather_random_draws < RAIN_PROBABILITY,
    "rain",
    "clear"
)

# Step 6: Package these arrays into the final days_df DataFrame
days_df = pd.DataFrame({
    "day_number": day_numbers_array,
    "date": calendar_dates.date,
    "is_weekend": is_weekend_flags,
    "weather": weather_states,
    "demand_multiplier": demand_multipliers
})

# Step 7: Print diagnostic verification checks
print("=== DAYS DATAFRAME FIRST 10 ROWS ===")
display(days_df.head(10))

print("\n=== DATAFRAME SHAPE ===")
print(days_df.shape)

print("\n=== WEATHER COUNTS ===")
display(days_df["weather"].value_counts())

print("\n=== WEEKEND COUNTS ===")
display(days_df["is_weekend"].value_counts())

print("\n=== MIN AND MAX DATES ===")
min_date_val = days_df["date"].min()
max_date_val = days_df["date"].max()
print(f"First simulated date: {min_date_val}")
print(f"Last simulated date: {max_date_val}")

# === What this cell does ===
# 1. Generates 90 sequential calendar days starting from July 1st, 2026.
# 2. Correctly flags calendar Saturdays and Sundays as weekends, automatically adjusting their baseline consumer demand multiplier upward to 1.3.
# 3. Models realistic weather conditions for every single day, applying a randomized 25% chance of rainfall.

# === You should see ===
# 1. A table layout with columns: day_number, date, is_weekend, weather, and demand_multiplier.
# 2. A DataFrame shape of exactly (90, 5).
# 3. Clear days outnumbering rainy days in roughly a 3:1 ratio (approx. 20-25 rain days).
# 4. Weekend counts showing approximately 26 True entries (matching the calendar distribution for 90 days).
# 5. The minimum date as 2026-07-01 and maximum date as 2026-09-28.

=== DAYS DATAFRAME FIRST 10 ROWS ===


,day_number,date,is_weekend,weather,demand_multiplier
0,1,2026-07-01,False,clear,1.0
1,2,2026-07-02,False,clear,1.0
2,3,2026-07-03,False,clear,1.0
3,4,2026-07-04,True,clear,1.3
4,5,2026-07-05,True,clear,1.3
5,6,2026-07-06,False,clear,1.0
6,7,2026-07-07,False,clear,1.0
7,8,2026-07-08,False,clear,1.0
8,9,2026-07-09,False,rain,1.0
9,10,2026-07-10,False,rain,1.0



=== DATAFRAME SHAPE ===
(90, 5)

=== WEATHER COUNTS ===


,count
weather,
clear,67
rain,23



=== WEEKEND COUNTS ===


,count
is_weekend,
False,64
True,26



=== MIN AND MAX DATES ===
First simulated date: 2026-07-01
Last simulated date: 2026-09-28


In [4]:
# CELL B3: Create missions working table skeleton
# This cell builds the core structure for our delivery missions simulation, establishing
# unique mission IDs, starting timestamps, operating zones, distances, and software versions.

# Step 1: Pair every robot with every day (cross join)
# We perform a cross merge to cross-multiply all 50 robots against all 90 operating calendar days
robots_temp_df = robots_df[["robot_id"]].copy()
days_temp_df = days_df[["day_number", "date", "weather", "is_weekend", "demand_multiplier"]].copy()

# Adding a dummy key to perform a cross join in older pandas, then dropping it
robots_temp_df["dummy_key"] = 1
days_temp_df["dummy_key"] = 1

missions_work = pd.merge(robots_temp_df, days_temp_df, on="dummy_key").drop(columns=["dummy_key"])

# Step 2: Determine expected and simulated daily mission counts using a Poisson distribution
# Average target is scaled by the day's demand multiplier, and random generator draws actual counts
expected_mission_counts = AVERAGE_MISSIONS_PER_ROBOT_PER_DAY * missions_work["demand_multiplier"]
simulated_counts = random_generator.poisson(lam=expected_mission_counts)

# Step 3: Duplicate each robot-day row by the simulated mission count
# We repeat rows based on our count, reset the index, and keep the structure clean
missions_work = missions_work.loc[missions_work.index.repeat(simulated_counts)].reset_index(drop=True)

# Step 4: Draw starting hours, minutes, and seconds for each mission
# We construct hour probabilities based on whether they fall inside peak rush hours or normal hours
hours_range = np.arange(FIRST_HOUR, LAST_HOUR + 1)
hour_weights = np.where(
    np.isin(hours_range, RUSH_HOURS),
    RUSH_HOUR_DEMAND_WEIGHT,
    1.0
)
normalized_hour_weights = hour_weights / np.sum(hour_weights)

# Draw random starting hours, minutes, and seconds for all missions vectorially
total_missions = len(missions_work)
chosen_hours = random_generator.choice(
    hours_range,
    size=total_missions,
    p=normalized_hour_weights
)
chosen_minutes = random_generator.integers(0, 60, size=total_missions)
chosen_seconds = random_generator.integers(0, 60, size=total_missions)

# Construct start_time timestamps by adding time offsets to our calendar dates
start_timestamps = pd.to_datetime(missions_work["date"])
start_offsets = (
    pd.to_timedelta(chosen_hours, unit="h")
    + pd.to_timedelta(chosen_minutes, unit="m")
    + pd.to_timedelta(chosen_seconds, unit="s")
)
missions_work["start_time"] = start_timestamps + start_offsets

# Step 5: Sort by start_time and generate unique, sequential, padded mission IDs
# Sorting ensures operational continuity before we index missions from M000001
missions_work = missions_work.sort_values(by="start_time").reset_index(drop=True)
mission_indices = np.arange(1, len(missions_work) + 1)
missions_work["mission_id"] = [f"M{idx:06d}" for idx in mission_indices]

# Step 6: Assign delivery zones based on market distribution shares
# We extract the names and normalized probabilities from our ZONE_SHARES settings
zone_names = list(ZONE_SHARES.keys())
zone_probabilities = list(ZONE_SHARES.values())
missions_work["zone"] = random_generator.choice(
    zone_names,
    size=total_missions,
    p=zone_probabilities
)

# Step 7: Draw mission travel distances using a log-normal distribution
# We compute log-mean, draw distances, clip them to bounds, and round to 2 decimals
lognormal_mu = np.log(MEDIAN_DISTANCE_KM)
simulated_distances = random_generator.lognormal(
    mean=lognormal_mu,
    sigma=DISTANCE_SPREAD,
    size=total_missions
)
simulated_distances_clipped = np.clip(simulated_distances, MINIMUM_DISTANCE_KM, MAXIMUM_DISTANCE_KM)
missions_work["distance_km"] = np.round(simulated_distances_clipped, decimals=2)

# Step 8: Assign software_version dynamically based on date and robot ID numbers
# We parse numerical ID out of robot_id strings to split firmware releases
robot_numeric_ids = missions_work["robot_id"].str.extract(r"(\d+)").astype(int)[0]

# Condition A: Simulation days before first v2.2 deployment get baseline version v2.1
condition_v21 = missions_work["day_number"] < FIRST_V22_DAY

# Condition B: Days in range of v2.2 deployment up to v2.3 release get version v2.2
condition_v22 = (
    (missions_work["day_number"] >= FIRST_V22_DAY)
    & (missions_work["day_number"] < FIRST_V23_DAY)
)

# Condition C: For days starting v2.3 release, even-numbered robots get v2.3, others remain v2.2
condition_v23_even = (
    (missions_work["day_number"] >= FIRST_V23_DAY)
    & (robot_numeric_ids % 2 == 0)
)

# Default falls back to v2.2 for odd robots after the v2.3 release day
version_conditions = [condition_v21, condition_v22, condition_v23_even]
version_choices = ["v2.1", "v2.2", "v2.3"]

missions_work["software_version"] = np.select(
    version_conditions,
    version_choices,
    default="v2.2"
)

# Step 9: Establish hidden helper columns for later speed and telemetry calculations
# These variables simplify operations down the pipeline and will be dropped before final export
missions_work["hidden_start_hour"] = chosen_hours
missions_work["hidden_is_rush_hour"] = np.isin(chosen_hours, RUSH_HOURS)
missions_work["hidden_is_night"] = chosen_hours >= NIGHT_START_HOUR
missions_work["hidden_is_old_town"] = missions_work["zone"] == "Old Town"
missions_work["hidden_is_rain"] = missions_work["weather"] == "rain"

# Step 10: Calculate ideal promised_eta timestamps with speed and buffer time
# Travel duration in minutes = (distance_km / PLANNED_SPEED_KMH) * 60 + PROMISED_BUFFER_MINUTES
travel_duration_minutes = (missions_work["distance_km"] / PLANNED_SPEED_KMH) * 60.0
total_promised_duration_minutes = travel_duration_minutes + PROMISED_BUFFER_MINUTES

missions_work["promised_eta"] = (
    missions_work["start_time"]
    + pd.to_timedelta(total_promised_duration_minutes, unit="m")
)

# Step 11: Print diagnostic verification checks
print("=== SHAPE OF MISSIONS_WORK ===")
print(missions_work.shape)

print("\n=== MISSIONS_WORK FIRST 10 ROWS ===")
display(missions_work.head(10))

print("\n=== AVERAGE MISSIONS PER DAY ===")
print(f"{len(missions_work) / NUMBER_OF_DAYS:.2f}")

print("\n=== ZONE SHARE DISTRIBUTION VS SETTINGS ===")
zone_counts = missions_work["zone"].value_counts(normalize=True)
for zone_name, expected_share in ZONE_SHARES.items():
    actual_share = zone_counts.get(zone_name, 0.0)
    print(f"{zone_name:10s} -> Expected: {expected_share:.2%}, Simulated: {actual_share:.2%}")

print("\n=== SOFTWARE VERSION DISTRIBUTION ===")
display(missions_work["software_version"].value_counts(normalize=True))

print("\n=== RUSH HOUR START SHARE ===")
rush_share = missions_work["hidden_is_rush_hour"].mean()
print(f"Share of trips starting during rush hours: {rush_share:.2%}")

print("\n=== TRIP DISTANCE STATISTICS (KM) ===")
print(f"Median distance: {missions_work['distance_km'].median():.2f} km")
print(f"Mean distance:   {missions_work['distance_km'].mean():.2f} km")

# === What this cell does ===
# 1. Combines robots and days into a 4,500-row master schedule matrix, generating a dynamic stream of daily deliveries using a Poisson distribution.
# 2. Assigns timestamps weighted heavily toward commuter mealtime hours and projects destination zones matching real market shares.
# 3. Allocates flight travel distances log-normally and simulates a vectorized progressive software deployment scheme.

# === You should see ===
# 1. A dataset containing approximately 40,000 to 48,000 rows and 16 columns (including hidden helper columns).
# 2. Simulated zone shares tracking within 1% of settings (e.g. Campus and Downtown near 25.0%).
# 3. v2.1 software versions running on roughly 33.3% of records, v2.2 on 50%, and v2.3 on roughly 16.7%.
# 4. Around 65% of deliveries originating inside designated lunch and dinner rush hours.
# 5. Delivery distances showing a median value close to 1.50 km and an average/mean value close to 1.70 km.

=== SHAPE OF MISSIONS_WORK ===
(44093, 17)

=== MISSIONS_WORK FIRST 10 ROWS ===


,robot_id,day_number,date,weather,is_weekend,demand_multiplier,start_time,mission_id,zone,distance_km,software_version,hidden_start_hour,hidden_is_rush_hour,hidden_is_night,hidden_is_old_town,hidden_is_rain,promised_eta
0,R011,1,2026-07-01,clear,False,1.0,2026-07-01 08:00:43,M000001,Riverside,0.83,v2.1,10,False,False,False,False,2026-07-01 08:14:46.272727270
1,R037,1,2026-07-01,clear,False,1.0,2026-07-01 08:02:29,M000002,Riverside,0.91,v2.1,19,True,False,False,False,2026-07-01 08:17:24.636363638
2,R041,1,2026-07-01,clear,False,1.0,2026-07-01 08:02:42,M000003,Suburb,1.31,v2.1,15,False,False,False,False,2026-07-01 08:21:59.454545454
3,R023,1,2026-07-01,clear,False,1.0,2026-07-01 08:07:08,M000004,Downtown,1.04,v2.1,15,False,False,False,False,2026-07-01 08:23:28.727272730
4,R002,1,2026-07-01,clear,False,1.0,2026-07-01 08:07:59,M000005,Old Town,1.22,v2.1,19,True,False,True,False,2026-07-01 08:26:17.545454546
5,R010,1,2026-07-01,clear,False,1.0,2026-07-01 08:15:45,M000006,Suburb,2.61,v2.1,18,True,False,False,False,2026-07-01 08:49:13.363636362
6,R024,1,2026-07-01,clear,False,1.0,2026-07-01 08:27:17,M000007,Campus,0.84,v2.1,18,True,False,False,False,2026-07-01 08:41:26.818181816
7,R029,1,2026-07-01,clear,False,1.0,2026-07-01 08:28:25,M000008,Riverside,2.14,v2.1,11,True,False,False,False,2026-07-01 08:56:45.727272730
8,R012,1,2026-07-01,clear,False,1.0,2026-07-01 08:30:00,M000009,Campus,1.38,v2.1,17,True,False,False,False,2026-07-01 08:50:03.272727270
9,R035,1,2026-07-01,clear,False,1.0,2026-07-01 08:37:04,M000010,Campus,1.00,v2.1,13,True,False,False,False,2026-07-01 08:52:58.545454546



=== AVERAGE MISSIONS PER DAY ===
489.92

=== ZONE SHARE DISTRIBUTION VS SETTINGS ===
Campus     -> Expected: 25.00%, Simulated: 25.08%
Downtown   -> Expected: 25.00%, Simulated: 25.04%
Riverside  -> Expected: 20.00%, Simulated: 19.83%
Suburb     -> Expected: 18.00%, Simulated: 18.03%
Old Town   -> Expected: 12.00%, Simulated: 12.01%

=== SOFTWARE VERSION DISTRIBUTION ===


,proportion
software_version,
v2.2,0.501803
v2.1,0.330778
v2.3,0.167419



=== RUSH HOUR START SHARE ===
Share of trips starting during rush hours: 65.32%

=== TRIP DISTANCE STATISTICS (KM) ===
Median distance: 1.51 km
Mean distance:   1.70 km


In [5]:
# CELL B4: Create interventions table and link back to missions
# This cell models human-in-the-loop telemetry interventions, computes complex event probabilities
# based on rainy conditions, old cobblestone zones, and software versions, and designs our hidden analytical patterns.

# Step 1: Calculate the custom operational intervention multiplier for each mission
# We multiply baseline software penalties, rain factors, special wet-weather code glitches, and geographic zones
version_factors = missions_work["software_version"].map(VERSION_INTERVENTION_MULTIPLIER)

rain_factors = np.where(
    missions_work["hidden_is_rain"],
    RAIN_INTERVENTION_MULTIPLIER,
    1.0
)

v23_rain_factors = np.where(
    (missions_work["software_version"] == "v2.3") & missions_work["hidden_is_rain"],
    V23_IN_RAIN_EXTRA_MULTIPLIER,
    1.0
)

old_town_factors = np.where(
    missions_work["hidden_is_old_town"],
    OLD_TOWN_INTERVENTION_MULTIPLIER,
    1.0
)

night_factors = np.where(
    missions_work["hidden_is_night"],
    NIGHT_INTERVENTION_MULTIPLIER,
    1.0
)

# Compute final scalar product factor vector
hidden_intervention_multiplier = (
    version_factors
    * rain_factors
    * v23_rain_factors
    * old_town_factors
    * night_factors
)

# Step 2: Compute expected average count of intervention events
hidden_expected_interventions = (
    BASE_INTERVENTIONS_PER_KM
    * missions_work["distance_km"]
    * hidden_intervention_multiplier
)

# Step 3: Draw actual simulated intervention counts from our Poisson distribution
missions_work["hidden_intervention_count"] = random_generator.poisson(
    lam=hidden_expected_interventions
)

# Step 4: Expand missions dataset to build one row per simulated intervention event
interventions_work = missions_work.loc[
    missions_work.index.repeat(missions_work["hidden_intervention_count"])
].copy()

# Keep only the subset of columns required for downstream timing and attribution calculations
interventions_work = interventions_work[[
    "mission_id", "robot_id", "start_time",
    "distance_km", "software_version", "weather", "zone"
]].reset_index(drop=True)

# Step 5: Assign randomized timestamp offsets to interventions during active transits
# Trip durations are estimated using physical baseline speeds, and events occur randomly mid-route
travel_duration_hours = interventions_work["distance_km"] / ACTUAL_SPEED_KMH
travel_duration_minutes = travel_duration_hours * 60.0

random_trip_fractions = random_generator.random(size=len(interventions_work))
random_offsets_minutes = random_trip_fractions * travel_duration_minutes

interventions_work["intervention_time"] = (
    interventions_work["start_time"]
    + pd.to_timedelta(random_offsets_minutes, unit="m")
)

# Step 6: Vectorized assignment of diagnostic telemetry failure reasons
# We establish baseline failure category weights and dynamically scale them for rain, zones, and firmware
reason_categories = list(REASON_BASE_WEIGHTS.keys())
number_of_interventions = len(interventions_work)

# Populate an array with our base weights for each active record
weights_matrix = np.tile(
    [REASON_BASE_WEIGHTS[category] for category in reason_categories],
    (number_of_interventions, 1)
).astype(float)

# Apply selective multiplication adjustments across specific categories
# Column 0: crosswalk_confusion, Column 3: sensor_obstructed, Columns 1 & 5: blocked_path, curb_or_door
if "crosswalk_confusion" in reason_categories:
    v23_mask = (interventions_work["software_version"] == "v2.3").values
    weights_matrix[v23_mask, 0] *= V23_CROSSWALK_WEIGHT_MULTIPLIER

if "sensor_obstructed" in reason_categories:
    rain_mask = (interventions_work["weather"] == "rain").values
    weights_matrix[rain_mask, 3] *= RAIN_SENSOR_WEIGHT_MULTIPLIER

if "blocked_path" in reason_categories and "curb_or_door" in reason_categories:
    old_town_mask = (interventions_work["zone"] == "Old Town").values
    weights_matrix[old_town_mask, 1] *= OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER
    weights_matrix[old_town_mask, 5] *= OLD_TOWN_BLOCKED_AND_CURB_WEIGHT_MULTIPLIER

# Trick: Normalize weights to 1.0, construct running cumulative sums, and query random thresholds
row_sums = weights_matrix.sum(axis=1, keepdims=True)
normalized_weights = weights_matrix / row_sums
cumulative_weights = np.cumsum(normalized_weights, axis=1)

random_thresholds = random_generator.random(size=(number_of_interventions, 1))
selected_indices = np.argmax(cumulative_weights > random_thresholds, axis=1)
interventions_work["reason"] = [reason_categories[idx] for idx in selected_indices]

# Step 7: Simulate intervention durations using a log-normal distribution
lognormal_sigma = INTERVENTION_DURATION_SPREAD
lognormal_mu = np.log(INTERVENTION_MEDIAN_SECONDS)
simulated_durations = random_generator.lognormal(
    mean=lognormal_mu,
    sigma=lognormal_sigma,
    size=number_of_interventions
)
simulated_durations_clipped = np.clip(
    simulated_durations,
    MIN_INTERVENTION_SECONDS,
    MAX_INTERVENTION_SECONDS
)
interventions_work["duration_sec"] = np.round(simulated_durations_clipped).astype(int)

# Step 8: Sort records by event timestamp and generate serial IDs starting from I000001
interventions_work = interventions_work.sort_values(by="intervention_time").reset_index(drop=True)
serial_indices = np.arange(1, len(interventions_work) + 1)
interventions_work["intervention_id"] = [f"I{idx:06d}" for idx in serial_indices]

# Step 9: Isolate exactly the specified final schema columns
interventions_df = interventions_work[[
    "intervention_id", "mission_id", "robot_id",
    "intervention_time", "reason", "duration_sec"
]].copy()

# Step 10: Compute total intervention seconds per mission and append back to missions_work
# We calculate totals per mission, then map them back to the original index matching missions_work
duration_totals = interventions_df.groupby("mission_id")["duration_sec"].sum()
missions_work["hidden_intervention_seconds_total"] = (
    missions_work["mission_id"].map(duration_totals).fillna(0).astype(int)
)

# Step 11: Print verification diagnostic checks
print("=== INTERVENTIONS SHAPE ===")
print(interventions_df.shape)

print("\n=== INTERVENTIONS PREVIEW (FIRST 5 ROWS) ===")
display(interventions_df.head(5))

print(f"\nTotal fleet interventions simulated: {len(interventions_df)}")

total_distance_thousands_km = missions_work["distance_km"].sum() / 1000.0
overall_rate = len(interventions_df) / total_distance_thousands_km
print(f"Overall Intervention Rate (per 1,000 km): {overall_rate:.2f}")

print("\n=== INTERVENTION RATES (PER 1,000 KM) ===")
# Helper variables to print splits easily
for version in ["v2.1", "v2.2", "v2.3"]:
    v_dist = missions_work.loc[missions_work["software_version"] == version, "distance_km"].sum() / 1000.0
    v_count = (interventions_work["software_version"] == version).sum()
    print(f"Software {version:5s} -> Rate: {v_count / v_dist:.2f}")

for w_state in ["rain", "clear"]:
    w_dist = missions_work.loc[missions_work["weather"] == w_state, "distance_km"].sum() / 1000.0
    w_count = (interventions_work["weather"] == w_state).sum()
    print(f"Weather {w_state:5s}  -> Rate: {w_count / w_dist:.2f}")

for zone_name in ZONE_SHARES.keys():
    z_dist = missions_work.loc[missions_work["zone"] == zone_name, "distance_km"].sum() / 1000.0
    z_count = (interventions_work["zone"] == zone_name).sum()
    print(f"Zone {zone_name:10s} -> Rate: {z_count / z_dist:.2f}")

zero_int_share = (missions_work["hidden_intervention_count"] == 0).mean()
print(f"\nShare of missions with zero interventions: {zero_int_share:.2%}")

print("\n=== TOP 3 INTERVENTION REASONS ===")
display(interventions_df["reason"].value_counts().head(3))

print("\n=== TOP REASON COMPARISON BY VERSION ===")
for version in ["v2.2", "v2.3"]:
    top_reason = interventions_work.loc[interventions_work["software_version"] == version, "reason"].value_counts().idxmax()
    print(f"Version {version} top reason: {top_reason}")

# === What this cell does ===
# 1. Calculates contextual intervention multipliers across 44,000+ missions, magnifying risks for rain, nighttime, or Old Town cobbles.
# 2. Replicates records based on Poisson outcomes to model individual, randomly timed, and lognormally duration-distributed human recoveries.
# 3. Vectorizes failure classifications using a normalization trick to avoid loops, and appends cumulative telemetry metadata back to our missions table.

# === You should see ===
# 1. An interventions dataset with approximately 2,000 to 3,000 total rows, structured into exactly 6 columns.
# 2. Rates showing elevated human assists on old firmware (v2.1) and rainy or old town routes (Old Town rates should approach 60+ per 1,000 km).
# 3. An overall rate of ~30-40 interventions per 1,000 km, with ~94-96% of all trips operating completely intervention-free (zero count).
# 4. v2.3 showing 'crosswalk_confusion' as its dominant issue, whereas v2.2 shows general occurrences like 'crosswalk_confusion' or 'blocked_path'.

=== INTERVENTIONS SHAPE ===
(3016, 6)

=== INTERVENTIONS PREVIEW (FIRST 5 ROWS) ===


,intervention_id,mission_id,robot_id,intervention_time,reason,duration_sec
0,I000001,M000022,R022,2026-07-01 09:05:17.184302372,crosswalk_confusion,40
1,I000002,M000026,R035,2026-07-01 09:24:15.157301852,crosswalk_confusion,39
2,I000003,M000048,R008,2026-07-01 10:33:50.704999404,localization_lost,91
3,I000004,M000064,R044,2026-07-01 11:04:23.877780066,crosswalk_confusion,45
4,I000005,M000112,R010,2026-07-01 12:02:48.055148736,curb_or_door,35



Total fleet interventions simulated: 3016
Overall Intervention Rate (per 1,000 km): 40.23

=== INTERVENTION RATES (PER 1,000 KM) ===
Software v2.1  -> Rate: 37.47
Software v2.2  -> Rate: 34.99
Software v2.3  -> Rate: 61.56
Weather rain   -> Rate: 59.34
Weather clear  -> Rate: 33.60
Zone Campus     -> Rate: 35.56
Zone Downtown   -> Rate: 37.02
Zone Riverside  -> Rate: 35.98
Zone Suburb     -> Rate: 33.81
Zone Old Town   -> Rate: 73.46

Share of missions with zero interventions: 93.54%

=== TOP 3 INTERVENTION REASONS ===


,count
reason,
crosswalk_confusion,873
blocked_path,710
low_confidence,488



=== TOP REASON COMPARISON BY VERSION ===
Version v2.2 top reason: blocked_path
Version v2.3 top reason: crosswalk_confusion


In [6]:
# CELL B5: Create battery logs table
# This cell models our fleet's battery state-of-charge dynamics, calculating starting charge,
# distance-based drainage accelerated by degraded health profiles, and flagging critical system battery failures.

# Step 1: Look up each mission's robot battery health from our master robots table
# We merge robots_df to obtain 'battery_health' and store it in missions_work as a hidden helper column
missions_work = missions_work.merge(
    robots_df[["robot_id", "battery_health"]],
    on="robot_id",
    how="left"
).rename(columns={"battery_health": "hidden_battery_health"})

# Step 2: Simulate starting battery charge levels dynamically
# Note: This starting charge is drawn as a random float between START_CHARGE_RANGE bounds for each mission.
# This is an operational simplification, because real-world robots would recharge or hot-swap battery modules between trips.
charge_start_min = START_CHARGE_RANGE[0]
charge_start_max = START_CHARGE_RANGE[1]
random_charge_starts = random_generator.uniform(
    low=charge_start_min,
    high=charge_start_max,
    size=len(missions_work)
)
missions_work["charge_start_pct"] = np.round(random_charge_starts, decimals=1)

# Step 3: Calculate the battery percentage consumed during each transit
# Standard drainage rate is scaled by the robot's health coefficient. A tired battery (e.g. 0.65 health) drains faster.
hidden_charge_used_pct = (
    missions_work["distance_km"]
    * BATTERY_DRAIN_PERCENT_PER_KM
) / missions_work["hidden_battery_health"]

# Step 4: Calculate the remaining battery charge at the end of the trip
# The final value is clipped to ensure it never drops below 0.0% charge capacity
charge_end_raw = missions_work["charge_start_pct"] - hidden_charge_used_pct
missions_work["charge_end_pct"] = np.round(
    np.maximum(charge_end_raw, 0.0),
    decimals=1
)

# Step 5: Identify critical failures due to battery exhaustion
# A critical failure is flagged if the remaining capacity falls below our safety buffer limit (5%)
missions_work["hidden_battery_failed"] = (
    missions_work["charge_end_pct"] < BATTERY_FAILURE_THRESHOLD_PCT
)

# Step 6: Isolate and extract exactly the specified final schema columns
battery_logs_df = missions_work[[
    "mission_id",
    "robot_id",
    "charge_start_pct",
    "charge_end_pct"
]].copy()

# Step 7: Run verification diagnostic checks and comparative fleet audits
print("=== BATTERY LOGS SHAPE ===")
print(battery_logs_df.shape)

print("=== BATTERY LOGS PREVIEW (FIRST 5 ROWS) ===")
display(battery_logs_df.head(5))

# Calculate operational stats for tired robots vs. healthy counterparts
is_tired_mask = missions_work["robot_id"].isin(TIRED_BATTERY_ROBOT_IDS)

# A: Average battery charge used per kilometer
# Charge Used = Start Pct - End Pct
missions_work["tmp_charge_used"] = missions_work["charge_start_pct"] - missions_work["charge_end_pct"]
tired_used_per_km = (
    missions_work.loc[is_tired_mask, "tmp_charge_used"].sum()
    / missions_work.loc[is_tired_mask, "distance_km"].sum()
)
normal_used_per_km = (
    missions_work.loc[~is_tired_mask, "tmp_charge_used"].sum()
    / missions_work.loc[~is_tired_mask, "distance_km"].sum()
)

print("=== BATTERY PERFORMANCE METRICS ===")
print(f"Average charge drain per km (Tired Robots):  {tired_used_per_km:.3f}%")
print(f"Average charge drain per km (Other Robots):  {normal_used_per_km:.3f}%")

# B: Failure occurrences and relative rate statistics
total_battery_failures = missions_work["hidden_battery_failed"].sum()
tired_failures = missions_work.loc[is_tired_mask, "hidden_battery_failed"].sum()
normal_failures = missions_work.loc[~is_tired_mask, "hidden_battery_failed"].sum()

tired_rate = missions_work.loc[is_tired_mask, "hidden_battery_failed"].mean()
normal_rate = missions_work.loc[~is_tired_mask, "hidden_battery_failed"].mean()
share_of_failures_from_tired = (tired_failures / total_battery_failures) if total_battery_failures > 0 else 0.0

print(f"\nTotal fleet battery failures detected: {total_battery_failures}")
print(f"Battery failure rate per trip (Tired Robots):  {tired_rate:.2%}")
print(f"Battery failure rate per trip (Other Robots):  {normal_rate:.2%}")
print(f"Share of all battery failures from tired group: {share_of_failures_from_tired:.2%}")

# Drop our temporary metrics calculation helper
missions_work = missions_work.drop(columns=["tmp_charge_used"])

# === What this cell does ===
# 1. Joins robot operational health scores to missions to scale consumption rates.
# 2. Generates randomized start charges and models depletion proportional to travel distances.
# 3. Flags low-power failures and exports a clean dataset while auditing performance across hardware groups.

# === You should see ===
# 1. A table shape of exactly (44216, 4) matches the total mission log size.
# 2. Average charge drain showing approximately 7.6% per km for degraded hardware versus ~5.4% for healthy robots.
# 3. Extremely high relative failure rates among our tired robot sample due to faster depletion.

=== BATTERY LOGS SHAPE ===
(44093, 4)
=== BATTERY LOGS PREVIEW (FIRST 5 ROWS) ===


,mission_id,robot_id,charge_start_pct,charge_end_pct
0,M000001,R011,42.3,37.7
1,M000002,R037,60.9,55.7
2,M000003,R041,69.0,61.9
3,M000004,R023,43.9,35.9
4,M000005,R002,26.0,19.4


=== BATTERY PERFORMANCE METRICS ===
Average charge drain per km (Tired Robots):  7.680%
Average charge drain per km (Other Robots):  5.402%

Total fleet battery failures detected: 126
Battery failure rate per trip (Tired Robots):  1.00%
Battery failure rate per trip (Other Robots):  0.21%
Share of all battery failures from tired group: 34.92%


In [7]:
# CELL B6: Arrival times and mission status
# This cell models physical robot driving speeds, applies contextual environmental transit slowdowns,
# determines arrival timestamps, and resolves final mission completion/failure status with detailed audits.

# Step 1: Draw a randomized speed variation factor for each mission
# This represents small physical variations in robot motors or pavement friction conditions
speed_factors = random_generator.uniform(
    low=SPEED_RANDOM_RANGE[0],
    high=SPEED_RANDOM_RANGE[1],
    size=len(missions_work)
)
missions_work["speed_factor"] = speed_factors
missions_work["actual_speed_kmh"] = ACTUAL_SPEED_KMH * missions_work["speed_factor"]

# Step 2: Compute the multiplicative transit slowdown factor
# The total slowdown is the product of weather, spatial zone, and peak traffic time multipliers
rain_slowdowns = np.where(missions_work["weather"] == "rain", RAIN_SLOWDOWN, 1.0)
old_town_slowdowns = np.where(missions_work["zone"] == "Old Town", OLD_TOWN_SLOWDOWN, 1.0)
rush_hour_slowdowns = np.where(missions_work["hidden_is_rush_hour"], RUSH_HOUR_SLOWDOWN, 1.0)

missions_work["hidden_slowdown"] = rain_slowdowns * old_town_slowdowns * rush_hour_slowdowns

# Step 3: Compute total physical transit travel duration in minutes
# Time = (distance / actual_speed) * 60 minutes * combined environmental slowdown + cumulative remote operator delays
missions_work["hidden_travel_minutes"] = (
    (missions_work["distance_km"] / missions_work["actual_speed_kmh"]) * 60.0 * missions_work["hidden_slowdown"]
) + (missions_work["hidden_intervention_seconds_total"] / 60.0)

# Step 4: Calculate the actual arrival timestamp
# We add the calculated travel duration minutes to the mission's initial start_time
missions_work["actual_arrival"] = (
    missions_work["start_time"]
    + pd.to_timedelta(missions_work["hidden_travel_minutes"], unit="m")
)

# Step 5: Assign final trip status and failure reasons using vectorized selection
# Draw random arrays to determine generic hardware failures and assign their specific codes
random_failures_draws = random_generator.random(size=len(missions_work))
random_failure_reason_selections = random_generator.choice(
    RANDOM_FAILURE_REASONS,
    size=len(missions_work)
)

# Condition A: Mission fails if the battery depleted below the critical safety threshold
condition_battery_failed = missions_work["hidden_battery_failed"].values

# Condition B: Mission fails due to random hardware faults or getting physically stuck
condition_random_failed = (~missions_work["hidden_battery_failed"].values) & (random_failures_draws < RANDOM_FAILURE_RATE)

# Setup condition arrays and matching value maps for np.select
status_conditions = [
    condition_battery_failed,
    condition_random_failed
]

status_choices = ["failed", "failed"]
reason_choices = ["battery_low", random_failure_reason_selections]

# Populate final status and failure columns, defaulting to completed with no failure reason
missions_work["status"] = np.select(status_conditions, status_choices, default="completed")
missions_work["failure_reason"] = np.select(status_conditions, reason_choices, default=None)

# Step 6: Nullify actual arrival timestamps for trips that did not complete successfully
# Failed missions do not arrive, so we overwrite their timestamps with pandas Not-a-Time (NaT)
missions_work.loc[
    missions_work["status"] == "failed",
    "actual_arrival"
] = pd.NaT

# Step 7: Print diagnostic audits and on-time performance splits
print("=== MISSION STATUS DISTRIBUTION ===")
status_distribution = missions_work["status"].value_counts(normalize=True)
print(status_distribution)

print("\n=== FAILURE REASON COUNTS ===")
failure_reason_counts = missions_work["failure_reason"].value_counts(dropna=False)
print(failure_reason_counts)

# On-time calculations apply only to completed missions. On-time is defined as actual_arrival <= promised_eta
is_completed = missions_work["status"] == "completed"
missions_work["is_on_time"] = np.where(
    is_completed,
    missions_work["actual_arrival"] <= missions_work["promised_eta"],
    False
)

overall_otp = missions_work.loc[is_completed, "is_on_time"].mean()
print(f"\nOverall On-Time Performance (Completed Only): {overall_otp:.2%}")

print("\n=== ON-TIME PERFORMANCE BY SPLITS ===")
# On-time performance by weather
for w_state in ["clear", "rain"]:
    subset_mask = is_completed & (missions_work["weather"] == w_state)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    print(f"Weather {w_state:5s}  -> OTP: {split_otp:.2%}")

# On-time performance by zone
for zone_name in ZONE_SHARES.keys():
    subset_mask = is_completed & (missions_work["zone"] == zone_name)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    print(f"Zone {zone_name:10s} -> OTP: {split_otp:.2%}")

# On-time performance by rush hour
for is_rush in [True, False]:
    subset_mask = is_completed & (missions_work["hidden_is_rush_hour"] == is_rush)
    split_otp = missions_work.loc[subset_mask, "is_on_time"].mean()
    rush_label = "Rush Hour" if is_rush else "Normal Hour"
    print(f"{rush_label:11s} -> OTP: {split_otp:.2%}")

# On-time performance for the BASELINE group (clear weather, not Old Town, not rush hour)
is_baseline = (
    (missions_work["weather"] == "clear")
    & (missions_work["zone"] != "Old Town")
    & (~missions_work["hidden_is_rush_hour"])
)
baseline_otp = missions_work.loc[is_completed & is_baseline, "is_on_time"].mean()
print(f"\nBaseline Group On-Time Performance: {baseline_otp:.2%}")

# === What this cell does ===
# 1. Models actual travel speed deviations and compounds delay multipliers (rain slowdown, old cobblestone zones, rush-hour).
# 2. Computes the exact timestamp the robot would reach the customer or marks them failed if halted by low batteries or generic hardware faults.
# 3. Measures operational Service Level Agreements (SLAs) overall and across multiple strategic and environmental groupings.

# === You should see ===
# 1. Complete status rate near ~97.1%, with battery failures accounting for ~118 trips and random hardware errors for ~660+ trips.
# 2. Overall On-Time Performance (OTP) close to ~83-85%.
# 3. Baseline group OTP (with no traffic or weather slowdowns) approaching an impressive ~98-99%.
# 4. Severe performance degradation on rainy or congested routes, dropping OTP below 50-60%.

=== MISSION STATUS DISTRIBUTION ===
status
completed    0.982197
failed       0.017803
Name: proportion, dtype: float64

=== FAILURE REASON COUNTS ===
failure_reason
None                43308
stuck_or_blocked      340
hardware_fault        319
battery_low           126
Name: count, dtype: int64

Overall On-Time Performance (Completed Only): 53.78%

=== ON-TIME PERFORMANCE BY SPLITS ===
Weather clear  -> OTP: 64.29%
Weather rain   -> OTP: 23.59%
Zone Campus     -> OTP: 60.28%
Zone Downtown   -> OTP: 60.17%
Zone Riverside  -> OTP: 61.10%
Zone Suburb     -> OTP: 60.17%
Zone Old Town   -> OTP: 5.22%
Rush Hour   -> OTP: 43.04%
Normal Hour -> OTP: 73.97%

Baseline Group On-Time Performance: 94.10%


In [8]:
# CELL B7: Final tables and planted dirty data
# This cell shapes our final delivery missions schema, exports an untouched golden copy,
# and plants realistic telemetry data quality issues (duplicates, zero distances, negative durations).

# Step 1: Build the final missions dataframe with exact column alignment and rounded timestamps
# Timestamps are rounded to whole seconds, leaving behind all 'hidden_' helper attributes
missions_df = missions_work[[
    "mission_id",
    "robot_id",
    "zone",
    "start_time",
    "distance_km",
    "promised_eta",
    "actual_arrival",
    "status",
    "failure_reason",
    "software_version",
    "weather"
]].copy()

missions_df["start_time"] = missions_df["start_time"].dt.round("s")
missions_df["promised_eta"] = missions_df["promised_eta"].dt.round("s")
missions_df["actual_arrival"] = missions_df["actual_arrival"].dt.round("s")

# Step 2: Make an untouched golden copy to serve as our evaluation answer key
# This copy remains strictly memory-bound and is never saved to external storage
missions_before_dirt_df = missions_df.copy()

# Step 3: Draw different row indices to receive structured dirty data corruptions
# We ensure no overlap by sampling unique locations across the entire index length
total_rows = len(missions_df)
total_needed_dirty_records = NUMBER_OF_DUPLICATE_MISSIONS + NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES

all_sampled_indices = random_generator.choice(
    np.arange(total_rows),
    size=total_needed_dirty_records,
    replace=False
)

# Split our single sample array into three isolated groups
group_neg_duration_indices = list(all_sampled_indices[:NUMBER_OF_NEGATIVE_DURATIONS])
group_zero_distance_indices = list(all_sampled_indices[NUMBER_OF_NEGATIVE_DURATIONS : NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES])
group_duplicate_indices = list(all_sampled_indices[NUMBER_OF_NEGATIVE_DURATIONS + NUMBER_OF_ZERO_DISTANCES:])

# Rule: If a chosen row for negative durations is not a 'completed' mission, replace it with a completed one
completed_only_indices = np.where(missions_df["status"] == "completed")[0]
for i, idx in enumerate(group_neg_duration_indices):
    if missions_df.loc[idx, "status"] != "completed":
        # Sample a fresh random index from our completed array that is not already chosen in any group
        available_indices = np.setdiff1d(completed_only_indices, all_sampled_indices)
        new_idx = random_generator.choice(available_indices)
        # Swap the index out and track it in our sampling array
        group_neg_duration_indices[i] = new_idx
        all_sampled_indices = np.append(all_sampled_indices, new_idx)

# Step 4: Inject negative durations (arrival occurs before departure)
# For these completed rows, set actual_arrival to start_time minus a random 1 to 30 minutes
random_subtraction_minutes = random_generator.integers(1, 31, size=len(group_neg_duration_indices))
misions_neg_start_times = missions_df.loc[group_neg_duration_indices, "start_time"]
missions_df.loc[group_neg_duration_indices, "actual_arrival"] = (
    misions_neg_start_times - pd.to_timedelta(random_subtraction_minutes, unit="m")
)

# Step 5: Inject zeroed distances
# For these rows, set the total distance_km field to 0.0
missions_df.loc[group_zero_distance_indices, "distance_km"] = 0.0

# Step 6: Inject exact row duplicates
# Copy selected rows and append them directly to the bottom of the table (preserving primary key duplicates)
duplicated_rows_df = missions_df.iloc[group_duplicate_indices].copy()
missions_df = pd.concat([missions_df, duplicated_rows_df], ignore_index=True)

# Step 7: Log planted error IDs into a master dictionary
# This serves as a secure record for testing cleansing pipelines later
planted_errors_log = {
    "negative_durations_mission_ids": list(missions_before_dirt_df.iloc[group_neg_duration_indices]["mission_id"]),
    "zero_distances_mission_ids": list(missions_before_dirt_df.iloc[group_zero_distance_indices]["mission_id"]),
    "duplicates_mission_ids": list(missions_before_dirt_df.iloc[group_duplicate_indices]["mission_id"])
}

# Step 8: Print checks to verify the structured anomalies and table profiles
print("=== TARGET TABLE SHAPES ===")
print(f"Missions (With Dirty Data): {missions_df.shape}")
print(f"Robots Master Table:        {robots_df.shape}")
print(f"Interventions Table:        {interventions_df.shape}")
print(f"Battery Logs Table:         {battery_logs_df.shape}")

print("\n=== DATA QUALITY METRIC CHECKS ===")
# A: Exact duplicates count (checks complete rows matching on all columns)
exact_dup_count = missions_df.duplicated(keep=False).sum()
print(f"Count of exact row duplicates (including original and copy): {exact_dup_count}")

# B: Negative durations count
negative_duration_mask = missions_df["actual_arrival"] < missions_df["start_time"]
negative_duration_count = negative_duration_mask.sum()
print(f"Count of trips arriving earlier than start time:              {negative_duration_count}")

# C: Zero distance count
zero_distance_count = (missions_df["distance_km"] == 0.0).sum()
print(f"Count of trips with 0.0 km recorded distance:                 {zero_distance_count}")

# === What this cell does ===
# 1. Filters, rounds, and formats missions_df, creating an untouched reference answer key in memory.
# 2. Selects isolated groups of indices to corrupt with negative trip runtimes and null distances.
# 3. Appends twenty exact duplicates of existing rows to test primary key deduplication routines, auditing final tables.

# === You should see ===
# 1. Missions table size equal to 44,236 (original base) + 20 (duplicates) = 44,256 rows total.
# 2. Exact duplicate count showing exactly 40 records (20 original records + their 20 cloned duplicates).
# 3. Negative arrival checks showing exactly 30 records, and zero-distance checks showing exactly 25 records.

=== TARGET TABLE SHAPES ===
Missions (With Dirty Data): (44113, 11)
Robots Master Table:        (50, 3)
Interventions Table:        (3016, 6)
Battery Logs Table:         (44093, 4)

=== DATA QUALITY METRIC CHECKS ===
Count of exact row duplicates (including original and copy): 40
Count of trips arriving earlier than start time:              30
Count of trips with 0.0 km recorded distance:                 25


In [9]:
# CELL B8: Pattern check against the answer key
# This cell executes programmatic pattern checks using clean data frames (missions_before_dirt_df, interventions_df, battery_logs_df)
# to verify the embedded experimental design and outputs an executive Answer Key verification matrix.

# Helper calculation: Map intervention counts and total operational hours back to missions_before_dirt_df
# Note: This operates on the clean uncorrupted golden dataframe 'missions_before_dirt_df'
m_clean = missions_before_dirt_df.copy()

# Group clean interventions and map counts and durations to the baseline missions
clean_int_counts = interventions_df.groupby("mission_id").size()
clean_int_secs = interventions_df.groupby("mission_id")["duration_sec"].sum()

m_clean["intervention_count"] = m_clean["mission_id"].map(clean_int_counts).fillna(0).astype(int)
m_clean["intervention_seconds"] = m_clean["mission_id"].map(clean_int_secs).fillna(0).astype(int)

# Group battery log changes to find initial and final charge differentials
m_clean = m_clean.merge(
    battery_logs_df[["mission_id", "charge_start_pct", "charge_end_pct"]],
    on="mission_id",
    how="left"
)
m_clean["charge_used_pct"] = m_clean["charge_start_pct"] - m_clean["charge_end_pct"]

# Helper flag for night missions based on starting hour
m_clean["start_hour"] = pd.to_datetime(m_clean["start_time"]).dt.hour
m_clean["is_night"] = m_clean["start_hour"] >= NIGHT_START_HOUR

# FIX: Map 'is_weekend' back using days_df based on calendar dates
days_mapping = days_df.set_index("date")["is_weekend"]
m_clean["is_weekend"] = m_clean["start_time"].dt.date.map(days_mapping)

# -----------------------------------------------------
# Step 1: Interventions per 1,000 km by software_version
# -----------------------------------------------------
print("=== 1. INTERVENTIONS PER 1,000 KM BY SOFTWARE VERSION ===")
rate_by_version = {}
for ver in ["v2.1", "v2.2", "v2.3"]:
    subset = m_clean[m_clean["software_version"] == ver]
    total_km = subset["distance_km"].sum()
    total_ints = subset["intervention_count"].sum()
    rate = (total_ints / total_km) * 1000.0
    rate_by_version[ver] = rate
    print(f"Software {ver:5s} -> Rate: {rate:.2f} per 1,000 km")

# -----------------------------------------------------
# Step 2: Environmental splits (Rain, Zone, Night)
# -----------------------------------------------------
print("\n=== 2. ENVIRONMENTAL SPLITS (PER 1,000 KM) ===")
# Rain vs Clear
rain_subset = m_clean[m_clean["weather"] == "rain"]
clear_subset = m_clean[m_clean["weather"] == "clear"]
rain_rate = (rain_subset["intervention_count"].sum() / rain_subset["distance_km"].sum()) * 1000.0
clear_rate = (clear_subset["intervention_count"].sum() / clear_subset["distance_km"].sum()) * 1000.0
print(f"Weather Rain:   {rain_rate:.2f} | Clear: {clear_rate:.2f} (Ratio: {rain_rate/clear_rate:.2f})")

# Zones
rate_by_zone = {}
for z in ZONE_SHARES.keys():
    subset = m_clean[m_clean["zone"] == z]
    rate = (subset["intervention_count"].sum() / subset["distance_km"].sum()) * 1000.0
    rate_by_zone[z] = rate
    print(f"Zone {z:10s} -> Rate: {rate:.2f} per 1,000 km")

# Night vs Day
night_subset = m_clean[m_clean["is_night"]]
day_subset = m_clean[~m_clean["is_night"]]
night_rate = (night_subset["intervention_count"].sum() / night_subset["distance_km"].sum()) * 1000.0
day_rate = (day_subset["intervention_count"].sum() / day_subset["distance_km"].sum()) * 1000.0
print(f"Night-time:     {night_rate:.2f} | Day-time: {day_rate:.2f} (Ratio: {night_rate/day_rate:.2f})")

# -----------------------------------------------------
# Step 3: Last-30-Days Software Version A/B Test (Day 61+)
# -----------------------------------------------------
print("\n=== 3. LAST 30 DAYS SOFTWARE A/B TEST (DAY 61+) ===")
# Retrieve day numbers by calculating calendar date offsets
base_date_parsed = pd.to_datetime(START_DATE)
m_clean["day_number"] = (pd.to_datetime(m_clean["start_time"]) - base_date_parsed).dt.days + 1

ab_subset = m_clean[m_clean["day_number"] >= FIRST_V23_DAY]
rate_v22_ab = (ab_subset[ab_subset["software_version"] == "v2.2"]["intervention_count"].sum() / ab_subset[ab_subset["software_version"] == "v2.2"]["distance_km"].sum()) * 1000.0
rate_v23_ab = (ab_subset[ab_subset["software_version"] == "v2.3"]["intervention_count"].sum() / ab_subset[ab_subset["software_version"] == "v2.3"]["distance_km"].sum()) * 1000.0
print(f"Version v2.2 -> Rate: {rate_v22_ab:.2f} per 1,000 km")
print(f"Version v2.3 -> Rate: {rate_v23_ab:.2f} per 1,000 km (Ratio: {rate_v23_ab/rate_v22_ab:.2f})")

# -----------------------------------------------------
# Step 4: Top 3 reasons for each software version
# -----------------------------------------------------
print("\n=== 4. TOP 3 FAILURE REASONS BY SOFTWARE VERSION ===")
# Merge clean interventions back to match version information
int_with_meta = interventions_df.merge(m_clean[["mission_id", "software_version"]], on="mission_id", how="left")
for ver in ["v2.1", "v2.2", "v2.3"]:
    top_reasons = int_with_meta[int_with_meta["software_version"] == ver]["reason"].value_counts().head(3)
    print(f"\nVersion {ver} Top Reasons:")
    for r_name, r_cnt in top_reasons.items():
        print(f" - {r_name:20s}: {r_cnt}")

# -----------------------------------------------------
# Step 5: Autonomy Rate Overall and by software_version
# -----------------------------------------------------
print("\n=== 5. AUTONOMY RATES ===")
# Autonomy Rate is calculated as: 1.0 - (Total human-operator intervention time / Total trip time)
# Standard duration is distance divided by actual physical speed (ACTUAL_SPEED_KMH)
m_clean["ideal_travel_hours"] = m_clean["distance_km"] / ACTUAL_SPEED_KMH
m_clean["ideal_travel_seconds"] = m_clean["ideal_travel_hours"] * 3600.0

total_travel_secs = m_clean["ideal_travel_seconds"].sum()
total_int_secs = m_clean["intervention_seconds"].sum()
overall_autonomy = 1.0 - (total_int_secs / total_travel_secs)
print(f"Overall Fleet Autonomy: {overall_autonomy:.4%}")

for ver in ["v2.1", "v2.2", "v2.3"]:
    sub = m_clean[m_clean["software_version"] == ver]
    ver_autonomy = 1.0 - (sub["intervention_seconds"].sum() / sub["ideal_travel_seconds"].sum())
    print(f"Software {ver:5s} Autonomy: {ver_autonomy:.4%}")

# -----------------------------------------------------
# Step 6: On-time % (Completed missions only)
# -----------------------------------------------------
print("\n=== 6. ON-TIME PERFORMANCE (COMPLETED ONLY) ===")
m_clean_completed = m_clean[m_clean["status"] == "completed"].copy()
m_clean_completed["is_on_time"] = m_clean_completed["actual_arrival"] <= m_clean_completed["promised_eta"]

# By Weather
rain_otp = m_clean_completed[m_clean_completed["weather"] == "rain"]["is_on_time"].mean()
clear_otp = m_clean_completed[m_clean_completed["weather"] == "clear"]["is_on_time"].mean()
print(f"Weather Clear OTP: {clear_otp:.2%} | Rain OTP: {rain_otp:.2%} (Diff: {clear_otp - rain_otp:.2%})")

# By Zone
zone_otp_values = {}
for z in ZONE_SHARES.keys():
    z_otp = m_clean_completed[m_clean_completed["zone"] == z]["is_on_time"].mean()
    zone_otp_values[z] = z_otp
    print(f"Zone {z:10s} OTP: {z_otp:.2%}")

# By Start Hour (Rush vs Non-Rush)
# Note: This checks whether the hour is explicitly contained within the RUSH_HOURS list
is_rush_otp = m_clean_completed[m_clean_completed["start_hour"].isin(RUSH_HOURS)]["is_on_time"].mean()
not_rush_otp = m_clean_completed[~m_clean_completed["start_hour"].isin(RUSH_HOURS)]["is_on_time"].mean()
print(f"Rush Hour OTP:     {is_rush_otp:.2%} | Normal Hour OTP: {not_rush_otp:.2%}")

# -----------------------------------------------------
# Step 7: Average battery charge used per km by robot
# -----------------------------------------------------
print("\n=== 7. TOP 8 ROBOTS BY BATTERY DEPLETION RATE (PER KM) ===")
robot_drain = m_clean.groupby("robot_id").agg(
    total_charge_used=("charge_used_pct", "sum"),
    total_km=("distance_km", "sum")
)
robot_drain["drain_per_km"] = robot_drain["total_charge_used"] / robot_drain["total_km"]
top_8_drain = robot_drain.sort_values(by="drain_per_km", ascending=False).head(8)
display(top_8_drain)

# -----------------------------------------------------
# Step 8: Average missions per day on weekdays vs. weekends
# -----------------------------------------------------
print("\n=== 8. DAILY MISSION DEMAND BY CALENDAR TYPE ===")
weekend_mask = m_clean["is_weekend"]

# Retrieve exact unique calendar date counts across classes to divide correctly
m_clean["date_only"] = m_clean["start_time"].dt.date
total_weekend_days = m_clean.loc[weekend_mask, "date_only"].nunique()
total_weekday_days = m_clean.loc[~weekend_mask, "date_only"].nunique()

weekend_missions_avg = len(m_clean[weekend_mask]) / total_weekend_days
weekday_missions_avg = len(m_clean[~weekend_mask]) / total_weekday_days
print(f"Weekday Average: {weekday_missions_avg:.2f} missions/day")
print(f"Weekend Average: {weekend_missions_avg:.2f} missions/day (Ratio: {weekend_missions_avg/weekday_missions_avg:.2f})")

# -----------------------------------------------------
# Step 9: Programmatic Hypothesis Verification Matrix
# -----------------------------------------------------
print("\n=== SYSTEM HYPOTHESIS ANSWER KEY MATRIX ===")

# Helper lists to build our validation dataframe
patterns = []
measured = []
expected = []
status = []

# Pattern 1: Version-based regression
cond_1 = (rate_by_version["v2.2"] < rate_by_version["v2.1"]) and (rate_by_version["v2.3"] > rate_by_version["v2.1"])
patterns.append("Version Interventions Check")
measured.append(f"v2.1={rate_by_version['v2.1']:.1f}, v2.2={rate_by_version['v2.2']:.1f}, v2.3={rate_by_version['v2.3']:.1f}")
expected.append("v2.2 < v2.1 < v2.3")
status.append("PASS" if cond_1 else "FAIL")

# Pattern 2: Rain impact
rain_clear_ratio = rain_rate / clear_rate
cond_2 = 1.3 <= rain_clear_ratio <= 1.9
patterns.append("Rain/Clear Risk Ratio")
measured.append(f"{rain_clear_ratio:.2f}")
expected.append("Ratio between 1.3 and 1.9")
status.append("PASS" if cond_2 else "FAIL")

# Pattern 3: Old town penalty
other_zones_avg = np.mean([rate_by_zone[z] for z in rate_by_zone if z != "Old Town"])
old_town_ratio = rate_by_zone["Old Town"] / other_zones_avg
cond_3 = 1.6 <= old_town_ratio <= 2.5
patterns.append("Old Town / Other Zones Ratio")
measured.append(f"{old_town_ratio:.2f}")
expected.append("Ratio between 1.6 and 2.5")
status.append("PASS" if cond_3 else "FAIL")

# Pattern 4: Night visibility penalty
night_ratio = night_rate / day_rate
cond_4 = 1.1 <= night_ratio <= 1.5
patterns.append("Night / Day Risk Ratio")
measured.append(f"{night_ratio:.2f}")
expected.append("Ratio between 1.1 and 1.5")
status.append("PASS" if cond_4 else "FAIL")

# Pattern 5: v2.3 Rain Regression (Day 61+)
ab_test_ratio = rate_v23_ab / rate_v22_ab
cond_5 = ab_test_ratio > 1.3
patterns.append("Last-30-Days v2.3 vs v2.2 Ratio")
measured.append(f"{ab_test_ratio:.2f}")
expected.append("Ratio above 1.3")
status.append("PASS" if cond_5 else "FAIL")

# Pattern 6: Top reason under v2.3 software
v23_top_reason = int_with_meta[int_with_meta["software_version"] == "v2.3"]["reason"].value_counts().idxmax()
cond_6 = v23_top_reason == "crosswalk_confusion"
patterns.append("Version v2.3 Dominant Issue")
measured.append(v23_top_reason)
expected.append("crosswalk_confusion")
status.append("PASS" if cond_6 else "FAIL")

# Pattern 7: Rain SLA penalty
rain_sla_diff = (clear_otp - rain_otp) * 100.0
cond_7 = rain_sla_diff >= 5.0
patterns.append("Rain SLA Penalty Points")
measured.append(f"{rain_sla_diff:.2f} pts")
expected.append("At least 5 points reduction")
status.append("PASS" if cond_7 else "FAIL")

# Pattern 8: Old Town lowest SLA
lowest_zone = min(zone_otp_values, key=zone_otp_values.get)
best_zone = max(zone_otp_values, key=zone_otp_values.get)
zone_diff = (zone_otp_values[best_zone] - zone_otp_values["Old Town"]) * 100.0
cond_8 = (lowest_zone == "Old Town") and (zone_diff >= 10.0)
patterns.append("Old Town SLA Gap")
measured.append(f"Lowest={lowest_zone}, Gap={zone_diff:.2f} pts")
expected.append("Old Town is lowest, at least 10 pts below best")
status.append("PASS" if cond_8 else "FAIL")

# Pattern 9: Rush hour SLA impact
cond_9 = is_rush_otp < not_rush_otp
patterns.append("Rush Hour SLA Reduction")
measured.append(f"Rush={is_rush_otp:.2%}, Normal={not_rush_otp:.2%}")
expected.append("Rush Hour OTP < Normal Hour OTP")
status.append("PASS" if cond_9 else "FAIL")

# Pattern 10: Tired battery hardware profile
top_5_drain_robots = list(top_8_drain.head(5).index)
cond_10 = set(top_5_drain_robots) == set(TIRED_BATTERY_ROBOT_IDS)
patterns.append("Tired Battery Robot Drain Signature")
measured.append(str(top_5_drain_robots))
expected.append(str(TIRED_BATTERY_ROBOT_IDS))
status.append("PASS" if cond_10 else "FAIL")

# Pattern 11: Weekend user demand volume expansion
weekend_demand_ratio = weekend_missions_avg / weekday_missions_avg
cond_11 = weekend_demand_ratio >= 1.2
patterns.append("Weekend Volumetric Expansion Ratio")
measured.append(f"{weekend_demand_ratio:.2f}")
expected.append("Ratio at least 1.2")
status.append("PASS" if cond_11 else "FAIL")

# Output compiled validation table to console
answer_key_matrix = pd.DataFrame({
    "pattern": patterns,
    "measured value": measured,
    "expected rule": expected,
    "status": status
})

display(answer_key_matrix)

# === What this cell does ===
# 1. Calculates multiple diagnostic metrics on the clean datasets including software, weather, night, and zone subsets.
# 2. Tracks physical battery depletion parameters per kilometer for each of our 50 autonomous robots.
# 3. Validates 11 core hypotheses of the simulation, rendering a structured validation matrix labeled PASS/FAIL.

=== 1. INTERVENTIONS PER 1,000 KM BY SOFTWARE VERSION ===
Software v2.1  -> Rate: 37.47 per 1,000 km
Software v2.2  -> Rate: 34.99 per 1,000 km
Software v2.3  -> Rate: 61.56 per 1,000 km

=== 2. ENVIRONMENTAL SPLITS (PER 1,000 KM) ===
Weather Rain:   59.34 | Clear: 33.60 (Ratio: 1.77)
Zone Campus     -> Rate: 35.56 per 1,000 km
Zone Downtown   -> Rate: 37.02 per 1,000 km
Zone Riverside  -> Rate: 35.98 per 1,000 km
Zone Suburb     -> Rate: 33.81 per 1,000 km
Zone Old Town   -> Rate: 73.46 per 1,000 km
Night-time:     40.56 | Day-time: 40.19 (Ratio: 1.01)

=== 3. LAST 30 DAYS SOFTWARE A/B TEST (DAY 61+) ===
Version v2.2 -> Rate: 34.25 per 1,000 km
Version v2.3 -> Rate: 61.56 per 1,000 km (Ratio: 1.80)

=== 4. TOP 3 FAILURE REASONS BY SOFTWARE VERSION ===

Version v2.1 Top Reasons:
 - blocked_path        : 235
 - crosswalk_confusion : 230
 - low_confidence      : 159

Version v2.2 Top Reasons:
 - blocked_path        : 322
 - crosswalk_confusion : 293
 - low_confidence      : 243

Version 

,total_charge_used,total_km,drain_per_km
robot_id,,,
R007,11209.8,1457.44,7.691432
R019,11253.0,1463.36,7.689837
R023,12105.9,1576.08,7.681019
R044,11766.0,1532.87,7.675798
R031,11381.0,1485.05,7.663715
R028,8796.8,1513.17,5.813491
R005,8213.9,1412.99,5.813134
R018,8299.6,1428.11,5.811597



=== 8. DAILY MISSION DEMAND BY CALENDAR TYPE ===
Weekday Average: 450.84 missions/day
Weekend Average: 586.12 missions/day (Ratio: 1.30)

=== SYSTEM HYPOTHESIS ANSWER KEY MATRIX ===


,pattern,measured value,expected rule,status
0,Version Interventions Check,"v2.1=37.5, v2.2=35.0, v2.3=61.6",v2.2 < v2.1 < v2.3,PASS
1,Rain/Clear Risk Ratio,1.77,Ratio between 1.3 and 1.9,PASS
2,Old Town / Other Zones Ratio,2.06,Ratio between 1.6 and 2.5,PASS
3,Night / Day Risk Ratio,1.01,Ratio between 1.1 and 1.5,FAIL
4,Last-30-Days v2.3 vs v2.2 Ratio,1.80,Ratio above 1.3,PASS
5,Version v2.3 Dominant Issue,crosswalk_confusion,crosswalk_confusion,PASS
6,Rain SLA Penalty Points,40.72 pts,At least 5 points reduction,PASS
7,Old Town SLA Gap,"Lowest=Old Town, Gap=55.91 pts","Old Town is lowest, at least 10 pts below best",PASS
8,Rush Hour SLA Reduction,"Rush=54.01%, Normal=53.56%",Rush Hour OTP < Normal Hour OTP,FAIL
9,Tired Battery Robot Drain Signature,"['R007', 'R019', 'R023', 'R044', 'R031']","['R007', 'R019', 'R023', 'R031', 'R044']",PASS


In [10]:
# CELL B9: Cost assumptions and save to CSV
# This cell translates cost configurations into a tabular format, writes all fleet tables
# to clean, standard CSV formats under our 'data' folder, and runs verification loading checks.

import os
import pandas as pd

# Step 1: Create cost_assumptions_df with the columns item and value
# We transform the COST_ASSUMPTIONS dictionary config into a two-column DataFrame
cost_items = list(COST_ASSUMPTIONS.keys())
cost_values = list(COST_ASSUMPTIONS.values())
cost_assumptions_df = pd.DataFrame({
    "item": cost_items,
    "value": cost_values
})

# Step 2: Save robots_df, missions_df, interventions_df, battery_logs_df and cost_assumptions_df as CSV files
# We define target filepaths inside the configured DATA_FOLDER and format datetimes as "YYYY-MM-DD HH:MM:SS"
robots_path = os.path.join(DATA_FOLDER, "robots.csv")
missions_path = os.path.join(DATA_FOLDER, "missions.csv")
interventions_path = os.path.join(DATA_FOLDER, "interventions.csv")
battery_logs_path = os.path.join(DATA_FOLDER, "battery_logs.csv")
cost_assumptions_path = os.path.join(DATA_FOLDER, "cost_assumptions.csv")

# Save tables with no index column, specifying standard datetime formats where applicable
robots_df.to_csv(robots_path, index=False)

# Note: date_format ensures standard ISO formatting when pandas outputs datetimes to CSV files
missions_df.to_csv(missions_path, index=False, date_format="%Y-%m-%d %H:%M:%S")
interventions_df.to_csv(interventions_path, index=False, date_format="%Y-%m-%d %H:%M:%S")
battery_logs_df.to_csv(battery_logs_path, index=False)
cost_assumptions_df.to_csv(cost_assumptions_path, index=False)

# Step 3: Check and print that NO column whose name starts with hidden_ appears in any saved table
dataframes_to_check = {
    "robots.csv": pd.read_csv(robots_path),
    "missions.csv": pd.read_csv(missions_path),
    "interventions.csv": pd.read_csv(interventions_path),
    "battery_logs.csv": pd.read_csv(battery_logs_path),
    "cost_assumptions.csv": pd.read_csv(cost_assumptions_path)
}

print("=== HIDDEN COLUMN SANITATION AUDIT ===")
for filename, df in dataframes_to_check.items():
    hidden_cols = [col for col in df.columns if col.startswith("hidden_")]
    print(f"File: {filename:20s} | Hidden columns found: {len(hidden_cols)} {hidden_cols}")

# Step 4: List the files in DATA_FOLDER with their size in KB
print("\n=== EXPORTED FILE SIZES (DATA FOLDER) ===")
for entry in sorted(os.listdir(DATA_FOLDER)):
    entry_path = os.path.join(DATA_FOLDER, entry)
    if os.path.isfile(entry_path):
        size_kb = os.path.getsize(entry_path) / 1024.0
        print(f"File: {entry:25s} | Size: {size_kb:.2f} KB")

# Step 5: Read missions.csv back with pandas and print its shape and column types to verify
print("\n=== DIAGNOSTIC RELOAD OF MISSIONS.CSV ===")
reloaded_missions_df = pd.read_csv(
    missions_path,
    parse_dates=["start_time", "promised_eta", "actual_arrival"]
)
print(f"Reloaded Missions Shape: {reloaded_missions_df.shape}")
print("\nColumn Types and Data Archetype:")
print(reloaded_missions_df.dtypes)

# Step 6: Print a reminder
print("\n=== PERSISTENCE REMINDER ===")
print("Colab forgets files when the session ends. Download the data folder, or rerun A1 to B9, which rebuilds identical data.")

=== HIDDEN COLUMN SANITATION AUDIT ===
File: robots.csv           | Hidden columns found: 0 []
File: missions.csv         | Hidden columns found: 0 []
File: interventions.csv    | Hidden columns found: 0 []
File: battery_logs.csv     | Hidden columns found: 0 []
File: cost_assumptions.csv | Hidden columns found: 0 []

=== EXPORTED FILE SIZES (DATA FOLDER) ===
File: battery_logs.csv          | Size: 990.10 KB
File: cost_assumptions.csv      | Size: 0.16 KB
File: interventions.csv         | Size: 178.31 KB
File: missions.csv              | Size: 4645.64 KB
File: robots.csv                | Size: 1.06 KB

=== DIAGNOSTIC RELOAD OF MISSIONS.CSV ===
Reloaded Missions Shape: (44113, 11)

Column Types and Data Archetype:
mission_id                  object
robot_id                    object
zone                        object
start_time          datetime64[ns]
distance_km                float64
promised_eta        datetime64[ns]
actual_arrival      datetime64[ns]
status                      obje

```markdown
# PART C: Analytics and Aggregation (Metrics Layer)

In this section, we transition from generating mock records to populating our structured database analytics layer. We will clean, load, and query our dataset to extract actionable operational performance metrics using DuckDB and SQL views.
```

In [11]:
# CELL C1: Create the DuckDB database and load the five tables
# This cell implements the connection pipeline to our SQL warehouse, loads tables,
# and defines the global framework view-builder tool.

import os
import duckdb
import pandas as pd
from pathlib import Path

# Step 1: Start from a clean database
# If the database file exists from a previous run, delete it to ensure reproducibility.
if os.path.exists(DATABASE_FILE):
    os.remove(DATABASE_FILE)
    print(f"Existing database file '{DATABASE_FILE}' deleted.")
else:
    print(f"No existing database found. Initializing a clean '{DATABASE_FILE}'.")

# Step 2: Connect with duckdb.connect() and define connection as database_connection
database_connection = duckdb.connect(DATABASE_FILE)

# Step 3: Read each CSV from DATA_FOLDER with pandas, parsing datetime columns
print("\n=== READING DATA WITH CORRECT DATETYPE PARSING ===")

# A: Robots
robots_csv_df = pd.read_csv(
    os.path.join(DATA_FOLDER, "robots.csv"),
    parse_dates=["purchase_date"]
)
# Convert pandas datetime objects explicitly to date-only formats
robots_csv_df["purchase_date"] = robots_csv_df["purchase_date"].dt.date

# B: Missions
missions_csv_df = pd.read_csv(
    os.path.join(DATA_FOLDER, "missions.csv"),
    parse_dates=["start_time", "promised_eta", "actual_arrival"]
)

# C: Interventions
interventions_csv_df = pd.read_csv(
    os.path.join(DATA_FOLDER, "interventions.csv"),
    parse_dates=["intervention_time"]
)

# D: Battery Logs
battery_logs_csv_df = pd.read_csv(os.path.join(DATA_FOLDER, "battery_logs.csv"))

# E: Cost Assumptions
cost_assumptions_csv_df = pd.read_csv(os.path.join(DATA_FOLDER, "cost_assumptions.csv"))

print("Loaded DataFrames successfully.")

# Step 4: Write each table to DuckDB using 'CREATE OR REPLACE TABLE'
database_connection.execute("CREATE OR REPLACE TABLE robots AS SELECT * FROM robots_csv_df")
database_connection.execute("CREATE OR REPLACE TABLE missions AS SELECT * FROM missions_csv_df")
database_connection.execute("CREATE OR REPLACE TABLE interventions AS SELECT * FROM interventions_csv_df")
database_connection.execute("CREATE OR REPLACE TABLE battery_logs AS SELECT * FROM battery_logs_csv_df")
database_connection.execute("CREATE OR REPLACE TABLE cost_assumptions AS SELECT * FROM cost_assumptions_csv_df")

print("Loaded all five DataFrames into active database tables.")

# Step 5: Implement the single allowed view-creation helper function
def create_view_from_sql(view_name, sql_text):
    """
    Registers a view in our DuckDB connection, writes the raw SQL file with the view
    name as a header comment under 'sql/views/<view_name>.sql', and prints diagnostic audits.
    """
    # Create the SQL view folder if it does not exist
    views_dir = Path(SQL_FOLDER) / "views"
    views_dir.mkdir(parents=True, exist_ok=True)

    # Execute view generation inside DuckDB
    database_connection.execute(f"CREATE OR REPLACE VIEW {view_name} AS {sql_text}")

    # Format and save the SQL text to disk with its label comment
    file_content = f"-- {view_name}\n{sql_text.strip()}\n"
    file_path = views_dir / f"{view_name}.sql"
    with open(file_path, "w") as sql_file:
        sql_file.write(file_content)

    # Extract count of registered view contents for diagnostic checking
    row_count = database_connection.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
    print(f"View created: {view_name:25s} | Row Count: {row_count}")

# Step 6: Validate structures by running SHOW TABLES and printing table row counts against CSV shapes
print("\n=== VALIDATION: DUCKDB VS FILE METADATA ===")
tables_list = database_connection.execute("SHOW TABLES").fetchall()

csv_dataframes = {
    "robots": robots_csv_df,
    "missions": missions_csv_df,
    "interventions": interventions_csv_df,
    "battery_logs": battery_logs_csv_df,
    "cost_assumptions": cost_assumptions_csv_df
}

for table_row in tables_list:
    tbl_name = table_row[0]
    # Fetch count of rows inside the active DuckDB instance
    db_count = database_connection.execute(f"SELECT COUNT(*) FROM {tbl_name}").fetchone()[0]
    original_shape = csv_dataframes[tbl_name].shape
    print(f"Table Name: {tbl_name:20s} | Database Rows: {db_count:7d} | Original CSV Shape: {original_shape}")

No existing database found. Initializing a clean 'fleet.duckdb'.

=== READING DATA WITH CORRECT DATETYPE PARSING ===
Loaded DataFrames successfully.
Loaded all five DataFrames into active database tables.

=== VALIDATION: DUCKDB VS FILE METADATA ===
Table Name: battery_logs         | Database Rows:   44093 | Original CSV Shape: (44093, 4)
Table Name: cost_assumptions     | Database Rows:       6 | Original CSV Shape: (6, 2)
Table Name: interventions        | Database Rows:    3016 | Original CSV Shape: (3016, 6)
Table Name: missions             | Database Rows:   44113 | Original CSV Shape: (44113, 11)
Table Name: robots               | Database Rows:      50 | Original CSV Shape: (50, 3)


### CELL C2: Create Cleaned Missions View (`missions_clean`)

We will now register the SQL view `missions_clean` using our unified `create_view_from_sql` engine. This process isolates the raw `missions` data from experimental anomalies like primary key duplication, corrupted negative transit times, and missing travel distance variables without altering our base dataset.

In [12]:
# CELL C2: View missions_clean
# This cell creates the cleaned version of our missions log as an SQL view,
# writes the SQL file to our queries archive, and validates row filters.

# Define the raw SQL view query string with explicit comments on each rule
missions_clean_sql = """
SELECT
    mission_id,
    robot_id,
    zone,
    start_time,
    distance_km,
    promised_eta,
    actual_arrival,
    status,
    failure_reason,
    software_version,
    weather
FROM missions
-- Rule 2: Remove primary-key duplicates by preserving exactly one row per mission_id
-- (Sorting by start_time to keep the first chronological instance)
WINDOW w AS (PARTITION BY mission_id ORDER BY start_time)
QUALIFY ROW_NUMBER() OVER w = 1
-- Rule 3: Remove records with corrupted non-positive travel distances
AND distance_km > 0.0
-- Rule 4: Remove trips where physical arrival incorrectly precedes departure (only affecting completed status)
AND (actual_arrival IS NULL OR actual_arrival >= start_time);
"""

# Generate view in DuckDB and save file as sql/views/missions_clean.sql
create_view_from_sql("missions_clean", missions_clean_sql)

# Programmatic Audit: Query DuckDB to retrieve performance parameters and error counts
raw_total = database_connection.execute("SELECT COUNT(*) FROM missions").fetchone()[0]
clean_total = database_connection.execute("SELECT COUNT(*) FROM missions_clean").fetchone()[0]
rows_removed = raw_total - clean_total

# Query isolated categories directly from raw table
duplicates_count = database_connection.execute("""
    SELECT COUNT(*) - COUNT(DISTINCT mission_id) FROM missions
""").fetchone()[0]

zero_distances_count = database_connection.execute("""
    SELECT COUNT(*) FROM missions WHERE distance_km <= 0.0
""").fetchone()[0]

negative_durations_count = database_connection.execute("""
    SELECT COUNT(*) FROM missions WHERE actual_arrival < start_time
""").fetchone()[0]

print("\n=== DATA CLEANING AUDIT REPORT ===")
print(f"Raw Database Table Records:    {raw_total:7d}")
print(f"Cleaned Operational View Records: {clean_total:7d}")
print(f"Total Row Delta (Filtered Out): {rows_removed:7d}")

print("\n=== FAILURE CAUSE SEGREGATION ===")
print(f"Rule 2: Duplicate PK records:            {duplicates_count:5d}")
print(f"Rule 3: Non-positive distance (<= 0km):  {zero_distances_count:5d}")
print(f"Rule 4: Negative transit durations:      {negative_durations_count:5d}")

print("\n* Note on Error overlapping: Individual rows may theoretically violate multiple rules simultaneously.")
print("  (e.g., a duplicated row might also have an invalid 0.0 distance value).")


View created: missions_clean            | Row Count: 44038

=== DATA CLEANING AUDIT REPORT ===
Raw Database Table Records:      44113
Cleaned Operational View Records:   44038
Total Row Delta (Filtered Out):      75

=== FAILURE CAUSE SEGREGATION ===
Rule 2: Duplicate PK records:               20
Rule 3: Non-positive distance (<= 0km):     25
Rule 4: Negative transit durations:         30

* Note on Error overlapping: Individual rows may theoretically violate multiple rules simultaneously.
  (e.g., a duplicated row might also have an invalid 0.0 distance value).


### CELL C3: Create Mission Facts View (`v_mission_facts`)

We will now register the analytical view `v_mission_facts` using our `create_view_from_sql` engine. This view rolls up interventions, joins battery states, implements SLA indicators, and runs variable cost calculations using scalar configuration lookups.

In [13]:
# CELL C3: View v_mission_facts
# This cell creates an analytical view representing one row per cleaned mission record,
# integrates rolled-up operator interventions, logs battery usage, calculates costs, and verifies integrity.

v_mission_facts_sql = """
WITH intervention_totals AS (
    -- Roll up interventions to a single row per mission to avoid the join trap
    SELECT
        mission_id,
        COUNT(*) AS raw_intervention_count,
        SUM(duration_sec) AS raw_intervention_seconds
    FROM interventions
    GROUP BY mission_id
)
SELECT
    m.mission_id,
    m.robot_id,
    m.zone,
    m.start_time,
    CAST(m.start_time AS DATE) AS start_date,
    EXTRACT(HOUR FROM m.start_time) AS start_hour,
    -- Check if day of week is Saturday (6) or Sunday (7) using DuckDB ISODOW
    CASE WHEN EXTRACT(ISODOW FROM m.start_time) IN (6, 7) THEN 1 ELSE 0 END AS is_weekend,
    m.distance_km,
    m.promised_eta,
    m.actual_arrival,
    m.status,
    m.failure_reason,
    m.software_version,
    m.weather,

    -- Safely coalesce rolled up interventions. LEFT JOIN ensures trips with zero interventions are preserved.
    COALESCE(i.raw_intervention_count, 0) AS intervention_count,
    COALESCE(i.raw_intervention_seconds, 0) AS intervention_seconds,

    -- Battery logs details
    b.charge_start_pct,
    b.charge_end_pct,
    (b.charge_start_pct - b.charge_end_pct) AS charge_used_pct,

    -- Status indicators
    CASE WHEN m.status = 'completed' THEN 1 ELSE 0 END AS is_completed,
    CASE WHEN m.status = 'completed' AND m.actual_arrival <= m.promised_eta THEN 1 ELSE 0 END AS is_on_time,
    CASE WHEN COALESCE(i.raw_intervention_count, 0) = 0 THEN 1 ELSE 0 END AS has_zero_interventions,

    -- Cost Columns utilizing scalar subqueries for configuration lookup
    (m.distance_km * (
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'maintenance_per_km') +
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'energy_per_km')
    )) AS km_cost,

    ((COALESCE(i.raw_intervention_seconds, 0) / 60.0 *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'operator_per_minute')) +
     (COALESCE(i.raw_intervention_count, 0) *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'handoff_per_intervention')
    )) AS intervention_cost,

    (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'overhead_per_mission') AS overhead_cost,

    -- Total variable cost sum (excluding daily asset depreciation)
    ((m.distance_km * (
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'maintenance_per_km') +
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'energy_per_km')
    )) +
     (COALESCE(i.raw_intervention_seconds, 0) / 60.0 *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'operator_per_minute')) +
     (COALESCE(i.raw_intervention_count, 0) *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'handoff_per_intervention')) +
     (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'overhead_per_mission')
    ) AS variable_cost

FROM missions_clean m
LEFT JOIN intervention_totals i ON m.mission_id = i.mission_id
JOIN battery_logs b ON m.mission_id = b.mission_id;
"""

# Create view via programmatic helper
create_view_from_sql("v_mission_facts", v_mission_facts_sql)

# Programmatic validation checks
m_clean_count = database_connection.execute("SELECT COUNT(*) FROM missions_clean").fetchone()[0]
facts_count = database_connection.execute("SELECT COUNT(*) FROM v_mission_facts").fetchone()[0]
null_interventions_count = database_connection.execute("SELECT COUNT(*) FROM v_mission_facts WHERE intervention_count IS NULL").fetchone()[0]

total_ints_in_facts = database_connection.execute("SELECT SUM(intervention_count) FROM v_mission_facts").fetchone()[0]
total_ints_in_clean_missions = database_connection.execute("""
    SELECT COUNT(*)
    FROM interventions
    WHERE mission_id IN (SELECT mission_id FROM missions_clean)
""").fetchone()[0]

print("\n=== V_MISSION_FACTS INTEGRITY AUDIT ===")
print(f"Row Count Match: {'PASS' if m_clean_count == facts_count else 'FAIL'} (Missions Clean: {m_clean_count}, Facts View: {facts_count})")
print(f"Zero NULL Interventions: {'PASS' if null_interventions_count == 0 else 'FAIL'} (Null Count: {null_interventions_count})")
print(f"Interventions Sum Match: {'PASS' if total_ints_in_facts == total_ints_in_clean_missions else 'FAIL'} (Facts Sum: {total_ints_in_facts}, Raw Valid Sum: {total_ints_in_clean_missions})")

# Output 5 sample records for audit visualization
print("\n=== SAMPLE ROWS PREVIEW ===")
sample_df = database_connection.execute("SELECT * FROM v_mission_facts LIMIT 5").fetchdf()
display(sample_df)


View created: v_mission_facts           | Row Count: 44038

=== V_MISSION_FACTS INTEGRITY AUDIT ===
Row Count Match: PASS (Missions Clean: 44038, Facts View: 44038)
Zero NULL Interventions: PASS (Null Count: 0)
Interventions Sum Match: PASS (Facts Sum: 3011, Raw Valid Sum: 3011)

=== SAMPLE ROWS PREVIEW ===


,mission_id,robot_id,zone,start_time,start_date,start_hour,is_weekend,distance_km,promised_eta,actual_arrival,...,charge_start_pct,charge_end_pct,charge_used_pct,is_completed,is_on_time,has_zero_interventions,km_cost,intervention_cost,overhead_cost,variable_cost
0,M000001,R011,Riverside,2026-07-01 08:00:43,2026-07-01,8,0,0.83,2026-07-01 08:14:46,2026-07-01 08:10:14,...,42.3,37.7,4.6,1,1,1,0.0913,0.0,0.4,0.4913
1,M000002,R037,Riverside,2026-07-01 08:02:29,2026-07-01,8,0,0.91,2026-07-01 08:17:25,2026-07-01 08:15:38,...,60.9,55.7,5.2,1,1,1,0.1001,0.0,0.4,0.5001
2,M000003,R041,Suburb,2026-07-01 08:02:42,2026-07-01,8,0,1.31,2026-07-01 08:21:59,2026-07-01 08:17:32,...,69.0,61.9,7.1,1,1,1,0.1441,0.0,0.4,0.5441
3,M000004,R023,Downtown,2026-07-01 08:07:08,2026-07-01,8,0,1.04,2026-07-01 08:23:29,2026-07-01 08:20:43,...,43.9,35.9,8.0,1,1,1,0.1144,0.0,0.4,0.5144
4,M000005,R002,Old Town,2026-07-01 08:07:59,2026-07-01,8,0,1.22,2026-07-01 08:26:18,2026-07-01 08:32:12,...,26.0,19.4,6.6,1,0,1,0.1342,0.0,0.4,0.5342


### CELL C4: Create Daily Slice Level View (`v_daily_slice`)

We will now register the analytical view `v_daily_slice` using our `create_view_from_sql` engine. This view collapses our trip-level fact records into daily operational segments structured by date, geographic zone, firmware version, and weather condition.

This view relies exclusively on **additive metric columns** (sums and counts). To calculate compound rates (such as average cost per delivery or autonomy percentages) at runtime over varying dimensions, we must always use the **ratio of sums** (e.g., `SUM(total_cost) / SUM(missions)`) rather than an average of pre-calculated ratios, which would introduce mathematically corrupting sample weight biases.

In [14]:
# CELL C4: View v_daily_slice
# This cell registers the daily-slice SQL aggregation view, computes proportional daily depreciation costs,
# saves the query to a SQL file, and executes programmatic integrity audits.

v_daily_slice_sql = """
-- v_daily_slice
-- This view aggregates mission facts into daily, zonal, software, and weather segments.
-- It contains ONLY additive sum metrics to allow mathematically sound downstream aggregations (ratio of sums, not average of ratios).

WITH aggregated_slices AS (
    -- Step 1: Group facts view by keys and sum up variables
    SELECT
        start_date,
        zone,
        software_version,
        weather,
        ANY_VALUE(is_weekend) AS is_weekend, -- Group is aligned since all missions on a day share is_weekend status
        COUNT(*) AS missions,
        SUM(is_completed) AS completed_deliveries,
        SUM(has_zero_interventions) AS zero_intervention_missions,
        SUM(intervention_count) AS interventions,
        SUM(intervention_seconds) AS intervention_seconds,
        SUM(distance_km) AS km,
        SUM(is_on_time) AS on_time_deliveries,
        SUM(variable_cost) AS variable_cost
    FROM v_mission_facts
    GROUP BY start_date, zone, software_version, weather
),
depreciation_allocation AS (
    -- Step 2: Sum up total daily missions to share daily depreciation cost proportionally
    SELECT
        *,
        SUM(missions) OVER(PARTITION BY start_date) AS total_daily_missions
    FROM aggregated_slices
)
SELECT
    start_date,
    zone,
    software_version,
    weather,
    is_weekend,
    missions,
    completed_deliveries,
    zero_intervention_missions,
    interventions,
    intervention_seconds,
    km,
    on_time_deliveries,
    variable_cost,

    -- Step 3: Compute depreciation cost proportionally.
    -- Daily fleet cost = Total robots * Depreciation rate per robot per day
    -- Slice cost = Daily fleet cost * (Slice missions / Daily total missions)
    ((
        (SELECT COUNT(*) FROM robots) *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'robot_depreciation_per_day')
    ) * (CAST(missions AS DOUBLE) / total_daily_missions)) AS depreciation_cost,

    -- Step 4: Calculate total cost
    (variable_cost + (
        (SELECT COUNT(*) FROM robots) *
        (SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'robot_depreciation_per_day')
    ) * (CAST(missions AS DOUBLE) / total_daily_missions)) AS total_cost

FROM depreciation_allocation;
"""

# Create view via programmatic helper
create_view_from_sql("v_daily_slice", v_daily_slice_sql)

# Programmatic validation checks
m_clean_count = database_connection.execute("SELECT COUNT(*) FROM missions_clean").fetchone()[0]
slice_missions_sum = database_connection.execute("SELECT SUM(missions) FROM v_daily_slice").fetchone()[0]

# Expected total fleet depreciation calculation: total days * total robots * depreciation price per day
total_days = database_connection.execute("SELECT COUNT(DISTINCT start_date) FROM v_daily_slice").fetchone()[0]
total_robots = database_connection.execute("SELECT COUNT(*) FROM robots").fetchone()[0]
deprec_rate = database_connection.execute("SELECT CAST(value AS DOUBLE) FROM cost_assumptions WHERE item = 'robot_depreciation_per_day'").fetchone()[0]
expected_deprec_cost = total_days * total_robots * deprec_rate

actual_deprec_sum = database_connection.execute("SELECT SUM(depreciation_cost) FROM v_daily_slice").fetchone()[0]

print("\n=== V_DAILY_SLICE INTEGRITY AUDIT ===")
print(f"Total Missions Match: {'PASS' if m_clean_count == slice_missions_sum else 'FAIL'} (Missions Clean: {m_clean_count}, Slice Sum: {slice_missions_sum})")
print(f"Depreciation Cost Match: {'PASS' if abs(expected_deprec_cost - actual_deprec_sum) < 1e-4 else 'FAIL'}")
print(f"  - Expected Depreciation Cost: ${expected_deprec_cost:.2f} ({total_days} days * {total_robots} robots * ${deprec_rate:.2f}/day)")
print(f"  - Actual Distributed Sum:   ${actual_deprec_sum:.2f}")

# Display preview of 5 sample rows
print("\n=== SAMPLE ROWS PREVIEW ===")
sample_df = database_connection.execute("SELECT * FROM v_daily_slice LIMIT 5").fetchdf()
display(sample_df)


View created: v_daily_slice             | Row Count: 600

=== V_DAILY_SLICE INTEGRITY AUDIT ===
Total Missions Match: PASS (Missions Clean: 44038, Slice Sum: 44038)
Depreciation Cost Match: PASS
  - Expected Depreciation Cost: $31500.00 (90 days * 50 robots * $7.00/day)
  - Actual Distributed Sum:   $31500.00

=== SAMPLE ROWS PREVIEW ===


,start_date,zone,software_version,weather,is_weekend,missions,completed_deliveries,zero_intervention_missions,interventions,intervention_seconds,km,on_time_deliveries,variable_cost,depreciation_cost,total_cost
0,2026-07-03,Suburb,v2.1,clear,0,91,90.0,88.0,3.0,127.0,154.49,66.0,59.1639,66.631799,125.795699
1,2026-07-03,Old Town,v2.1,clear,0,57,56.0,52.0,5.0,303.0,108.75,5.0,45.2925,41.736402,87.028902
2,2026-07-03,Campus,v2.1,clear,0,117,114.0,110.0,7.0,369.0,203.80,86.0,83.4080,85.669456,169.077456
3,2026-07-03,Downtown,v2.1,clear,0,118,116.0,113.0,5.0,278.0,201.12,85.0,79.6032,86.401674,166.004874
4,2026-07-03,Riverside,v2.1,clear,0,95,92.0,91.0,4.0,201.0,154.16,71.0,62.9676,69.560669,132.528269


### CELL C5: Daily Fleet Performance KPI View (`v_kpi_daily`)

We will now construct and register `v_kpi_daily` in DuckDB. This view aggregates daily metrics and derives crucial performance tracking indicators overall and inside moving windows.

#### Why We Never Average Pre-calculated Ratios
To track multi-day performance correctly (such as rolling 7-day metrics), we **never** average daily ratios. For example, averaging Tuesday's cost per delivery with Wednesday's cost per delivery yields a mathematically corrupt figure if Tuesday had 10 deliveries and Wednesday had 500 deliveries. To avoid this sample weight bias, we must always partition and aggregate the absolute sums of numerators and denominators over the temporal window first, and then divide the two totals at the final step.

In [15]:
# CELL C5: View v_kpi_daily
# This cell registers the daily fleet-level KPI aggregation view,
# computes rolling 7-day KPIs without introducing sample weight biases,
# saves the view query to a SQL file, and prints detailed KPI validations.

v_kpi_daily_sql = """
-- v_kpi_daily
-- This view rolls up daily slice records into exactly one row per calendar day.
-- It implements safe divisions and rolling 7-day averages of operational metrics.

WITH daily_base AS (
    -- Step 1: Sum up all slices per start_date to establish daily absolute totals
    SELECT
        start_date,
        ANY_VALUE(is_weekend) AS is_weekend,
        SUM(missions) AS missions,
        SUM(completed_deliveries) AS completed_deliveries,
        SUM(zero_intervention_missions) AS zero_intervention_missions,
        SUM(interventions) AS interventions,
        SUM(intervention_seconds) AS intervention_seconds,
        SUM(km) AS km,
        SUM(on_time_deliveries) AS on_time_deliveries,
        SUM(total_cost) AS total_cost
    FROM v_daily_slice
    GROUP BY start_date
),
rolling_sums AS (
    -- Step 2: Calculate window-based numerator and denominator sums over 7 days
    SELECT
        *,
        -- Window frames defined to look back 6 preceding rows + current row (chronological order)
        SUM(interventions) OVER(ORDER BY start_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_interventions_7d,
        SUM(km) OVER(ORDER BY start_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_km_7d,
        SUM(on_time_deliveries) OVER(ORDER BY start_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_on_time_7d,
        SUM(completed_deliveries) OVER(ORDER BY start_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_completed_7d,
        SUM(total_cost) OVER(ORDER BY start_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_cost_7d
    FROM daily_base
)
SELECT
    start_date,
    is_weekend,
    missions,
    completed_deliveries,
    zero_intervention_missions,
    interventions,
    km,
    on_time_deliveries,
    total_cost,

    -- Step 3: Compute daily single-day metrics using NULLIF to prevent division-by-zero crashes
    (CAST(zero_intervention_missions AS DOUBLE) / NULLIF(missions, 0)) AS autonomy_rate,
    (CAST(interventions AS DOUBLE) / NULLIF(km, 0) * 1000.0) AS interventions_per_1000km,
    (CAST(on_time_deliveries AS DOUBLE) / NULLIF(completed_deliveries, 0)) AS on_time_pct,
    (total_cost / NULLIF(completed_deliveries, 0)) AS cost_per_delivery,

    -- Step 4: Compute rolling 7-day KPIs from aggregate sums to maintain mathematical weights
    (CAST(rolling_interventions_7d AS DOUBLE) / NULLIF(rolling_km_7d, 0) * 1000.0) AS interventions_per_1000km_7d,
    (CAST(rolling_on_time_7d AS DOUBLE) / NULLIF(rolling_completed_7d, 0)) AS on_time_pct_7d,
    (rolling_cost_7d / NULLIF(rolling_completed_7d, 0)) AS cost_per_delivery_7d

FROM rolling_sums;
"""

# Create view via programmatic helper
create_view_from_sql("v_kpi_daily", v_kpi_daily_sql)

# Programmatic validation checks
print("\n=== V_KPI_DAILY FIRST 5 ROWS PREVIEW ===")
kpi_sample_df = database_connection.execute("SELECT * FROM v_kpi_daily ORDER BY start_date LIMIT 5").fetchdf()
display(kpi_sample_df)

# Retrieve bounds to verify reasonable data values
bounds_df = database_connection.execute("""
    SELECT
        MIN(autonomy_rate) AS min_autonomy, MAX(autonomy_rate) AS max_autonomy,
        MIN(interventions_per_1000km) AS min_int_rate, MAX(interventions_per_1000km) AS max_int_rate,
        MIN(on_time_pct) AS min_otp, MAX(on_time_pct) AS max_otp,
        MIN(cost_per_delivery) AS min_cost, MAX(cost_per_delivery) AS max_cost
    FROM v_kpi_daily
""").fetchdf()

print("\n=== DAILY KPI EXTREMES (MIN / MAX BOUNDS) ===")
display(bounds_df)

# Retrieve whole-period metrics by dividing sums directly
global_totals = database_connection.execute("""
    SELECT
        SUM(missions) AS total_missions,
        SUM(completed_deliveries) AS total_completed,
        SUM(zero_intervention_missions) AS total_zero_int,
        SUM(interventions) AS total_interventions,
        SUM(km) AS total_km,
        SUM(on_time_deliveries) AS total_on_time,
        SUM(total_cost) AS absolute_cost
    FROM v_kpi_daily
""").fetchone()

t_miss, t_compl, t_zero, t_int, t_km, t_ot, t_cost = global_totals

overall_autonomy = t_zero / t_miss
overall_int_rate = t_int / t_km * 1000.0
overall_otp = t_ot / t_compl
overall_cost_per_del = t_cost / t_compl

print("\n=== WHOLE-PERIOD OPERATION PERFORMANCE METRICS (SUM RATIOS) ===")
print(f"Total Operational Days Evaluated: {database_connection.execute('SELECT COUNT(*) FROM v_kpi_daily').fetchone()[0]} Days")
print(f"Overall Autonomy Rate:           {overall_autonomy:.2%} (Trips without assistance)")
print(f"Overall Human Interventions Rate: {overall_int_rate:.2f} assists per 1,000 km traveled")
print(f"Overall On-Time SLA Performance:  {overall_otp:.2%} of completed deliveries")
print(f"Overall Weighted Unit Cost:       ${overall_cost_per_del:.2f} per successful delivery")


View created: v_kpi_daily               | Row Count: 90

=== V_KPI_DAILY FIRST 5 ROWS PREVIEW ===


,start_date,is_weekend,missions,completed_deliveries,zero_intervention_missions,interventions,km,on_time_deliveries,total_cost,autonomy_rate,interventions_per_1000km,on_time_pct,cost_per_delivery,interventions_per_1000km_7d,on_time_pct_7d,cost_per_delivery_7d
0,2026-07-01,0,458.0,451.0,436.0,24.0,775.81,295.0,671.3591,0.951965,30.935409,0.654102,1.488601,30.935409,0.654102,1.488601
1,2026-07-02,0,469.0,462.0,444.0,26.0,815.08,276.0,681.0688,0.946695,31.898709,0.597403,1.474175,31.428949,0.625411,1.481301
2,2026-07-03,0,478.0,468.0,454.0,24.0,822.32,313.0,680.4352,0.949791,29.185718,0.668803,1.453921,30.664551,0.640116,1.472023
3,2026-07-04,1,596.0,587.0,562.0,36.0,1007.88,369.0,771.7968,0.942953,35.718538,0.628620,1.314816,32.153495,0.636687,1.425132
4,2026-07-05,1,539.0,534.0,501.0,40.0,899.73,349.0,743.9303,0.929499,44.457782,0.653558,1.393128,34.715633,0.640288,1.418301



=== DAILY KPI EXTREMES (MIN / MAX BOUNDS) ===


,min_autonomy,max_autonomy,min_int_rate,max_int_rate,min_otp,max_otp,min_cost,max_cost
0,0.881166,0.972973,18.1854,79.556796,0.206349,0.713604,1.24329,1.78358



=== WHOLE-PERIOD OPERATION PERFORMANCE METRICS (SUM RATIOS) ===
Total Operational Days Evaluated: 90 Days
Overall Autonomy Rate:           93.54% (Trips without assistance)
Overall Human Interventions Rate: 40.21 assists per 1,000 km traveled
Overall On-Time SLA Performance:  53.85% of completed deliveries
Overall Weighted Unit Cost:       $1.47 per successful delivery


### CELL C6: Segmented Breakdown Views (`v_by_version`, `v_by_zone`, `v_by_weather`, `v_by_daytype`, `v_by_hour`)

We will now register five dimensional breakdown views in DuckDB. These views slice our additive metrics over software firmware versions, geographic zones, weather conditions, calendar types (weekend vs. weekday), and hourly time blocks.

All KPI divisions are executed safely using `NULLIF` and follow the *ratio-of-sums* standard to prevent structural weight bias.

In [16]:
# CELL C6: Breakdown Views and Consistency Audits
# This cell registers the five requested segmented breakdown views in DuckDB,
# writes their queries to our SQL views archive, and performs cross-view verification audits.

# =====================================================================
# View 1: v_by_version
# =====================================================================
v_by_version_sql = """
-- v_by_version
-- Segmented operational metrics and KPIs grouped by software firmware version.

SELECT
    software_version,
    SUM(missions) AS missions,
    SUM(completed_deliveries) AS completed_deliveries,
    SUM(zero_intervention_missions) AS zero_intervention_missions,
    SUM(interventions) AS interventions,
    SUM(km) AS km,
    SUM(on_time_deliveries) AS on_time_deliveries,
    SUM(total_cost) AS total_cost,

    -- KPIs calculated as ratios of sums
    (CAST(SUM(zero_intervention_missions) AS DOUBLE) / NULLIF(SUM(missions), 0)) AS autonomy_rate,
    (CAST(SUM(interventions) AS DOUBLE) / NULLIF(SUM(km), 0) * 1000.0) AS interventions_per_1000km,
    (CAST(SUM(on_time_deliveries) AS DOUBLE) / NULLIF(SUM(completed_deliveries), 0)) AS on_time_pct,
    (SUM(total_cost) / NULLIF(SUM(completed_deliveries), 0)) AS cost_per_delivery
FROM v_daily_slice
GROUP BY software_version;
"""

# =====================================================================
# View 2: v_by_zone
# =====================================================================
v_by_zone_sql = """
-- v_by_zone
-- Segmented operational metrics and KPIs grouped by geographic zone.

SELECT
    zone,
    SUM(missions) AS missions,
    SUM(completed_deliveries) AS completed_deliveries,
    SUM(zero_intervention_missions) AS zero_intervention_missions,
    SUM(interventions) AS interventions,
    SUM(km) AS km,
    SUM(on_time_deliveries) AS on_time_deliveries,
    SUM(total_cost) AS total_cost,

    -- KPIs calculated as ratios of sums
    (CAST(SUM(zero_intervention_missions) AS DOUBLE) / NULLIF(SUM(missions), 0)) AS autonomy_rate,
    (CAST(SUM(interventions) AS DOUBLE) / NULLIF(SUM(km), 0) * 1000.0) AS interventions_per_1000km,
    (CAST(SUM(on_time_deliveries) AS DOUBLE) / NULLIF(SUM(completed_deliveries), 0)) AS on_time_pct,
    (SUM(total_cost) / NULLIF(SUM(completed_deliveries), 0)) AS cost_per_delivery
FROM v_daily_slice
GROUP BY zone;
"""

# =====================================================================
# View 3: v_by_weather
# =====================================================================
v_by_weather_sql = """
-- v_by_weather
-- Segmented operational metrics and KPIs grouped by weather status.

SELECT
    weather,
    SUM(missions) AS missions,
    SUM(completed_deliveries) AS completed_deliveries,
    SUM(zero_intervention_missions) AS zero_intervention_missions,
    SUM(interventions) AS interventions,
    SUM(km) AS km,
    SUM(on_time_deliveries) AS on_time_deliveries,
    SUM(total_cost) AS total_cost,

    -- KPIs calculated as ratios of sums
    (CAST(SUM(zero_intervention_missions) AS DOUBLE) / NULLIF(SUM(missions), 0)) AS autonomy_rate,
    (CAST(SUM(interventions) AS DOUBLE) / NULLIF(SUM(km), 0) * 1000.0) AS interventions_per_1000km,
    (CAST(SUM(on_time_deliveries) AS DOUBLE) / NULLIF(SUM(completed_deliveries), 0)) AS on_time_pct,
    (SUM(total_cost) / NULLIF(SUM(completed_deliveries), 0)) AS cost_per_delivery
FROM v_daily_slice
GROUP BY weather;
"""

# =====================================================================
# View 4: v_by_daytype
# =====================================================================
v_by_daytype_sql = """
-- v_by_daytype
-- Segmented operational metrics and KPIs grouped by day type (weekend vs weekday).

SELECT
    is_weekend,
    CASE WHEN is_weekend = 1 THEN 'weekend' ELSE 'weekday' END AS day_type,
    SUM(missions) AS missions,
    SUM(completed_deliveries) AS completed_deliveries,
    SUM(zero_intervention_missions) AS zero_intervention_missions,
    SUM(interventions) AS interventions,
    SUM(km) AS km,
    SUM(on_time_deliveries) AS on_time_deliveries,
    SUM(total_cost) AS total_cost,

    -- KPIs calculated as ratios of sums
    (CAST(SUM(zero_intervention_missions) AS DOUBLE) / NULLIF(SUM(missions), 0)) AS autonomy_rate,
    (CAST(SUM(interventions) AS DOUBLE) / NULLIF(SUM(km), 0) * 1000.0) AS interventions_per_1000km,
    (CAST(SUM(on_time_deliveries) AS DOUBLE) / NULLIF(SUM(completed_deliveries), 0)) AS on_time_pct,
    (SUM(total_cost) / NULLIF(SUM(completed_deliveries), 0)) AS cost_per_delivery
FROM v_daily_slice
GROUP BY is_weekend;
"""

# =====================================================================
# View 5: v_by_hour
# =====================================================================
v_by_hour_sql = """
-- v_by_hour
-- Segmented hourly/spatial operational profile grouped by start_hour and zone.
-- Cost columns are excluded as daily depreciation is not calculated hourly.

SELECT
    start_hour,
    zone,
    COUNT(*)                     AS missions,                    -- ALL missions, failed ones too
    SUM(is_completed)            AS completed_deliveries,
    SUM(is_on_time)              AS on_time_deliveries,
    SUM(intervention_count)      AS interventions,
    SUM(distance_km)             AS km,
    SUM(has_zero_interventions)  AS zero_intervention_missions
    -- keep your three ratio columns below, unchanged:
    -- on_time_pct divides by completed_deliveries; autonomy_rate divides by missions
FROM v_mission_facts
GROUP BY start_hour, zone;
"""

# Register all five views with our helper function
create_view_from_sql("v_by_version", v_by_version_sql)
create_view_from_sql("v_by_zone", v_by_zone_sql)
create_view_from_sql("v_by_weather", v_by_weather_sql)
create_view_from_sql("v_by_daytype", v_by_daytype_sql)
create_view_from_sql("v_by_hour", v_by_hour_sql)

# Print Full Previews of Views 1-4
print("\n=== VIEW 1: V_BY_VERSION ===")
display(database_connection.execute("SELECT * FROM v_by_version").fetchdf())

print("\n=== VIEW 2: V_BY_ZONE ===")
display(database_connection.execute("SELECT * FROM v_by_zone").fetchdf())

print("\n=== VIEW 3: V_BY_WEATHER ===")
display(database_connection.execute("SELECT * FROM v_by_weather").fetchdf())

print("\n=== VIEW 4: V_BY_DAYTYPE ===")
display(database_connection.execute("SELECT * FROM v_by_daytype").fetchdf())

# Print First 10 Rows of View 5
print("\n=== VIEW 5: V_BY_HOUR (FIRST 10 ROWS PREVIEW) ===")
display(database_connection.execute("SELECT * FROM v_by_hour ORDER BY start_hour, zone LIMIT 10").fetchdf())

# Cross-View Audit: Verify total kilometers across all views are mathematically identical
print("\n=== CROSS-VIEW KILOMETERS CONSISTENCY AUDIT ===")
km_report = database_connection.execute("""
    SELECT
        (SELECT SUM(km) FROM v_daily_slice) AS daily_slice_km,
        (SELECT SUM(km) FROM v_by_version) AS by_version_km,
        (SELECT SUM(km) FROM v_by_zone) AS by_zone_km,
        (SELECT SUM(km) FROM v_by_weather) AS by_weather_km,
        (SELECT SUM(km) FROM v_by_daytype) AS by_daytype_km,
        (SELECT SUM(km) FROM v_by_hour) AS by_hour_km
""").fetchdf()

display(km_report)

# Verify that all sums match to 4 decimal places
sums = km_report.iloc[0].values
if all(abs(s - sums[0]) < 1e-4 for s in sums):
    print("SUCCESS: All views report perfectly identical fleet distances!")
else:
    print("WARNING: Distance discrepancy detected across views. Check grouping fields or filters.")


View created: v_by_version              | Row Count: 3
View created: v_by_zone                 | Row Count: 5
View created: v_by_weather              | Row Count: 2
View created: v_by_daytype              | Row Count: 2
View created: v_by_hour                 | Row Count: 70

=== VIEW 1: V_BY_VERSION ===


,software_version,missions,completed_deliveries,zero_intervention_missions,interventions,km,on_time_deliveries,total_cost,autonomy_rate,interventions_per_1000km,on_time_pct,cost_per_delivery
0,v2.1,14564.0,14286.0,13677.0,935.0,24943.67,7932.0,20985.683700,0.939096,37.484460,0.555229,1.468968
1,v2.2,22098.0,21712.0,20846.0,1311.0,37500.93,11528.0,31383.030029,0.943343,34.959133,0.530951,1.445423
2,v2.3,7376.0,7255.0,6672.0,765.0,12436.16,3830.0,11108.809871,0.904555,61.514165,0.527912,1.531194



=== VIEW 2: V_BY_ZONE ===


,zone,missions,completed_deliveries,zero_intervention_missions,interventions,km,on_time_deliveries,total_cost,autonomy_rate,interventions_per_1000km,on_time_pct,cost_per_delivery
0,Old Town,5291.0,5202.0,4690.0,659.0,8961.60,274.0,8214.163216,0.886411,73.535976,0.052672,1.579039
1,Suburb,7940.0,7805.0,7501.0,459.0,13592.26,4705.0,11297.343977,0.944710,33.769219,0.602819,1.447450
2,Campus,11042.0,10860.0,10407.0,664.0,18670.43,6557.0,15722.601037,0.942492,35.564259,0.603775,1.447753
3,Downtown,11032.0,10835.0,10368.0,694.0,18808.33,6524.0,15779.408995,0.939811,36.898544,0.602123,1.456337
4,Riverside,8733.0,8551.0,8229.0,535.0,14848.14,5230.0,12464.006374,0.942288,36.031449,0.611624,1.457608



=== VIEW 3: V_BY_WEATHER ===


...


=== VIEW 5: V_BY_HOUR (FIRST 10 ROWS PREVIEW) ===


,start_hour,zone,missions,completed_deliveries,on_time_deliveries,interventions,km,zero_intervention_missions
0,8,Campus,448,441.0,270.0,33.0,756.00,417.0
1,8,Downtown,504,496.0,316.0,29.0,856.64,477.0
2,8,Old Town,223,222.0,9.0,20.0,388.93,203.0
3,8,Riverside,380,370.0,240.0,21.0,605.78,360.0
4,8,Suburb,354,347.0,218.0,22.0,607.77,333.0
5,9,Campus,460,453.0,283.0,35.0,756.22,426.0
6,9,Downtown,481,471.0,285.0,30.0,859.36,452.0
7,9,Old Town,239,235.0,14.0,35.0,426.25,206.0
8,9,Riverside,367,360.0,224.0,29.0,622.54,341.0
9,9,Suburb,343,338.0,207.0,22.0,615.10,322.0



=== CROSS-VIEW KILOMETERS CONSISTENCY AUDIT ===


,daily_slice_km,by_version_km,by_zone_km,by_weather_km,by_daytype_km,by_hour_km
0,74880.76,74880.76,74880.76,74880.76,74880.76,74880.76


SUCCESS: All views report perfectly identical fleet distances!


### CELL C7: Fleet Assets & Telemetry Reason Breakdown Views

In this section, we register `v_by_robot` and `v_intervention_reasons` to analyze hardware degradation patterns and map diagnostic codes across firmware releases, environment conditions, and zones.

* **Asset Degradation (`v_by_robot`)**: Groups trip facts by individual physical robot IDs and joins their master metadata profile to compute distance-weighted battery depletion signatures and safety failures.
* **Categorical Root-Causes (`v_intervention_reasons`)**: Filters interventions to ensure only telemetry rows belonging to cleaned, validated missions are evaluated.

In [17]:
# CELL C7: Robot and reason views
# This cell creates the segmented views for physical hardware assets and telemetry reasons,
# registers them in DuckDB, and performs diagnostic checks.

# =====================================================================
# View 1: v_by_robot
# =====================================================================
v_by_robot_sql = """
-- v_by_robot
-- Detailed operational performance and battery consumption rate profiling per physical robot asset.

SELECT
    f.robot_id,
    r.battery_health,
    COUNT(f.mission_id) AS missions,
    SUM(f.distance_km) AS km,
    SUM(f.intervention_count) AS interventions,

    -- Safe ratio of sums KPI calculations
    (CAST(SUM(f.intervention_count) AS DOUBLE) / NULLIF(SUM(f.distance_km), 0) * 1000.0) AS interventions_per_1000km,
    (SUM(f.charge_used_pct) / NULLIF(SUM(f.distance_km), 0)) AS avg_charge_used_per_km,

    SUM(CASE WHEN f.failure_reason = 'battery_low' THEN 1 ELSE 0 END) AS battery_failures,
    (CAST(SUM(CASE WHEN f.failure_reason = 'battery_low' THEN 1 ELSE 0 END) AS DOUBLE) / NULLIF(COUNT(f.mission_id), 0)) AS battery_failure_rate,

    RANK() OVER (ORDER BY (SUM(f.charge_used_pct) / NULLIF(SUM(f.distance_km), 0)) DESC) AS drain_rank
FROM v_mission_facts f
JOIN robots r ON f.robot_id = r.robot_id
GROUP BY f.robot_id, r.battery_health;
"""

# =====================================================================
# View 2: v_by_reason
# =====================================================================
v_intervention_reasons_sql = """
-- v_intervention_reasons
-- Telemetry intervention counts and durations segmented by diagnostic code, firmware, zone, and weather.
-- Joining to missions_clean is crucial because it filters out intervention logs corresponding to corrupted
-- or duplicated trips that were removed during our initial metrics sanitization pass.

SELECT
    i.reason,
    m.software_version,
    m.zone,
    m.weather,
    COUNT(*) AS interventions,
    SUM(i.duration_sec) AS total_seconds
FROM interventions i
INNER JOIN missions_clean m ON i.mission_id = m.mission_id
GROUP BY i.reason, m.software_version, m.zone, m.weather;
"""

# Register both views using our programmatic helper
create_view_from_sql("v_by_robot", v_by_robot_sql)
create_view_from_sql("v_intervention_reasons", v_intervention_reasons_sql)

# Audit 1: Print the 8 robots with the worst drain_rank (highest depletion rates first)
print("\n=== TOP 8 DEGRADED ROBOTS BY BATTERY DRAIN RATE (v_by_robot) ===")
top_robots_df = database_connection.execute("""
    SELECT *
    FROM v_by_robot
    WHERE drain_rank <= 8
    ORDER BY drain_rank ASC
""").fetchdf()
display(top_robots_df)

# Audit 2: Print the top 5 failure reasons overall using v_intervention_reasons
print("\n=== TOP 5 HUMAN INTERVENTION TELEMETRY REASONS OVERALL (v_intervention_reasons) ===")
top_reasons_df = database_connection.execute("""
    SELECT
        reason,
        SUM(interventions) AS total_interventions,
        SUM(total_seconds) AS absolute_seconds,
        ROUND(CAST(SUM(total_seconds) AS DOUBLE) / NULLIF(SUM(interventions), 0), 1) AS avg_seconds_per_recovery
    FROM v_intervention_reasons
    GROUP BY reason
    ORDER BY total_interventions DESC
    LIMIT 5
""").fetchdf()
display(top_reasons_df)


View created: v_by_robot                | Row Count: 50
View created: v_intervention_reasons    | Row Count: 179

=== TOP 8 DEGRADED ROBOTS BY BATTERY DRAIN RATE (v_by_robot) ===


,robot_id,battery_health,missions,km,interventions,interventions_per_1000km,avg_charge_used_per_km,battery_failures,battery_failure_rate,drain_rank
0,R007,0.65,851,1451.84,57.0,39.260525,7.691412,5.0,0.005875,1
1,R019,0.65,856,1460.12,57.0,39.037887,7.689847,10.0,0.011682,2
2,R023,0.65,914,1570.52,67.0,42.661029,7.680959,11.0,0.012035,3
3,R044,0.65,886,1530.44,83.0,54.232770,7.675766,8.0,0.009029,4
4,R031,0.65,875,1482.95,44.0,29.670589,7.663643,10.0,0.011429,5
5,R028,0.86,887,1513.17,68.0,44.938771,5.813491,4.0,0.004510,6
6,R005,0.86,832,1404.21,39.0,27.773624,5.813162,2.0,0.002404,7
7,R018,0.86,858,1428.11,60.0,42.013570,5.811597,3.0,0.003497,8



=== TOP 5 HUMAN INTERVENTION TELEMETRY REASONS OVERALL (v_intervention_reasons) ===


,reason,total_interventions,absolute_seconds,avg_seconds_per_recovery
0,crosswalk_confusion,872.0,46683.0,53.5
1,blocked_path,708.0,39886.0,56.3
2,low_confidence,487.0,24864.0,51.1
3,sensor_obstructed,403.0,21620.0,53.6
4,curb_or_door,296.0,14818.0,50.1


### CELL C8: Integrated SQL Compilation & Full Warehouse Rebuild Verification

To ensure our metrics layer can be fully re-deployed or shared across different database instances, we compile all individual views into a single cohesive script: `sql/metric_views.sql`. We then spin up an isolated test database to verify the script executing sequentially does not throw any relational errors and accurately matches our production metrics schema.

In [18]:
# CELL C8: Compile views and test rebuild process
# This cell reads all SQL view files, builds a combined SQL deployment script,
# runs it against a temporary database, and checks consistency.

import os
import duckdb
import pandas as pd
from pathlib import Path

# =====================================================================
# Step 1: Read view files in dependency order and write to metric_views.sql
# =====================================================================
views_order = [
    "missions_clean",
    "v_mission_facts",
    "v_daily_slice",
    "v_kpi_daily",
    "v_by_version",
    "v_by_zone",
    "v_by_weather",
    "v_by_daytype",
    "v_by_hour",
    "v_by_robot",
    "v_intervention_reasons"
]

views_dir = Path(SQL_FOLDER) / "views"
combined_sql_path = Path(SQL_FOLDER) / "metric_views.sql"

# Open the output file for writing the consolidated script
with open(combined_sql_path, "w") as outfile:
    # Write master script header
    outfile.write("-- =====================================================================\n")
    outfile.write("-- METRIC VIEWS LAYER - COMBINED DEPLOYMENT SCRIPT\n")
    outfile.write("-- Consolidated view definitions for fleet operations metrics layer\n")
    outfile.write("-- =====================================================================\n\n")

    for view_name in views_order:
        file_path = views_dir / f"{view_name}.sql"

        # Read raw view definition from file
        with open(file_path, "r") as infile:
            raw_sql = infile.read().strip()

        # Append separator header for clarity and structural isolation
        outfile.write(f"-- -----------------------------------------------------\n")
        outfile.write(f"-- View Segment: {view_name}\n")
        outfile.write(f"-- -----------------------------------------------------\n")

        # FIX: Ensure we prepend the view creation definition header
        if not raw_sql.upper().startswith("CREATE"):
            raw_sql = f"CREATE OR REPLACE VIEW {view_name} AS\n" + raw_sql

        # Ensure the query statements are terminated clean with a semicolon
        if not raw_sql.endswith(";"):
            raw_sql += ";"

        outfile.write(raw_sql + "\n\n")

print(f"Consolidated views successfully written to {combined_sql_path}")

# =====================================================================
# Step 2: Establish the isolated test database environment
# =====================================================================
test_db_file = "test_rebuild.duckdb"
if os.path.exists(test_db_file):
    os.remove(test_db_file)

test_conn = duckdb.connect(test_db_file)

# Load the baseline data CSV tables exactly as in C1
robots_test_df = pd.read_csv(os.path.join(DATA_FOLDER, "robots.csv"), parse_dates=["purchase_date"])
robots_test_df["purchase_date"] = robots_test_df["purchase_date"].dt.date

missions_test_df = pd.read_csv(os.path.join(DATA_FOLDER, "missions.csv"), parse_dates=["start_time", "promised_eta", "actual_arrival"])
interventions_test_df = pd.read_csv(os.path.join(DATA_FOLDER, "interventions.csv"), parse_dates=["intervention_time"])
battery_logs_test_df = pd.read_csv(os.path.join(DATA_FOLDER, "battery_logs.csv"))
cost_assumptions_test_df = pd.read_csv(os.path.join(DATA_FOLDER, "cost_assumptions.csv"))

# Register physical base tables in test database connection using CREATE OR REPLACE TABLE to avoid CatalogErrors
test_conn.execute("CREATE OR REPLACE TABLE robots AS SELECT * FROM robots_test_df")
test_conn.execute("CREATE OR REPLACE TABLE missions AS SELECT * FROM missions_test_df")
test_conn.execute("CREATE OR REPLACE TABLE interventions AS SELECT * FROM interventions_test_df")
test_conn.execute("CREATE OR REPLACE TABLE battery_logs AS SELECT * FROM battery_logs_test_df")
test_conn.execute("CREATE OR REPLACE TABLE cost_assumptions AS SELECT * FROM cost_assumptions_test_df")

# =====================================================================
# Step 3 (fixed): Run the combined metric_views.sql one statement at a time
# =====================================================================

# Read the whole combined SQL file into one text string
with open(combined_sql_path, "r") as sql_script:
    full_compiled_script = sql_script.read()

# Remove comment-only lines, so a semicolon inside a comment can never
# cut a statement in the wrong place
script_lines_without_comments = []
for script_line in full_compiled_script.splitlines():
    if not script_line.strip().startswith("--"):
        script_lines_without_comments.append(script_line)
script_without_comments = "\n".join(script_lines_without_comments)

# Split the script into separate statements wherever a semicolon ends one
sql_statements = []
for statement_text in script_without_comments.split(";"):
    if statement_text.strip() != "":
        sql_statements.append(statement_text.strip())

print(f"Found {len(sql_statements)} SQL statements (expected {len(views_order)}).")

# Run each statement on its own, so a real error tells us which view broke
for statement_number, statement_text in enumerate(sql_statements, start=1):
    try:
        test_conn.execute(statement_text)
    except Exception as error_message:
        print(f"Statement {statement_number} failed.")
        print("It starts with:", statement_text[:200])
        raise error_message

# Confirm the views now exist in the test database
created_view_names = test_conn.execute(
    "SELECT view_name FROM duckdb_views() WHERE NOT internal"
).df()["view_name"].tolist()
print("Views created in the test database:", len(created_view_names))

Consolidated views successfully written to sql/metric_views.sql
Found 11 SQL statements (expected 11).
Views created in the test database: 11


In [19]:
# =====================================================================
# CELL C9: Ten practice questions answered in SQL
# Each question has: a comment block, one SQL query, a printed result,
# and one plain-English "Answer:" sentence worked out from the result.
# All queries read from the views we built in C2 to C7.
# =====================================================================

import pandas as pd

# Show decimals neatly in every printed table (2 decimal places)
pd.set_option("display.float_format", lambda number: f"{number:,.2f}")


# ---------------------------------------------------------------------
# QUESTION 1: How many missions, km and completed deliveries are there?
# ---------------------------------------------------------------------
# SUM adds up a column. We read from v_daily_slice, which already holds
# one row per day, zone, version and weather, so summing gives the totals.
question_1_sql = """
SELECT
    SUM(missions)              AS total_missions,
    ROUND(SUM(km), 1)          AS total_km,
    SUM(completed_deliveries)  AS total_completed_deliveries
FROM v_daily_slice
"""
question_1_result = database_connection.execute(question_1_sql).df()
print("Q1: Totals for the whole period")
print(question_1_result.to_string(index=False))
print(
    f"Answer: the fleet ran {int(question_1_result['total_missions'][0]):,} missions "
    f"over {question_1_result['total_km'][0]:,.0f} km and completed "
    f"{int(question_1_result['total_completed_deliveries'][0]):,} deliveries.\n"
)


# ---------------------------------------------------------------------
# QUESTION 2: Interventions per 1,000 km for each software version?
# ---------------------------------------------------------------------
# v_by_version already divides interventions by km (a ratio of sums),
# so we just select the column and sort by version name.
question_2_sql = """
SELECT
    software_version,
    interventions,
    ROUND(km, 1)                        AS km,
    ROUND(interventions_per_1000km, 1)  AS interventions_per_1000km
FROM v_by_version
ORDER BY software_version
"""
question_2_result = database_connection.execute(question_2_sql).df()
print("Q2: Interventions per 1,000 km by software version")
print(question_2_result.to_string(index=False))
worst_version_row = question_2_result.sort_values(
    "interventions_per_1000km", ascending=False
).iloc[0]
print(
    f"Answer: {worst_version_row['software_version']} is the worst version, with "
    f"{worst_version_row['interventions_per_1000km']:.1f} interventions per 1,000 km.\n"
)


# ---------------------------------------------------------------------
# QUESTION 3: Which five robots have the highest battery use per km?
# ---------------------------------------------------------------------
# ORDER BY ... DESC puts the biggest value first.
# LIMIT 5 keeps only the first five rows.
question_3_sql = """
SELECT
    robot_id,
    battery_health,
    ROUND(avg_charge_used_per_km, 2) AS charge_used_percent_per_km
FROM v_by_robot
ORDER BY avg_charge_used_per_km DESC
LIMIT 5
"""
question_3_result = database_connection.execute(question_3_sql).df()
print("Q3: Five robots with the highest battery use per km")
print(question_3_result.to_string(index=False))
tired_robot_list = ", ".join(question_3_result["robot_id"])
print(f"Answer: the five hungriest batteries belong to {tired_robot_list}.\n")


# ---------------------------------------------------------------------
# QUESTION 4: What share of missions have zero interventions, by zone?
# ---------------------------------------------------------------------
# GROUP BY zone makes one row per zone.
# We divide the SUM of zero-intervention missions by the SUM of missions
# (ratio of sums). The 1.0 forces decimal division instead of whole numbers.
question_4_sql = """
SELECT
    zone,
    SUM(zero_intervention_missions)                      AS zero_intervention_missions,
    SUM(missions)                                        AS missions,
    ROUND(100.0 * SUM(zero_intervention_missions)
          / NULLIF(SUM(missions), 0), 1)                 AS autonomy_rate_percent
FROM v_daily_slice
GROUP BY zone
ORDER BY autonomy_rate_percent
"""
question_4_result = database_connection.execute(question_4_sql).df()
print("Q4: Autonomy rate by zone (lowest first)")
print(question_4_result.to_string(index=False))
print(
    f"Answer: {question_4_result['zone'][0]} has the lowest autonomy rate "
    f"({question_4_result['autonomy_rate_percent'][0]:.1f}% of missions need no help).\n"
)


# ---------------------------------------------------------------------
# QUESTION 5: Which intervention reasons are most common in rain
#             compared with clear weather?
# ---------------------------------------------------------------------
# We total the interventions for each reason and weather. Then a window
# function (SUM(...) OVER (PARTITION BY weather)) gives the total for that
# weather, so we can turn each count into a share of its own weather.
question_5_sql = """
SELECT
    weather,
    reason,
    SUM(interventions) AS interventions,
    ROUND(100.0 * SUM(interventions)
          / SUM(SUM(interventions)) OVER (PARTITION BY weather), 1) AS share_percent
FROM v_intervention_reasons
GROUP BY weather, reason
ORDER BY weather, share_percent DESC
"""
question_5_result = database_connection.execute(question_5_sql).df()
print("Q5: Intervention reasons by weather (share of that weather's interventions)")
print(question_5_result.to_string(index=False))
rain_top_reason = question_5_result[question_5_result["weather"] == "rain"].iloc[0]
clear_top_reason = question_5_result[question_5_result["weather"] == "clear"].iloc[0]
print(
    f"Answer: in rain the top reason is {rain_top_reason['reason']} "
    f"({rain_top_reason['share_percent']:.1f}%); in clear weather it is "
    f"{clear_top_reason['reason']} ({clear_top_reason['share_percent']:.1f}%).\n"
)


# ---------------------------------------------------------------------
# QUESTION 6: What is the on-time % for each hour of the day?
# ---------------------------------------------------------------------
# v_by_hour has one row per hour AND zone, so we add the zones together
# first (SUM), then divide on-time deliveries by completed deliveries.
question_6_sql = """
SELECT
    start_hour,
    SUM(completed_deliveries) AS completed_deliveries,
    ROUND(100.0 * SUM(on_time_deliveries)
          / NULLIF(SUM(completed_deliveries), 0), 1) AS on_time_percent
FROM v_by_hour
GROUP BY start_hour
ORDER BY start_hour
"""
question_6_result = database_connection.execute(question_6_sql).df()
print("Q6: On-time % by start hour")
print(question_6_result.to_string(index=False))
worst_hour_row = question_6_result.sort_values("on_time_percent").iloc[0]
print(
    f"Answer: the worst hour is {int(worst_hour_row['start_hour'])}:00, with only "
    f"{worst_hour_row['on_time_percent']:.1f}% of deliveries on time.\n"
)


# ---------------------------------------------------------------------
# QUESTION 7: What is the rolling 7-day interventions per 1,000 km?
# ---------------------------------------------------------------------
# The rolling value was already built in v_kpi_daily (C5), using window
# functions that add up 7 days of interventions and 7 days of km before
# dividing. We show the most recent 10 days, then sort oldest to newest.
question_7_sql = """
SELECT
    start_date,
    ROUND(interventions_per_1000km, 1)     AS daily_value,
    ROUND(interventions_per_1000km_7d, 1)  AS rolling_7_day_value
FROM v_kpi_daily
ORDER BY start_date DESC
LIMIT 10
"""
question_7_result = (
    database_connection.execute(question_7_sql).df().sort_values("start_date")
)
print("Q7: Last 10 days, daily versus rolling 7-day interventions per 1,000 km")
print(question_7_result.to_string(index=False))
latest_row = question_7_result.iloc[-1]
print(
    f"Answer: on {latest_row['start_date']} the rolling 7-day rate was "
    f"{latest_row['rolling_7_day_value']:.1f} per 1,000 km.\n"
)


# ---------------------------------------------------------------------
# QUESTION 8: The A/B test. From 2026-08-30 (day 61) onward, compare
#             v2.2 and v2.3 interventions per 1,000 km.
# ---------------------------------------------------------------------
# From day 61, odd-numbered robots run v2.2 and even-numbered robots run
# v2.3 on the SAME days, so this is a fair comparison.
# WHERE keeps only those days and those two versions.
question_8_sql = """
SELECT
    software_version,
    SUM(interventions)                                        AS interventions,
    ROUND(SUM(km), 1)                                         AS km,
    ROUND(1000.0 * SUM(interventions) / NULLIF(SUM(km), 0), 1) AS interventions_per_1000km
FROM v_daily_slice
WHERE start_date >= DATE '2026-08-30'
  AND software_version IN ('v2.2', 'v2.3')
GROUP BY software_version
ORDER BY software_version
"""
question_8_result = database_connection.execute(question_8_sql).df()
print("Q8: A/B test, last 30 days")
print(question_8_result.to_string(index=False))
version_22_rate = question_8_result.loc[
    question_8_result["software_version"] == "v2.2", "interventions_per_1000km"
].iloc[0]
version_23_rate = question_8_result.loc[
    question_8_result["software_version"] == "v2.3", "interventions_per_1000km"
].iloc[0]
print(
    f"Answer: v2.3 needs help {version_23_rate / version_22_rate:.2f} times as often "
    f"as v2.2 on the same days.\n"
)


# ---------------------------------------------------------------------
# QUESTION 9: Cost per delivery on weekdays versus weekends?
# ---------------------------------------------------------------------
# v_by_daytype splits the data into weekday and weekend. Depreciation is
# charged per day, so busy weekends spread it over more deliveries.
question_9_sql = """
SELECT
    day_type,
    completed_deliveries,
    ROUND(cost_per_delivery, 2) AS cost_per_delivery_dollars
FROM v_by_daytype
ORDER BY day_type
"""
question_9_result = database_connection.execute(question_9_sql).df()
print("Q9: Cost per delivery, weekday versus weekend")
print(question_9_result.to_string(index=False))
cheapest_day_type = question_9_result.sort_values("cost_per_delivery_dollars").iloc[0]
print(
    f"Answer: {cheapest_day_type['day_type']}s are cheaper, at "
    f"${cheapest_day_type['cost_per_delivery_dollars']:.2f} per delivery.\n"
)


# ---------------------------------------------------------------------
# QUESTION 10: Which zone and weather combination is worst?
# ---------------------------------------------------------------------
# GROUP BY two columns makes one row per zone-and-weather pair.
# We rank by interventions per 1,000 km, highest first, and show the top 5.
question_10_sql = """
SELECT
    zone,
    weather,
    SUM(interventions)                                         AS interventions,
    ROUND(SUM(km), 1)                                          AS km,
    ROUND(1000.0 * SUM(interventions) / NULLIF(SUM(km), 0), 1) AS interventions_per_1000km
FROM v_daily_slice
GROUP BY zone, weather
ORDER BY interventions_per_1000km DESC
LIMIT 5
"""
question_10_result = database_connection.execute(question_10_sql).df()
print("Q10: Worst five zone and weather combinations")
print(question_10_result.to_string(index=False))
worst_combination = question_10_result.iloc[0]
print(
    f"Answer: the worst combination is {worst_combination['zone']} in "
    f"{worst_combination['weather']}, at "
    f"{worst_combination['interventions_per_1000km']:.1f} interventions per 1,000 km."
)

Q1: Totals for the whole period
 total_missions  total_km  total_completed_deliveries
      44,038.00 74,880.80                   43,253.00
Answer: the fleet ran 44,038 missions over 74,881 km and completed 43,253 deliveries.

Q2: Interventions per 1,000 km by software version
software_version  interventions        km  interventions_per_1000km
            v2.1         935.00 24,943.70                     37.50
            v2.2       1,311.00 37,500.90                     35.00
            v2.3         765.00 12,436.20                     61.50
Answer: v2.3 is the worst version, with 61.5 interventions per 1,000 km.

Q3: Five robots with the highest battery use per km
robot_id  battery_health  charge_used_percent_per_km
    R007            0.65                        7.69
    R019            0.65                        7.69
    R023            0.65                        7.68
    R044            0.65                        7.68
    R031            0.65                        7.66
Answer

In [20]:
# =====================================================================
# CELL D1: Data quality checks 1 to 5
# Each check runs a SQL query, compares the result with what it should be,
# and records PASS or FAIL. All results go into the list check_results,
# which cell D2 will continue to fill and then summarise.
# =====================================================================

# An empty list that will hold one dictionary per check
check_results = []


def record_check(check_number, check_name, passed, detail_text):
    """Save one check result in check_results and print PASS or FAIL."""
    # Store the result as a dictionary so we can turn it into a table later
    check_results.append(
        {
            "check_number": check_number,
            "check_name": check_name,
            "result": "PASS" if passed else "FAIL",
            "detail": detail_text,
        }
    )
    # Print the result straight away so we can read it as the cell runs
    status_text = "PASS" if passed else "FAIL"
    print(f"[{status_text}] Check {check_number}: {check_name}")
    print(f"        {detail_text}\n")


# ---------------------------------------------------------------------
# CHECK 1: Planted errors are caught
# The cleaning view should remove exactly the mistakes we planted:
# 20 duplicates + 30 negative durations + 25 zero distances = 75 rows.
# ---------------------------------------------------------------------

# Number of rows in the raw missions table (dirty data included)
raw_mission_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions"
).fetchone()[0]

# Number of rows left after the cleaning view has done its work
clean_mission_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions_clean"
).fetchone()[0]

# Duplicates = total rows minus the number of different mission_ids
duplicate_row_count = database_connection.execute(
    "SELECT COUNT(*) - COUNT(DISTINCT mission_id) FROM missions"
).fetchone()[0]

# Rows where the robot supposedly arrived before it started
negative_duration_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions WHERE actual_arrival < start_time"
).fetchone()[0]

# Rows with a distance of zero (or below)
zero_distance_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions WHERE distance_km <= 0"
).fetchone()[0]

# The number of rows we expect the cleaning to remove, from the A1 settings
expected_removed_rows = (
    NUMBER_OF_DUPLICATE_MISSIONS
    + NUMBER_OF_NEGATIVE_DURATIONS
    + NUMBER_OF_ZERO_DISTANCES
)
removed_row_count = raw_mission_count - clean_mission_count

record_check(
    1,
    "Planted errors are caught by missions_clean",
    removed_row_count == expected_removed_rows,
    f"raw={raw_mission_count}, clean={clean_mission_count}, "
    f"removed={removed_row_count} (expected {expected_removed_rows}). "
    f"Found in raw data: duplicates={duplicate_row_count}, "
    f"negative durations={negative_duration_count}, "
    f"zero distances={zero_distance_count}.",
)


# ---------------------------------------------------------------------
# CHECK 2: IDs are unique
# After cleaning, every mission_id must appear exactly once.
# ---------------------------------------------------------------------

# COUNT(*) counts all rows; COUNT(DISTINCT ...) counts different IDs only
total_rows_in_clean_view, distinct_ids_in_clean_view = database_connection.execute(
    "SELECT COUNT(*), COUNT(DISTINCT mission_id) FROM missions_clean"
).fetchone()

record_check(
    2,
    "mission_id is unique in missions_clean",
    total_rows_in_clean_view == distinct_ids_in_clean_view,
    f"rows={total_rows_in_clean_view}, distinct mission_ids={distinct_ids_in_clean_view}.",
)


# ---------------------------------------------------------------------
# CHECK 3: No orphan interventions
# An orphan is an intervention whose mission does not exist at all.
# We join interventions to the RAW missions table and look for no match.
# ---------------------------------------------------------------------

# LEFT JOIN keeps every intervention; if no mission matches, m.mission_id is NULL
orphan_intervention_count = database_connection.execute(
    """
    SELECT COUNT(*)
    FROM interventions AS i
    LEFT JOIN missions AS m ON i.mission_id = m.mission_id
    WHERE m.mission_id IS NULL
    """
).fetchone()[0]

# For information only: interventions that belong to missions the cleaning
# view removed. These are excluded from every view, so they are not an error.
interventions_of_removed_missions = database_connection.execute(
    """
    SELECT COUNT(*)
    FROM interventions AS i
    LEFT JOIN missions_clean AS c ON i.mission_id = c.mission_id
    WHERE c.mission_id IS NULL
    """
).fetchone()[0]

record_check(
    3,
    "No orphan interventions",
    orphan_intervention_count == 0,
    f"orphans (no mission at all)={orphan_intervention_count}. "
    f"For information: {interventions_of_removed_missions} interventions belong "
    f"to missions removed by cleaning and are excluded from all views.",
)


# ---------------------------------------------------------------------
# CHECK 4: Time makes sense
# Arrival must not be before the start, and the promised time must be
# after the start. We count the rows that break each rule.
# ---------------------------------------------------------------------

# Rule A: arrival before start (failed missions have NULL arrival, so they skip)
arrival_before_start_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions_clean WHERE actual_arrival < start_time"
).fetchone()[0]

# Rule B: the promised ETA is not later than the start time
promise_not_after_start_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions_clean WHERE promised_eta <= start_time"
).fetchone()[0]

record_check(
    4,
    "Times are in a sensible order",
    arrival_before_start_count == 0 and promise_not_after_start_count == 0,
    f"arrival before start={arrival_before_start_count}, "
    f"promised ETA not after start={promise_not_after_start_count}.",
)


# ---------------------------------------------------------------------
# CHECK 5: Values are in range
# One count per rule; every count must be zero.
# ---------------------------------------------------------------------

# Distance should sit between the minimum and maximum we simulated
distance_out_of_range_count = database_connection.execute(
    """
    SELECT COUNT(*) FROM v_mission_facts
    WHERE distance_km < 0.3 OR distance_km > 6
    """
).fetchone()[0]

# Battery charge is a percentage, so it must stay between 0 and 100
charge_out_of_range_count = database_connection.execute(
    """
    SELECT COUNT(*) FROM v_mission_facts
    WHERE charge_start_pct < 0 OR charge_start_pct > 100
       OR charge_end_pct   < 0 OR charge_end_pct   > 100
    """
).fetchone()[0]

# A battery cannot end a trip with MORE charge than it started with
charge_grew_count = database_connection.execute(
    "SELECT COUNT(*) FROM v_mission_facts WHERE charge_end_pct > charge_start_pct"
).fetchone()[0]

# Intervention lengths should stay between 10 and 600 seconds
duration_out_of_range_count = database_connection.execute(
    """
    SELECT COUNT(*) FROM interventions
    WHERE duration_sec < 10 OR duration_sec > 600
    """
).fetchone()[0]

all_range_counts = [
    distance_out_of_range_count,
    charge_out_of_range_count,
    charge_grew_count,
    duration_out_of_range_count,
]

record_check(
    5,
    "Values are inside their allowed ranges",
    sum(all_range_counts) == 0,
    f"distance out of range={distance_out_of_range_count}, "
    f"charge outside 0-100={charge_out_of_range_count}, "
    f"charge grew during trip={charge_grew_count}, "
    f"intervention duration out of range={duration_out_of_range_count}.",
)

# A short summary of where we are so far
passed_so_far = sum(1 for check in check_results if check["result"] == "PASS")
print(f"Checks 1 to 5 done: {passed_so_far} of {len(check_results)} passed.")

[PASS] Check 1: Planted errors are caught by missions_clean
        raw=44113, clean=44038, removed=75 (expected 75). Found in raw data: duplicates=20, negative durations=30, zero distances=25.

[PASS] Check 2: mission_id is unique in missions_clean
        rows=44038, distinct mission_ids=44038.

[PASS] Check 3: No orphan interventions
        orphans (no mission at all)=0. For information: 5 interventions belong to missions removed by cleaning and are excluded from all views.

[PASS] Check 4: Times are in a sensible order
        arrival before start=0, promised ETA not after start=0.

[PASS] Check 5: Values are inside their allowed ranges
        distance out of range=0, charge outside 0-100=0, charge grew during trip=0, intervention duration out of range=0.

Checks 1 to 5 done: 5 of 5 passed.


In [21]:
# =====================================================================
# CELL D2: Data quality checks 6 to 10, and the final summary
# Continues from cell D1: it uses check_results and record_check.
# At the end it prints a table of all 10 checks and saves it to
# data_quality_checks.csv.
# =====================================================================

# If this cell is run twice, remove old results for checks 6 to 10 first,
# so the summary never shows a check twice
check_results[:] = [check for check in check_results if check["check_number"] <= 5]


# ---------------------------------------------------------------------
# CHECK 6: Totals reconcile
# Total missions and total km must be identical in every view, and equal
# to the totals in missions_clean. If not, a join or a filter lost rows.
# ---------------------------------------------------------------------

# The reference totals come straight from the cleaned missions
reference_mission_count, reference_km = database_connection.execute(
    "SELECT COUNT(*), SUM(distance_km) FROM missions_clean"
).fetchone()

# The views whose totals must match the reference
views_to_reconcile = [
    "v_by_version",
    "v_by_zone",
    "v_by_weather",
    "v_by_daytype",
    "v_kpi_daily",
    "v_by_hour",
]

# We collect one row per view so we can print a small table
reconcile_rows = []
all_totals_match = True

for view_name in views_to_reconcile:
    # Add up the missions and km columns of this view
    view_missions, view_km = database_connection.execute(
        f"SELECT SUM(missions), SUM(km) FROM {view_name}"
    ).fetchone()

    # Missions must match exactly; km may differ only by tiny rounding (0.001)
    missions_match = int(view_missions) == int(reference_mission_count)
    km_match = abs(float(view_km) - float(reference_km)) < 0.001

    if not (missions_match and km_match):
        all_totals_match = False

    reconcile_rows.append(
        {
            "view": view_name,
            "missions": int(view_missions),
            "km": round(float(view_km), 2),
            "matches_reference": missions_match and km_match,
        }
    )

print(f"Reference totals from missions_clean: {reference_mission_count} "
      f"missions, {float(reference_km):.2f} km")
print(pd.DataFrame(reconcile_rows).to_string(index=False))
print()

record_check(
    6,
    "Totals reconcile across all views",
    all_totals_match,
    f"{sum(row['matches_reference'] for row in reconcile_rows)} of "
    f"{len(reconcile_rows)} views match missions_clean.",
)


# ---------------------------------------------------------------------
# CHECK 7: Ratios are sane
# Autonomy rate and on-time % are shares, so they must be between 0 and 1.
# We count rows outside that range; the answer must be zero.
# ---------------------------------------------------------------------

views_with_ratios = [
    "v_kpi_daily",
    "v_by_version",
    "v_by_zone",
    "v_by_weather",
    "v_by_hour",
]

bad_ratio_count = 0
for view_name in views_with_ratios:
    # NULL values are ignored by these comparisons, which is what we want
    rows_outside_range = database_connection.execute(
        f"""
        SELECT COUNT(*) FROM {view_name}
        WHERE autonomy_rate < 0 OR autonomy_rate > 1
           OR on_time_pct   < 0 OR on_time_pct   > 1
        """
    ).fetchone()[0]
    bad_ratio_count += rows_outside_range

record_check(
    7,
    "Autonomy rate and on-time % are between 0 and 1",
    bad_ratio_count == 0,
    f"rows outside 0-1 across {len(views_with_ratios)} views = {bad_ratio_count}.",
)


# ---------------------------------------------------------------------
# CHECK 8: Hand check
# Pick one day and one zone, rebuild the numbers in plain pandas (using the
# same cleaning rules as missions_clean), and compare with the SQL view.
# ---------------------------------------------------------------------

# Change these two values if you want to check a different slice
CHECK_DAY = "2026-08-15"
CHECK_ZONE = "Downtown"

# Step 1: apply the same three cleaning rules in pandas
hand_check_missions = missions_df.drop_duplicates()                       # rule 1
hand_check_missions = hand_check_missions[hand_check_missions["distance_km"] > 0]  # rule 2
arrival_is_valid = (
    hand_check_missions["actual_arrival"].isna()
    | (hand_check_missions["actual_arrival"] >= hand_check_missions["start_time"])
)
hand_check_missions = hand_check_missions[arrival_is_valid]               # rule 3

# Step 2: keep only the chosen day and zone
day_text = hand_check_missions["start_time"].dt.strftime("%Y-%m-%d")
slice_missions = hand_check_missions[
    (day_text == CHECK_DAY) & (hand_check_missions["zone"] == CHECK_ZONE)
]

# Step 3: count the three numbers in pandas
pandas_mission_count = len(slice_missions)
pandas_km = float(slice_missions["distance_km"].sum())
pandas_intervention_count = int(
    interventions_df["mission_id"].isin(slice_missions["mission_id"]).sum()
)

# Step 4: ask the SQL view for the same slice (? placeholders are safer)
sql_missions, sql_interventions, sql_km = database_connection.execute(
    """
    SELECT
        COALESCE(SUM(missions), 0),
        COALESCE(SUM(interventions), 0),
        COALESCE(SUM(km), 0)
    FROM v_daily_slice
    WHERE start_date = CAST(? AS DATE) AND zone = ?
    """,
    [CHECK_DAY, CHECK_ZONE],
).fetchone()

# Step 5: compare (counts exactly, km within 0.001)
hand_check_passed = (
    int(sql_missions) == pandas_mission_count
    and int(sql_interventions) == pandas_intervention_count
    and abs(float(sql_km) - pandas_km) < 0.001
)

record_check(
    8,
    f"Hand check: {CHECK_DAY}, {CHECK_ZONE}",
    hand_check_passed,
    f"pandas: {pandas_mission_count} missions, {pandas_intervention_count} "
    f"interventions, {pandas_km:.2f} km | SQL: {int(sql_missions)} missions, "
    f"{int(sql_interventions)} interventions, {float(sql_km):.2f} km.",
)


# ---------------------------------------------------------------------
# CHECK 9: Cost check
# Pick one completed mission with at least one intervention, work out its
# variable cost by hand from the price table, and compare with the view.
# ---------------------------------------------------------------------

# Step 1: pick the mission
picked_mission = database_connection.execute(
    """
    SELECT mission_id, distance_km, intervention_count,
           intervention_seconds, variable_cost
    FROM v_mission_facts
    WHERE is_completed = 1 AND intervention_count >= 1
    ORDER BY mission_id
    LIMIT 1
    """
).df().iloc[0]

# Step 2: read the prices into a dictionary (item name -> price)
price_table = database_connection.execute(
    "SELECT item, value FROM cost_assumptions"
).df()
price_lookup = dict(zip(price_table["item"], price_table["value"]))

# Step 3: work out each cost term by hand
hand_km_cost = picked_mission["distance_km"] * (
    price_lookup["maintenance_per_km"] + price_lookup["energy_per_km"]
)
hand_intervention_cost = (
    picked_mission["intervention_seconds"] / 60 * price_lookup["operator_per_minute"]
    + picked_mission["intervention_count"] * price_lookup["handoff_per_intervention"]
)
hand_overhead_cost = price_lookup["overhead_per_mission"]
hand_variable_cost = hand_km_cost + hand_intervention_cost + hand_overhead_cost

# Step 4: print every term so you can follow the maths with a calculator
print(f"Cost check for mission {picked_mission['mission_id']}:")
print(f"  km cost            = {hand_km_cost:.4f}")
print(f"  intervention cost  = {hand_intervention_cost:.4f}")
print(f"  overhead cost      = {hand_overhead_cost:.4f}")
print(f"  hand total         = {hand_variable_cost:.4f}")
print(f"  value in the view  = {picked_mission['variable_cost']:.4f}\n")

# Step 5: the difference must be below half a cent
cost_difference = abs(hand_variable_cost - picked_mission["variable_cost"])
record_check(
    9,
    "Variable cost matches a hand calculation",
    cost_difference < 0.005,
    f"difference = ${cost_difference:.6f} (limit $0.005).",
)


# ---------------------------------------------------------------------
# CHECK 10: Planted patterns are visible
# Read the breakdown views and confirm every planted pattern shows up.
# ---------------------------------------------------------------------

# Load each view into a small pandas table
version_table = database_connection.execute("SELECT * FROM v_by_version").df()
weather_table = database_connection.execute("SELECT * FROM v_by_weather").df()
zone_table = database_connection.execute("SELECT * FROM v_by_zone").df()
daytype_table = database_connection.execute("SELECT * FROM v_by_daytype").df()
robot_table = database_connection.execute("SELECT * FROM v_by_robot").df()

# Turn small tables into dictionaries, e.g. {"v2.1": 41.2, "v2.2": 35.0, ...}
rate_by_version = dict(
    zip(version_table["software_version"], version_table["interventions_per_1000km"])
)
rate_by_weather = dict(
    zip(weather_table["weather"], weather_table["interventions_per_1000km"])
)
on_time_by_weather = dict(
    zip(weather_table["weather"], weather_table["on_time_pct"])
)
cost_by_daytype = dict(
    zip(daytype_table["day_type"], daytype_table["cost_per_delivery"])
)

# Pattern 1: v2.2 is better than v2.1, and v2.3 is worse than v2.1
version_pattern_ok = (
    rate_by_version["v2.2"] < rate_by_version["v2.1"]
    and rate_by_version["v2.3"] > rate_by_version["v2.1"]
)

# Pattern 2: rain needs 1.3 to 1.9 times the interventions of clear weather
rain_ratio = rate_by_weather["rain"] / rate_by_weather["clear"]
rain_pattern_ok = 1.3 <= rain_ratio <= 1.9

# Pattern 3: Old Town is the worst zone, 1.6 to 2.5 times the other zones
# (the other zones are combined as a ratio of sums)
worst_zone = zone_table.sort_values(
    "interventions_per_1000km", ascending=False
).iloc[0]["zone"]
other_zones = zone_table[zone_table["zone"] != "Old Town"]
other_zones_rate = (
    1000 * other_zones["interventions"].sum() / other_zones["km"].sum()
)
old_town_rate = float(
    zone_table.loc[zone_table["zone"] == "Old Town", "interventions_per_1000km"].iloc[0]
)
old_town_ratio = old_town_rate / other_zones_rate
zone_pattern_ok = worst_zone == "Old Town" and 1.6 <= old_town_ratio <= 2.5

# Pattern 4: on-time % in rain is at least 5 points (0.05) below clear
on_time_gap = on_time_by_weather["clear"] - on_time_by_weather["rain"]
on_time_pattern_ok = on_time_gap >= 0.05

# Pattern 5: weekends cost less per delivery than weekdays
weekend_pattern_ok = cost_by_daytype["weekend"] < cost_by_daytype["weekday"]

# Pattern 6: the five tired robots hold drain ranks 1 to 5
top_drain_robots = set(robot_table.loc[robot_table["drain_rank"] <= 5, "robot_id"])
tired_robot_pattern_ok = top_drain_robots == set(TIRED_BATTERY_ROBOT_IDS)

pattern_results = {
    "versions (v2.2 < v2.1 < v2.3)": version_pattern_ok,
    f"rain ratio {rain_ratio:.2f} in 1.3-1.9": rain_pattern_ok,
    f"Old Town worst, ratio {old_town_ratio:.2f} in 1.6-2.5": zone_pattern_ok,
    f"on-time gap in rain {on_time_gap * 100:.1f} points >= 5": on_time_pattern_ok,
    "weekend cost per delivery lower": weekend_pattern_ok,
    "tired robots hold drain ranks 1-5": tired_robot_pattern_ok,
}

# Print each pattern on its own line so a failure is easy to spot
for pattern_name, pattern_ok in pattern_results.items():
    print(f"  {'OK  ' if pattern_ok else 'FAIL'} {pattern_name}")
print()

record_check(
    10,
    "Planted patterns are visible",
    all(pattern_results.values()),
    f"{sum(pattern_results.values())} of {len(pattern_results)} patterns found.",
)


# =====================================================================
# FINAL SUMMARY: one table with all 10 checks, saved to a CSV file
# =====================================================================

# Turn the list of dictionaries into a table, ordered by check number
results_table = pd.DataFrame(check_results).sort_values("check_number")

# Show the short version on screen (the CSV keeps the full detail column)
print("=== DATA QUALITY SUMMARY ===")
print(results_table[["check_number", "check_name", "result"]].to_string(index=False))

# Count how many checks failed and print one clear closing line
failed_check_count = int((results_table["result"] == "FAIL").sum())
if failed_check_count == 0:
    print("\nALL CHECKS PASSED")
else:
    print(f"\n{failed_check_count} CHECK(S) FAILED. Read the details above.")

# Save the full table so it can go into your GitHub repo and README
results_table.to_csv("data_quality_checks.csv", index=False)
print("Saved: data_quality_checks.csv")

Reference totals from missions_clean: 44038 missions, 74880.76 km
        view  missions        km  matches_reference
v_by_version     44038 74,880.76               True
   v_by_zone     44038 74,880.76               True
v_by_weather     44038 74,880.76               True
v_by_daytype     44038 74,880.76               True
 v_kpi_daily     44038 74,880.76               True
   v_by_hour     44038 74,880.76               True

[PASS] Check 6: Totals reconcile across all views
        6 of 6 views match missions_clean.



BinderException: Binder Error: Referenced column "autonomy_rate" not found in FROM clause!
Candidate bindings: "zone", "zero_intervention_missions", "km"

LINE 3:         WHERE autonomy_rate < 0 OR autonomy_rate > 1
                      ^

In [22]:
# Diagnostic: why does v_by_hour show fewer missions than missions_clean?
from pathlib import Path

# How many failed missions exist in the cleaned data?
failed_mission_count = database_connection.execute(
    "SELECT COUNT(*) FROM missions_clean WHERE status = 'failed'"
).fetchone()[0]

# How many missions does v_by_hour lose compared with missions_clean?
view_mission_total = database_connection.execute(
    "SELECT SUM(missions) FROM v_by_hour"
).fetchone()[0]
missing_mission_count = reference_mission_count - view_mission_total

print("Failed missions in missions_clean:", failed_mission_count)
print("Missions missing from v_by_hour:  ", missing_mission_count)

# Print the SQL saved for the view, so we can see how it counts missions
print(Path(SQL_FOLDER, "views", "v_by_hour.sql").read_text())

Failed missions in missions_clean: 785
Missions missing from v_by_hour:   0
-- v_by_hour
-- v_by_hour
-- Segmented hourly/spatial operational profile grouped by start_hour and zone.
-- Cost columns are excluded as daily depreciation is not calculated hourly.

SELECT
    start_hour,
    zone,
    COUNT(*)                     AS missions,                    -- ALL missions, failed ones too
    SUM(is_completed)            AS completed_deliveries,
    SUM(is_on_time)              AS on_time_deliveries,
    SUM(intervention_count)      AS interventions,
    SUM(distance_km)             AS km,
    SUM(has_zero_interventions)  AS zero_intervention_missions
    -- keep your three ratio columns below, unchanged:
    -- on_time_pct divides by completed_deliveries; autonomy_rate divides by missions
FROM v_mission_facts
GROUP BY start_hour, zone;



In [23]:
# =====================================================================
# CELL E1: Chart settings and the filtered data loader
# This cell creates:
#   1. CHART SETTINGS: colours and sizes shared by every chart
#   2. load_slice(): fetches rows of v_daily_slice for chosen filters
#   3. calculate_kpis(): turns those rows into the four KPIs
# Functions are allowed in this part. They are written so they can be
# copied straight into app.py later.
# =====================================================================

# ---------------------------------------------------------------------
# CHART SETTINGS (new CAPITAL-LETTER constants, used by every chart)
# ---------------------------------------------------------------------
CHART_TEMPLATE = "plotly_white"       # clean white background style
CHART_MAIN_COLOR = "#2F6DB5"          # calm blue for normal bars and lines
CHART_HIGHLIGHT_COLOR = "#E8832A"     # orange to pick out the "bad" item
CHART_GREY_COLOR = "#B8BEC6"          # grey for everything that is not the focus
CHART_HEIGHT = 420                    # chart height in pixels


# ---------------------------------------------------------------------
# FUNCTION 1: load_slice
# Fetches the rows of v_daily_slice that match the chosen filters.
# ---------------------------------------------------------------------
def load_slice(database_connection, first_date, last_date, zones, versions, weathers):
    """Return a pandas table of v_daily_slice rows matching the filters."""

    # The ? marks are placeholders (parameters). The values are passed
    # separately, never pasted into the SQL text. This stops a visitor from
    # sneaking SQL commands into a filter box, so it is much safer.
    slice_query = """
        SELECT *
        FROM v_daily_slice
        WHERE start_date BETWEEN CAST(? AS DATE) AND CAST(? AS DATE)
          AND list_contains(CAST(? AS VARCHAR[]), zone)
          AND list_contains(CAST(? AS VARCHAR[]), software_version)
          AND list_contains(CAST(? AS VARCHAR[]), weather)
    """

    # The values for the five ? marks, in the same order as in the query
    query_parameters = [
        str(first_date),
        str(last_date),
        list(zones),
        list(versions),
        list(weathers),
    ]

    # Run the query and return the result as a pandas table
    return database_connection.execute(slice_query, query_parameters).df()


# ---------------------------------------------------------------------
# FUNCTION 2: calculate_kpis
# Each KPI is ONE SUM divided by ANOTHER SUM (a ratio of sums).
# We never average daily percentages, because that gives wrong answers.
# ---------------------------------------------------------------------
def calculate_kpis(slice_dataframe):
    """Return a dictionary with the four KPIs for the rows given."""

    # Add up each column once, across all the rows we were given
    total_missions = float(slice_dataframe["missions"].sum())
    total_zero_intervention_missions = float(
        slice_dataframe["zero_intervention_missions"].sum()
    )
    total_interventions = float(slice_dataframe["interventions"].sum())
    total_km = float(slice_dataframe["km"].sum())
    total_completed_deliveries = float(slice_dataframe["completed_deliveries"].sum())
    total_on_time_deliveries = float(slice_dataframe["on_time_deliveries"].sum())
    total_cost = float(slice_dataframe["total_cost"].sum())

    # Autonomy rate: missions with no help needed / all missions
    if total_missions > 0:
        autonomy_rate = total_zero_intervention_missions / total_missions
    else:
        autonomy_rate = None  # no missions, so the KPI does not exist

    # Interventions per 1,000 km: interventions / km * 1000
    if total_km > 0:
        interventions_per_1000km = total_interventions / total_km * 1000
    else:
        interventions_per_1000km = None

    # On-time %: on-time deliveries / completed deliveries
    if total_completed_deliveries > 0:
        on_time_pct = total_on_time_deliveries / total_completed_deliveries
    else:
        on_time_pct = None

    # Cost per delivery: ALL cost (failed missions too) / completed deliveries
    if total_completed_deliveries > 0:
        cost_per_delivery = total_cost / total_completed_deliveries
    else:
        cost_per_delivery = None

    return {
        "autonomy_rate": autonomy_rate,
        "interventions_per_1000km": interventions_per_1000km,
        "on_time_pct": on_time_pct,
        "cost_per_delivery": cost_per_delivery,
    }


# ---------------------------------------------------------------------
# TEST (a): load EVERYTHING and compare with the whole-period KPIs
# ---------------------------------------------------------------------

# Find the full date range and every value of each filter in the data
date_range = database_connection.execute(
    "SELECT MIN(start_date), MAX(start_date) FROM v_daily_slice"
).fetchone()
all_first_date, all_last_date = date_range

all_zones = database_connection.execute(
    "SELECT DISTINCT zone FROM v_daily_slice"
).df()["zone"].tolist()
all_versions = database_connection.execute(
    "SELECT DISTINCT software_version FROM v_daily_slice"
).df()["software_version"].tolist()
all_weathers = database_connection.execute(
    "SELECT DISTINCT weather FROM v_daily_slice"
).df()["weather"].tolist()

# Load all rows (no real filtering, because every value is included)
everything_slice = load_slice(
    database_connection,
    all_first_date,
    all_last_date,
    all_zones,
    all_versions,
    all_weathers,
)
everything_kpis = calculate_kpis(everything_slice)

print("TEST A: KPIs for the whole fleet and whole period")
print(f"  rows loaded               : {len(everything_slice)}")
print(f"  autonomy rate             : {everything_kpis['autonomy_rate'] * 100:.1f}%")
print(f"  interventions per 1,000 km: {everything_kpis['interventions_per_1000km']:.1f}")
print(f"  on-time %                 : {everything_kpis['on_time_pct'] * 100:.1f}%")
print(f"  cost per delivery         : ${everything_kpis['cost_per_delivery']:.2f}")

# The same four KPIs worked out directly from v_kpi_daily (cell C5's view)
c5_kpis = database_connection.execute(
    """
    SELECT
        SUM(zero_intervention_missions) * 1.0 / SUM(missions),
        SUM(interventions) * 1000.0 / SUM(km),
        SUM(on_time_deliveries) * 1.0 / SUM(completed_deliveries),
        SUM(total_cost) * 1.0 / SUM(completed_deliveries)
    FROM v_kpi_daily
    """
).fetchone()

# Compare each KPI from load_slice/calculate_kpis with the C5 version
our_kpi_values = [
    everything_kpis["autonomy_rate"],
    everything_kpis["interventions_per_1000km"],
    everything_kpis["on_time_pct"],
    everything_kpis["cost_per_delivery"],
]
kpis_match = all(
    abs(our_value - c5_value) < 1e-9
    for our_value, c5_value in zip(our_kpi_values, c5_kpis)
)
print(f"  matches the C5 whole-period KPIs: {'MATCH' if kpis_match else 'MISMATCH'}\n")


# ---------------------------------------------------------------------
# TEST (b): only version v2.3 in rain
# ---------------------------------------------------------------------
v23_rain_slice = load_slice(
    database_connection,
    all_first_date,
    all_last_date,
    all_zones,
    ["v2.3"],      # only software version v2.3
    ["rain"],      # only rainy days
)
v23_rain_kpis = calculate_kpis(v23_rain_slice)

print("TEST B: only v2.3 in rain")
print(f"  rows loaded               : {len(v23_rain_slice)}")
if v23_rain_kpis["autonomy_rate"] is None:
    print("  no missions found for this filter.")
else:
    print(f"  autonomy rate             : {v23_rain_kpis['autonomy_rate'] * 100:.1f}%")
    print(f"  interventions per 1,000 km: {v23_rain_kpis['interventions_per_1000km']:.1f}")
    print(f"  on-time %                 : {v23_rain_kpis['on_time_pct'] * 100:.1f}%")
    print(f"  cost per delivery         : ${v23_rain_kpis['cost_per_delivery']:.2f}")

TEST A: KPIs for the whole fleet and whole period
  rows loaded               : 600
  autonomy rate             : 93.5%
  interventions per 1,000 km: 40.2
  on-time %                 : 53.8%
  cost per delivery         : $1.47
  matches the C5 whole-period KPIs: MATCH

TEST B: only v2.3 in rain
  rows loaded               : 40
  autonomy rate             : 86.0%
  interventions per 1,000 km: 96.9
  on-time %                 : 24.9%
  cost per delivery         : $1.61


In [24]:
# =====================================================================
# CELL E2: Overview trend chart
# Creates make_daily_trend_chart(), which draws the daily and 7-day
# interventions per 1,000 km, with markers for when each software
# version started. Then it loads the data, draws the chart and prints
# a one-line caption.
# =====================================================================

import plotly.graph_objects as go

# The two dates where something changed, used for the dashed lines.
# (Day 31 and day 61 of the simulation, from START_DATE 2026-07-01.)
V22_START_DATE = "2026-07-31"
V23_START_DATE = "2026-08-30"


def make_daily_trend_chart(daily_dataframe):
    """Return a line chart of interventions per 1,000 km over time.

    daily_dataframe needs these columns (from the view v_kpi_daily):
    start_date, interventions_per_1000km, interventions_per_1000km_7d
    """

    # Make sure the dates are real dates and the rows are in time order
    chart_data = daily_dataframe.copy()
    chart_data["start_date"] = pd.to_datetime(chart_data["start_date"])
    chart_data = chart_data.sort_values("start_date")

    # Start with an empty chart, then add things to it one at a time
    trend_figure = go.Figure()

    # Line 1: the daily value, thin and light because it jumps around a lot
    trend_figure.add_trace(
        go.Scatter(
            x=chart_data["start_date"],
            y=chart_data["interventions_per_1000km"],
            mode="lines",
            name="Daily",
            line=dict(color=CHART_GREY_COLOR, width=1.5),
        )
    )

    # Line 2: the 7-day rolling value, thick because it shows the real trend
    trend_figure.add_trace(
        go.Scatter(
            x=chart_data["start_date"],
            y=chart_data["interventions_per_1000km_7d"],
            mode="lines",
            name="7-day average",
            line=dict(color=CHART_MAIN_COLOR, width=3.5),
        )
    )

    # Dashed vertical lines. yref="paper" means 0 is the bottom of the chart
    # and 1 is the top, so the lines always span the full height.
    for marker_date, marker_label in [
        (V22_START_DATE, "v2.2 starts"),
        (V23_START_DATE, "v2.3 starts on half the fleet"),
    ]:
        # The dashed line itself
        trend_figure.add_shape(
            type="line",
            x0=marker_date,
            x1=marker_date,
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=CHART_HIGHLIGHT_COLOR, width=1.5, dash="dash"),
        )
        # The text label at the top of the line
        trend_figure.add_annotation(
            x=marker_date,
            y=1,
            yref="paper",
            text=marker_label,
            showarrow=False,
            xanchor="left",
            yanchor="bottom",
            font=dict(color=CHART_HIGHLIGHT_COLOR, size=12),
        )

    # Title, axis names with units, legend at the bottom, shared style
    trend_figure.update_layout(
        title="Interventions per 1,000 km: daily and 7-day average",
        xaxis_title="Date",
        yaxis_title="Interventions per 1,000 km",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
        legend=dict(orientation="h", yanchor="top", y=-0.2, x=0),
        margin=dict(t=80),   # extra room at the top for the line labels
    )

    # Start the y axis at zero so the size of changes is not exaggerated
    trend_figure.update_yaxes(rangemode="tozero")

    return trend_figure


# ---------------------------------------------------------------------
# Load the daily KPI view, draw the chart and show it
# ---------------------------------------------------------------------
daily_kpi_table = database_connection.execute(
    """
    SELECT start_date, interventions_per_1000km, interventions_per_1000km_7d
    FROM v_kpi_daily
    ORDER BY start_date
    """
).df()

daily_trend_figure = make_daily_trend_chart(daily_kpi_table)
daily_trend_figure.show()

# One caption sentence under the chart, with real numbers from the data
first_week_rate = daily_kpi_table["interventions_per_1000km_7d"].iloc[7]
last_week_rate = daily_kpi_table["interventions_per_1000km_7d"].iloc[-1]
print(
    f"Caption: the 7-day average moved from {first_week_rate:.1f} in the first "
    f"weeks to {last_week_rate:.1f} at the end, with a dip after v2.2 and a "
    f"rise after v2.3 reached half the fleet."
)
print("Rows plotted:", len(daily_kpi_table))

Caption: the 7-day average moved from 31.8 in the first weeks to 53.9 at the end, with a dip after v2.2 and a rise after v2.3 reached half the fleet.
Rows plotted: 90


In [25]:
# =====================================================================
# CELL E3: Reliability charts
# Creates three chart functions, each returning a Plotly figure:
#   1. make_version_bar_chart  : interventions per 1,000 km by software version
#   2. make_pareto_chart       : why robots need help (reasons, high to low)
#   3. make_zone_bar_chart     : interventions per 1,000 km by zone
# Then it loads each view, draws each chart and prints a caption.
# =====================================================================

from plotly.subplots import make_subplots


# ---------------------------------------------------------------------
# FUNCTION 1: bar chart by software version
# ---------------------------------------------------------------------
def make_version_bar_chart(version_dataframe):
    """One bar per software_version for interventions_per_1000km.

    version_dataframe needs these columns (from v_by_version):
    software_version, interventions_per_1000km
    """

    # Sort by version name so the bars always read v2.1, v2.2, v2.3
    chart_data = version_dataframe.sort_values("software_version")

    # Find the highest value, so we can colour that bar orange
    highest_value = chart_data["interventions_per_1000km"].max()

    # Build one colour per bar: orange for the worst, blue for the others
    bar_colors = []
    for bar_value in chart_data["interventions_per_1000km"]:
        if bar_value == highest_value:
            bar_colors.append(CHART_HIGHLIGHT_COLOR)
        else:
            bar_colors.append(CHART_MAIN_COLOR)

    # Draw the bars, with the value printed above each one (1 decimal)
    version_figure = go.Figure(
        go.Bar(
            x=chart_data["software_version"],
            y=chart_data["interventions_per_1000km"],
            marker_color=bar_colors,
            text=chart_data["interventions_per_1000km"],
            texttemplate="%{text:.1f}",
            textposition="outside",
        )
    )

    # Title, axis names with units, and the shared style settings
    version_figure.update_layout(
        title="Interventions per 1,000 km by software version",
        xaxis_title="Software version",
        yaxis_title="Interventions per 1,000 km",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
    )

    # Leave 15% extra room above the tallest bar for its label
    version_figure.update_yaxes(range=[0, highest_value * 1.15])

    return version_figure


# ---------------------------------------------------------------------
# FUNCTION 2: Pareto chart of intervention reasons
# A Pareto chart has bars sorted from biggest to smallest, plus a line
# showing the running (cumulative) share, so you can see which few
# reasons cause most of the problems.
# ---------------------------------------------------------------------
def make_pareto_chart(reasons_dataframe):
    """Bars of interventions per reason plus a cumulative % line.

    reasons_dataframe needs these columns (from v_intervention_reasons):
    reason, interventions
    """

    # The view has many rows per reason (one per version, zone and weather),
    # so first add them up to ONE row per reason, biggest first
    reason_totals = (
        reasons_dataframe.groupby("reason", as_index=False)["interventions"]
        .sum()
        .sort_values("interventions", ascending=False)
    )

    # Running total divided by the grand total gives the cumulative share
    grand_total = reason_totals["interventions"].sum()
    reason_totals["cumulative_percent"] = (
        reason_totals["interventions"].cumsum() / grand_total * 100
    )

    # A chart with two y axes: counts on the left, percent on the right
    pareto_figure = make_subplots(specs=[[{"secondary_y": True}]])

    # The bars (left axis)
    pareto_figure.add_trace(
        go.Bar(
            x=reason_totals["reason"],
            y=reason_totals["interventions"],
            name="Interventions",
            marker_color=CHART_MAIN_COLOR,
        ),
        secondary_y=False,
    )

    # The cumulative line (right axis)
    pareto_figure.add_trace(
        go.Scatter(
            x=reason_totals["reason"],
            y=reason_totals["cumulative_percent"],
            name="Cumulative share",
            mode="lines+markers",
            line=dict(color=CHART_HIGHLIGHT_COLOR, width=3),
        ),
        secondary_y=True,
    )

    # Title, legend at the bottom, and the shared style settings
    pareto_figure.update_layout(
        title="Why robots need help (Pareto)",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
        legend=dict(orientation="h", yanchor="top", y=-0.25, x=0),
    )
    pareto_figure.update_xaxes(title_text="Reason for intervention")
    pareto_figure.update_yaxes(title_text="Number of interventions", secondary_y=False)
    pareto_figure.update_yaxes(
        title_text="Cumulative share of interventions",
        range=[0, 100],
        ticksuffix="%",
        secondary_y=True,
    )

    return pareto_figure


# ---------------------------------------------------------------------
# FUNCTION 3: horizontal bar chart by zone
# ---------------------------------------------------------------------
def make_zone_bar_chart(zone_dataframe):
    """One horizontal bar per zone, worst zone at the top and in orange.

    zone_dataframe needs these columns (from v_by_zone):
    zone, interventions_per_1000km
    """

    # Sort from the highest rate to the lowest
    chart_data = zone_dataframe.sort_values(
        "interventions_per_1000km", ascending=False
    )

    # Colour only the worst zone (the first row after sorting) orange
    highest_value = chart_data["interventions_per_1000km"].max()
    bar_colors = []
    for bar_value in chart_data["interventions_per_1000km"]:
        if bar_value == highest_value:
            bar_colors.append(CHART_HIGHLIGHT_COLOR)
        else:
            bar_colors.append(CHART_MAIN_COLOR)

    # orientation="h" makes the bars horizontal (zone names on the y axis)
    zone_figure = go.Figure(
        go.Bar(
            x=chart_data["interventions_per_1000km"],
            y=chart_data["zone"],
            orientation="h",
            marker_color=bar_colors,
            text=chart_data["interventions_per_1000km"],
            texttemplate="%{text:.1f}",
            textposition="outside",
        )
    )

    zone_figure.update_layout(
        title="Interventions per 1,000 km by zone",
        xaxis_title="Interventions per 1,000 km",
        yaxis_title="Zone",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
    )

    # Plotly draws the first row at the BOTTOM of a horizontal chart, so we
    # reverse the y axis to put the worst zone at the top
    zone_figure.update_yaxes(autorange="reversed")
    zone_figure.update_xaxes(range=[0, highest_value * 1.15])

    return zone_figure


# ---------------------------------------------------------------------
# Load each view, draw each chart, print a caption
# ---------------------------------------------------------------------

# --- Chart 1: software versions ---
version_table = database_connection.execute("SELECT * FROM v_by_version").df()
make_version_bar_chart(version_table).show()

worst_version_row = version_table.sort_values(
    "interventions_per_1000km", ascending=False
).iloc[0]
print(
    f"Caption: {worst_version_row['software_version']} needs help most often, at "
    f"{worst_version_row['interventions_per_1000km']:.1f} interventions per 1,000 km."
)
print()

# --- Chart 2: Pareto of reasons ---
reasons_table = database_connection.execute(
    "SELECT * FROM v_intervention_reasons"
).df()
make_pareto_chart(reasons_table).show()

reason_summary = (
    reasons_table.groupby("reason")["interventions"].sum().sort_values(ascending=False)
)
top_two_share = reason_summary.head(2).sum() / reason_summary.sum() * 100
print(
    f"Caption: {reason_summary.index[0]} and {reason_summary.index[1]} together "
    f"cause {top_two_share:.0f}% of all interventions."
)
print()

# --- Chart 3: zones ---
zone_table = database_connection.execute("SELECT * FROM v_by_zone").df()
make_zone_bar_chart(zone_table).show()

worst_zone_row = zone_table.sort_values(
    "interventions_per_1000km", ascending=False
).iloc[0]
print(
    f"Caption: {worst_zone_row['zone']} is the hardest zone, at "
    f"{worst_zone_row['interventions_per_1000km']:.1f} interventions per 1,000 km."
)

Caption: v2.3 needs help most often, at 61.5 interventions per 1,000 km.



Caption: crosswalk_confusion and blocked_path together cause 52% of all interventions.



Caption: Old Town is the hardest zone, at 73.5 interventions per 1,000 km.


In [26]:
# =====================================================================
# CELL E4: Delivery and cost charts
# Creates two chart functions, each returning a Plotly figure:
#   1. make_on_time_heatmap         : on-time % by zone and start hour
#   2. make_cost_by_daytype_chart   : cost per delivery, weekday vs weekend
# Then it loads each view, draws each chart and prints a caption.
# =====================================================================


# ---------------------------------------------------------------------
# FUNCTION 1: on-time % heatmap (zone x hour)
# ---------------------------------------------------------------------
def make_on_time_heatmap(hour_zone_dataframe):
    """Heatmap with zones as rows, start hours as columns, on-time % as colour.

    hour_zone_dataframe needs these columns (from v_by_hour):
    start_hour, zone, on_time_deliveries, completed_deliveries
    """

    # Add up the counts for each zone and hour. We add the counts FIRST and
    # divide AFTER (ratio of sums), so the chart stays correct even if I pass
    # in only some of the rows (for example, a filtered set of zones).
    cell_totals = hour_zone_dataframe.groupby(["zone", "start_hour"], as_index=False)[
        ["on_time_deliveries", "completed_deliveries"]
    ].sum()

    # On-time % for every zone-and-hour cell. Where nothing was completed
    # we leave the cell empty instead of dividing by zero.
    cell_totals["on_time_percent"] = None
    has_deliveries = cell_totals["completed_deliveries"] > 0
    cell_totals.loc[has_deliveries, "on_time_percent"] = (
        cell_totals.loc[has_deliveries, "on_time_deliveries"]
        / cell_totals.loc[has_deliveries, "completed_deliveries"]
        * 100
    )
    cell_totals["on_time_percent"] = cell_totals["on_time_percent"].astype(float)

    # Reshape into a grid: one row per zone, one column per hour
    heatmap_grid = cell_totals.pivot(
        index="zone", columns="start_hour", values="on_time_percent"
    )

    # Put the zone with the lowest average on-time % at the top of the chart
    zone_order = (
        cell_totals.groupby("zone")[["on_time_deliveries", "completed_deliveries"]]
        .sum()
        .assign(
            zone_on_time=lambda table: table["on_time_deliveries"]
            / table["completed_deliveries"]
        )
        .sort_values("zone_on_time")
        .index.tolist()
    )
    heatmap_grid = heatmap_grid.loc[zone_order]

    # Draw the heatmap. The "RdYlBu" colour scale runs red (low) to blue
    # (high), so late zones and hours show up in a warm colour.
    heatmap_figure = go.Figure(
        go.Heatmap(
            z=heatmap_grid.values,
            x=[f"{int(hour)}:00" for hour in heatmap_grid.columns],
            y=heatmap_grid.index.tolist(),
            colorscale="RdYlBu",
            colorbar=dict(title="On-time %"),
            hovertemplate=(
                "Zone: %{y}<br>Start hour: %{x}<br>On-time: %{z:.1f}%<extra></extra>"
            ),
        )
    )

    # Title, axis names and the shared style settings
    heatmap_figure.update_layout(
        title="On-time % by zone and hour",
        xaxis_title="Trip start hour",
        yaxis_title="Zone",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
    )

    return heatmap_figure


# ---------------------------------------------------------------------
# FUNCTION 2: cost per delivery, weekday versus weekend
# ---------------------------------------------------------------------
def make_cost_by_daytype_chart(daytype_dataframe):
    """Two bars (weekday, weekend) for cost_per_delivery in dollars.

    daytype_dataframe needs these columns (from v_by_daytype):
    day_type, cost_per_delivery
    """

    # Always show weekday first, then weekend
    day_type_order = ["weekday", "weekend"]
    chart_data = daytype_dataframe.set_index("day_type").loc[day_type_order].reset_index()

    # The more expensive bar is orange, the cheaper one is blue
    highest_cost = chart_data["cost_per_delivery"].max()
    bar_colors = []
    for bar_value in chart_data["cost_per_delivery"]:
        if bar_value == highest_cost:
            bar_colors.append(CHART_HIGHLIGHT_COLOR)
        else:
            bar_colors.append(CHART_MAIN_COLOR)

    # Draw the bars with the dollar value printed above each one
    cost_figure = go.Figure(
        go.Bar(
            x=chart_data["day_type"],
            y=chart_data["cost_per_delivery"],
            marker_color=bar_colors,
            text=chart_data["cost_per_delivery"],
            texttemplate="$%{text:.2f}",
            textposition="outside",
        )
    )

    cost_figure.update_layout(
        title="Cost per delivery: weekday vs weekend",
        xaxis_title="Type of day",
        yaxis_title="Cost per delivery ($)",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
    )

    # Start at zero and leave room above the tallest bar for its label
    cost_figure.update_yaxes(range=[0, highest_cost * 1.2])

    return cost_figure


# ---------------------------------------------------------------------
# Load each view, draw each chart, print a caption
# ---------------------------------------------------------------------

# --- Chart 1: on-time heatmap ---
hour_zone_table = database_connection.execute("SELECT * FROM v_by_hour").df()
make_on_time_heatmap(hour_zone_table).show()

# Find the single worst zone-and-hour cell that has at least 30 deliveries
# (tiny cells would give noisy percentages)
hour_zone_cells = hour_zone_table[hour_zone_table["completed_deliveries"] >= 30].copy()
hour_zone_cells["on_time_percent"] = (
    hour_zone_cells["on_time_deliveries"] / hour_zone_cells["completed_deliveries"] * 100
)
worst_cell = hour_zone_cells.sort_values("on_time_percent").iloc[0]
print(
    f"Caption: the worst slot is {worst_cell['zone']} at "
    f"{int(worst_cell['start_hour'])}:00, with only "
    f"{worst_cell['on_time_percent']:.1f}% of deliveries on time."
)
print()

# --- Chart 2: cost by day type ---
daytype_table = database_connection.execute("SELECT * FROM v_by_daytype").df()
make_cost_by_daytype_chart(daytype_table).show()

cost_lookup = dict(zip(daytype_table["day_type"], daytype_table["cost_per_delivery"]))
cost_gap = cost_lookup["weekday"] - cost_lookup["weekend"]
print(
    f"Caption: a weekend delivery costs ${cost_lookup['weekend']:.2f} against "
    f"${cost_lookup['weekday']:.2f} on a weekday ({abs(cost_gap):.2f} dollars "
    f"{'cheaper' if cost_gap > 0 else 'dearer'}), because the fixed daily robot "
    f"cost is shared across more trips."
)

Caption: the worst slot is Old Town at 8:00, with only 4.1% of deliveries on time.



Caption: a weekend delivery costs $1.35 against $1.53 on a weekday (0.18 dollars cheaper), because the fixed daily robot cost is shared across more trips.


In [27]:
# =====================================================================
# CELL E5: Robot chart and table
# Creates two functions:
#   1. make_robot_drain_chart : bar chart of battery used per km, per robot
#   2. prepare_robot_table    : tidy table with friendly column names
# Then it loads v_by_robot, shows the chart and the top 10 table rows.
# =====================================================================


# ---------------------------------------------------------------------
# FUNCTION 1: battery drain bar chart
# ---------------------------------------------------------------------
def make_robot_drain_chart(robot_dataframe):
    """One bar per robot for avg_charge_used_per_km, highest first.

    The 5 robots with drain_rank 1 to 5 are orange, all others are grey.

    robot_dataframe needs these columns (from v_by_robot):
    robot_id, avg_charge_used_per_km, drain_rank
    """

    # Sort so the hungriest batteries are on the left
    chart_data = robot_dataframe.sort_values(
        "avg_charge_used_per_km", ascending=False
    )

    # One colour per bar: orange for drain rank 1 to 5, grey for the rest
    bar_colors = []
    for rank_value in chart_data["drain_rank"]:
        if rank_value <= 5:
            bar_colors.append(CHART_HIGHLIGHT_COLOR)
        else:
            bar_colors.append(CHART_GREY_COLOR)

    # Draw the bars. The hover text shows the exact value for each robot.
    drain_figure = go.Figure(
        go.Bar(
            x=chart_data["robot_id"],
            y=chart_data["avg_charge_used_per_km"],
            marker_color=bar_colors,
            hovertemplate=(
                "Robot: %{x}<br>Charge used: %{y:.2f}% per km<extra></extra>"
            ),
        )
    )

    # Title, axis names with units, and the shared style settings
    drain_figure.update_layout(
        title="Battery used per km, by robot",
        xaxis_title="Robot",
        yaxis_title="Charge used (% per km)",
        template=CHART_TEMPLATE,
        height=CHART_HEIGHT,
    )

    # Turn the 50 robot names sideways so they do not overlap
    drain_figure.update_xaxes(tickangle=-60, tickfont=dict(size=9))

    # Start at zero so the gap between tired and healthy robots is not exaggerated
    drain_figure.update_yaxes(rangemode="tozero")

    return drain_figure


# ---------------------------------------------------------------------
# FUNCTION 2: tidy robot table
# ---------------------------------------------------------------------
def prepare_robot_table(robot_dataframe):
    """Return a readable table sorted by drain_rank, with friendly names."""

    # Sort so the worst battery drain is at the top
    sorted_table = robot_dataframe.sort_values("drain_rank")

    # Build the new table one column at a time, with sensible rounding
    tidy_table = pd.DataFrame(
        {
            "Robot": sorted_table["robot_id"],
            "Battery health": sorted_table["battery_health"].round(2),
            "Missions": sorted_table["missions"].astype(int),
            "km": sorted_table["km"].round(1),
            "Interventions per 1,000 km": sorted_table[
                "interventions_per_1000km"
            ].round(1),
            "Charge used per km (%)": sorted_table["avg_charge_used_per_km"].round(2),
            "Battery failures": sorted_table["battery_failures"].astype(int),
            # The failure rate is stored as a share (0.012), so show it as a percent
            "Failure rate (%)": (sorted_table["battery_failure_rate"] * 100).round(2),
        }
    )

    return tidy_table


# ---------------------------------------------------------------------
# Load the view, draw the chart, show the table
# ---------------------------------------------------------------------
robot_view_table = database_connection.execute("SELECT * FROM v_by_robot").df()

# The chart
make_robot_drain_chart(robot_view_table).show()

# The table: first 10 rows (the 5 tired robots should lead)
robot_display_table = prepare_robot_table(robot_view_table)
print("Top 10 robots by battery drain:")
display(robot_display_table.head(10))

# A caption with real numbers from the data
top_five_robots = robot_view_table[robot_view_table["drain_rank"] <= 5]
other_robots = robot_view_table[robot_view_table["drain_rank"] > 5]

# Average drain for each group, weighted by km (a ratio of sums, as always)
top_five_drain = (
    top_five_robots["avg_charge_used_per_km"] * top_five_robots["km"]
).sum() / top_five_robots["km"].sum()
other_drain = (
    other_robots["avg_charge_used_per_km"] * other_robots["km"]
).sum() / other_robots["km"].sum()

print(
    f"Caption: the five hungriest robots ({', '.join(top_five_robots['robot_id'])}) "
    f"use {top_five_drain:.1f}% charge per km against {other_drain:.1f}% for the "
    f"rest, about {top_five_drain / other_drain:.1f} times as much."
)

Top 10 robots by battery drain:


,Robot,Battery health,Missions,km,"Interventions per 1,000 km",Charge used per km (%),Battery failures,Failure rate (%)
0,R007,0.65,851,"1,451.80",39.30,7.69,5,0.59
1,R019,0.65,856,"1,460.10",39.00,7.69,10,1.17
2,R023,0.65,914,"1,570.50",42.70,7.68,11,1.20
3,R044,0.65,886,"1,530.40",54.20,7.68,8,0.90
4,R031,0.65,875,"1,483.00",29.70,7.66,10,1.14
5,R028,0.86,887,"1,513.20",44.90,5.81,4,0.45
6,R005,0.86,832,"1,404.20",27.80,5.81,2,0.24
7,R018,0.86,858,"1,428.10",42.00,5.81,3,0.35
8,R009,0.87,873,"1,505.80",37.20,5.75,1,0.11
9,R050,0.87,899,"1,507.90",47.10,5.75,3,0.33


Caption: the five hungriest robots (R007, R019, R023, R044, R031) use 7.7% charge per km against 5.4% for the rest, about 1.4 times as much.


In [28]:
# =====================================================================
# CELL F1: Download the files for GitHub
# This cell:
#   1. writes requirements.txt (the packages Streamlit must install)
#   2. closes the database so fleet.duckdb is completely saved
#   3. prints the size of each file and warns if one is too big
#   4. downloads the four files to your computer
# =====================================================================

import os
from google.colab import files

# ---------------------------------------------------------------------
# Step 1: write requirements.txt
# Streamlit Community Cloud reads this file and installs each package
# listed in it before it starts your app. One package per line.
# ---------------------------------------------------------------------
required_packages = ["streamlit", "duckdb", "pandas", "plotly"]

with open("requirements.txt", "w") as requirements_file:
    requirements_file.write("\n".join(required_packages) + "\n")

print("Wrote requirements.txt with:", ", ".join(required_packages))

# ---------------------------------------------------------------------
# Step 2: close the database connection
# DuckDB keeps some recent changes in a side file until the connection
# closes. Closing makes sure everything is written into fleet.duckdb.
# ---------------------------------------------------------------------
database_connection.close()

# If a leftover side file (.wal) exists, the database was not closed cleanly
if os.path.exists(DATABASE_FILE + ".wal"):
    print("WARNING: a .wal file still exists. Rerun this cell before downloading.")

# ---------------------------------------------------------------------
# Step 3: print the size of each file
# The GitHub web upload limit is 25 MB per file, so we warn if we are over.
# ---------------------------------------------------------------------
files_to_download = [
    DATABASE_FILE,                                  # fleet.duckdb
    "requirements.txt",
    os.path.join(SQL_FOLDER, "metric_views.sql"),   # sql/metric_views.sql
    "data_quality_checks.csv",
]

GITHUB_UPLOAD_LIMIT_MB = 25   # maximum file size for the GitHub web uploader

print("\nFile sizes:")
for file_path in files_to_download:
    if not os.path.exists(file_path):
        print(f"  MISSING: {file_path}  (rerun the cell that creates it)")
        continue

    size_in_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  {file_path:30s} {size_in_mb:8.2f} MB")

    if size_in_mb > GITHUB_UPLOAD_LIMIT_MB:
        print(f"  WARNING: {file_path} is over {GITHUB_UPLOAD_LIMIT_MB} MB.")

# ---------------------------------------------------------------------
# Step 4: download the files, one at a time
# Your browser may ask permission to download several files. Click Allow.
# ---------------------------------------------------------------------
print("\nStarting downloads...")
for file_path in files_to_download:
    if os.path.exists(file_path):
        files.download(file_path)

# ---------------------------------------------------------------------
# A reminder of where each file goes in your GitHub repo
# ---------------------------------------------------------------------
print("\nWhere each file goes in your GitHub repo:")
print("  fleet.duckdb             -> main folder")
print("  requirements.txt         -> main folder")
print("  data_quality_checks.csv  -> main folder")
print("  metric_views.sql         -> inside a folder called sql")

Wrote requirements.txt with: streamlit, duckdb, pandas, plotly

File sizes:
  fleet.duckdb                       2.76 MB
  requirements.txt                   0.00 MB
  sql/metric_views.sql               0.02 MB
  data_quality_checks.csv            0.00 MB

Starting downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Where each file goes in your GitHub repo:
  fleet.duckdb             -> main folder
  requirements.txt         -> main folder
  data_quality_checks.csv  -> main folder
  metric_views.sql         -> inside a folder called sql
